# Set up

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 0: Setup — chạy 1 lần
# ═══════════════════════════════════════════════════════════════
!sudo apt-get update -qq
!sudo apt-get install -qq -y libcairo2-dev libpango1.0-dev ffmpeg texlive texlive-latex-extra texlive-fonts-extra texlive-latex-recommended
!pip install manim "manim-voiceover[gtts]" edge-tts nest_asyncio pydub -q

import nest_asyncio
nest_asyncio.apply()

print("Setup done!")


# SLIDE 81–85

## Generative

In [ ]:
%%writefile /content/slides_81_85.py
import nest_asyncio
nest_asyncio.apply()

import asyncio
import hashlib
import random
from pathlib import Path

from manim import *
from manim_voiceover import VoiceoverScene
from manim_voiceover.services.base import SpeechService

import edge_tts

# ─────────────────────────────────────────────
# EdgeTTSService — must be in this file
# ─────────────────────────────────────────────
VOICE = "en-US-GuyNeural"


class EdgeTTSService(SpeechService):
    """Custom TTS service using Microsoft Edge TTS, Colab-compatible."""

    def __init__(self, voice="en-US-GuyNeural", **kwargs):
        self.voice = voice
        super().__init__(**kwargs)

    def generate_from_text(self, text, cache_dir=None, path=None, **kwargs):
        if cache_dir is None:
            cache_dir = self.cache_dir
        Path(cache_dir).mkdir(parents=True, exist_ok=True)

        input_data = {"input_text": text, "service": "edge_tts", "voice": self.voice}

        cached = self.get_cached_result(input_data, cache_dir)
        if cached is not None:
            return cached

        audio_basename = hashlib.md5(text.encode()).hexdigest()
        audio_file = audio_basename + ".mp3"
        full_path = str(Path(cache_dir) / audio_file)

        communicate = edge_tts.Communicate(text, self.voice)
        loop = asyncio.get_event_loop()
        loop.run_until_complete(communicate.save(full_path))

        json_dict = {
            "input_text": text,
            "input_data": input_data,
            "original_audio": audio_file,
        }

        return json_dict


class Scene01Title(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title = Text("Recent Results I:", font_size=44, color=WHITE).shift(UP * 0.5)
        subtitle = Text("Expanders", font_size=38, color=WHITE, weight=BOLD).next_to(title, DOWN, buff=0.4)

        with self.voiceover(
            text="We now move to one of the most exciting recent developments in geometric machine learning: the connection between expander graphs and symmetry."
        ) as tracker:
            self.play(FadeIn(title, shift=DOWN * 0.3), run_time=tracker.duration * 0.4)
            self.play(FadeIn(subtitle, shift=UP * 0.3), run_time=tracker.duration * 0.4)
            self.wait(tracker.duration * 0.15)

        # Tagline
        left_part = Text("Expander Graphs", font_size=28, color=BLUE)
        times_sign = MathTex(r"\times", font_size=36, color=WHITE)
        mid_part = Text("Symmetry Groups", font_size=28, color=GREEN)
        arrow_sign = MathTex(r"\longrightarrow", font_size=36, color=YELLOW)
        right_part = Text("Efficient Geometric ML", font_size=28, color=YELLOW)
        tagline = VGroup(left_part, times_sign, mid_part, arrow_sign, right_part).arrange(RIGHT, buff=0.25)
        tagline.next_to(subtitle, DOWN, buff=0.9)

        with self.voiceover(
            text="This section will show how ideas from combinatorics and group theory can make working with large symmetry groups computationally feasible."
        ) as tracker:
            self.play(FadeIn(tagline, shift=UP * 0.2), run_time=tracker.duration * 0.5)
            self.wait(tracker.duration * 0.4)

        self.play(FadeOut(title), FadeOut(subtitle), FadeOut(tagline), run_time=0.8)
        self.wait(0.3)


class Scene02Groups(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        scene_title = Text("The Scale Problem", font_size=36, color=WHITE).to_edge(UP, buff=0.5)
        self.play(Write(scene_title), run_time=0.8)

        # --- Sentence 0: invariance/equivariance needed ---
        inv_eq = MathTex(r"\text{Goal: } f(g \cdot x) = f(x) \quad \forall g \in G", font_size=38, color=WHITE).next_to(scene_title, DOWN, buff=0.8)

        with self.voiceover(
            text="Let's start with the core problem. In many machine learning tasks, we need our models to be invariant or equivariant under a group of symmetries."
        ) as tracker:
            self.play(FadeIn(inv_eq), run_time=tracker.duration * 0.5)
            self.wait(tracker.duration * 0.4)

        # --- Sentence 1: Permutation group with animated shuffle ---
        self.play(FadeOut(inv_eq), run_time=0.4)

        perm_line = MathTex(r"S_d = \text{all permutations of } d \text{ elements}", font_size=34, color=WHITE).next_to(scene_title, DOWN, buff=0.6)
        nums_orig = VGroup(*[
            MathTex(str(i), font_size=36, color=WHITE)
            for i in range(1, 11)
        ]).arrange(RIGHT, buff=0.35).next_to(perm_line, DOWN, buff=0.7)

        saved_positions = [nums_orig[i].get_center().copy() for i in range(10)]

        with self.voiceover(
            text="Consider the permutation group S d — the group of all possible ways to reorder d elements. This is a finite group, but it is enormous."
        ) as tracker:
            self.play(Write(perm_line), run_time=0.8)
            self.play(FadeIn(nums_orig), run_time=0.5)
            # Shuffle animation
            for _ in range(3):
                nums_list = list(range(10))
                random.shuffle(nums_list)
                anims = [nums_orig[i].animate.move_to(saved_positions[nums_list[i]]) for i in range(10)]
                self.play(*anims, run_time=0.8)
                self.wait(0.3)

        # --- Sentence 2: d=50 example ---
        self.play(FadeOut(nums_orig), FadeOut(perm_line), run_time=0.4)

        size_eq = MathTex(r"d = 50", font_size=44, color=WHITE).next_to(scene_title, DOWN, buff=0.7)
        implies = MathTex(r"\Downarrow", font_size=40, color=YELLOW).next_to(size_eq, DOWN, buff=0.3)
        factorial = MathTex(r"|S_{50}| = 50! \approx 3 \times 10^{64}", font_size=40, color=WHITE).next_to(implies, DOWN, buff=0.3)

        with self.voiceover(
            text="For example, when d equals just 50, the size of S d — that is, 50 factorial — is approximately 3 times 10 to the power of 64."
        ) as tracker:
            self.play(Write(size_eq), run_time=0.6)
            self.play(FadeIn(implies), run_time=0.3)
            self.play(Write(factorial), run_time=tracker.duration * 0.4)
            self.wait(tracker.duration * 0.15)

        # --- Sentence 3: comparison with atoms ---
        comp_box = VGroup(
            MathTex(r"3 \times 10^{64}", font_size=34, color=RED),
            MathTex(r"\gg", font_size=40, color=WHITE),
            MathTex(r"10^{50}", font_size=34, color=BLUE),
            Text("(atoms in Earth)", font_size=22, color=BLUE),
        ).arrange(RIGHT, buff=0.3).next_to(factorial, DOWN, buff=0.6)

        with self.voiceover(
            text="To put that in perspective, this number is way bigger than the total number of atoms in the entire Earth, which is estimated at around 10 to the power of 50."
        ) as tracker:
            self.play(FadeIn(comp_box), run_time=tracker.duration * 0.4)
            self.wait(tracker.duration * 0.5)

        # --- Sentence 4 & 5: Group averaging formula ---
        self.play(FadeOut(size_eq), FadeOut(implies), FadeOut(factorial), FadeOut(comp_box), run_time=0.5)

        avg_title = Text("Group Averaging", font_size=30, color=YELLOW).next_to(scene_title, DOWN, buff=0.5)
        avg_formula = MathTex(
            r"\bar{f}(x)", r"=", r"\frac{1}{|G|}", r"\sum_{g \in G}", r"f(g^{-1} \cdot x)",
            font_size=42, color=WHITE
        ).next_to(avg_title, DOWN, buff=0.5)

        with self.voiceover(
            text="Now recall the technique of group averaging. To make any function f invariant, we compute the average of f over all group elements."
        ) as tracker:
            self.play(Write(avg_title), run_time=0.5)
            self.play(Write(avg_formula), run_time=tracker.duration * 0.5)
            self.wait(tracker.duration * 0.2)

        with self.voiceover(
            text="Specifically, f bar of x equals one over the size of G, times the sum of f of g inverse dot x, for every g in G. The key issue is that sum."
        ) as tracker:
            # Highlight components sequentially
            self.play(Indicate(avg_formula[2], color=YELLOW), run_time=0.8)  # 1/|G|
            self.play(Indicate(avg_formula[3], color=RED), run_time=0.8)    # sum over G
            self.play(Indicate(avg_formula[4], color=BLUE), run_time=0.8)   # f(g^-1 x)
            self.wait(tracker.duration * 0.2)

        # --- Sentence 6: impractical ---
        sum_highlight = SurroundingRectangle(avg_formula[3], color=RED, buff=0.1, stroke_width=2)
        problem_text = Text("This sum has |G| terms!", font_size=24, color=RED).next_to(sum_highlight, DOWN, buff=0.3)

        with self.voiceover(
            text="When the group has 3 times 10 to the 64 elements, summing over every single one of them is completely impractical. Even the fastest computer could never finish."
        ) as tracker:
            self.play(Create(sum_highlight), FadeIn(problem_text), run_time=0.8)
            impractical = MathTex(
                r"3 \times 10^{64} \text{ terms} \implies \text{IMPOSSIBLE}",
                font_size=32, color=RED
            ).next_to(problem_text, DOWN, buff=0.4)
            self.play(Write(impractical), run_time=1.0)
            self.wait(tracker.duration * 0.2)

        # --- Sentence 7: key question ---
        self.play(
            *[FadeOut(m) for m in [avg_title, avg_formula, sum_highlight, problem_text, impractical, scene_title]],
            run_time=0.6
        )

        question = Text("Key Question:", font_size=36, color=YELLOW).shift(UP * 1.0)
        question_body = Text(
            "Can a small subset of G retain enough\nstructure for Geometric ML?",
            font_size=30, color=WHITE, line_spacing=1.4
        ).next_to(question, DOWN, buff=0.5)
        q_rect = SurroundingRectangle(VGroup(question, question_body), color=YELLOW, buff=0.3, stroke_width=2)

        with self.voiceover(
            text="So here is the key question: can we find a small subset of a large finite group that retains enough of the group's structure for geometric ML — like data augmentation, averaging, and optimization?"
        ) as tracker:
            self.play(Write(question), run_time=0.6)
            self.play(FadeIn(question_body), Create(q_rect), run_time=1.2)
            self.wait(tracker.duration * 0.3)

        self.play(FadeOut(question), FadeOut(question_body), FadeOut(q_rect), run_time=0.6)
        self.wait(0.3)


class Scene03GeneratingSets(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        s3_title = Text("Generating Sets", font_size=36, color=WHITE).to_edge(UP, buff=0.5)

        with self.voiceover(
            text="To answer this question, we turn to a classical concept from group theory: the generating set."
        ) as tracker:
            self.play(Write(s3_title), run_time=tracker.duration * 0.4)
            self.wait(tracker.duration * 0.5)

        # --- Sentence 1: Definition ---
        def_text = MathTex(
            r"S \subseteq G", r"\text{ generates } G", r"\text{ if } \forall g \in G:",
            font_size=32, color=WHITE
        ).next_to(s3_title, DOWN, buff=0.5)
        def_formula = MathTex(
            r"g = s_1 \cdot s_2 \cdots s_k", r"\quad (s_i \in S)",
            font_size=34, color=WHITE
        ).next_to(def_text, DOWN, buff=0.3)
        def_box = SurroundingRectangle(VGroup(def_text, def_formula), color=BLUE, buff=0.2, stroke_width=1.5)

        with self.voiceover(
            text="A subset S of a group G is called a generating set if every element of G can be written as a product of elements from S."
        ) as tracker:
            self.play(Write(def_text), run_time=tracker.duration * 0.35)
            self.play(Write(def_formula), run_time=tracker.duration * 0.35)
            self.play(Create(def_box), run_time=0.5)
            self.wait(tracker.duration * 0.1)

        # --- Sentence 2: Concrete example S5 ---
        self.play(FadeOut(def_text), FadeOut(def_formula), FadeOut(def_box), run_time=0.4)

        example_title = Text("Example: S₅ with adjacent transpositions", font_size=28, color=YELLOW).next_to(s3_title, DOWN, buff=0.5)
        gen_set = MathTex(
            r"S = \{(1\,2),\; (2\,3),\; (3\,4),\; (4\,5)\}",
            font_size=32, color=WHITE
        ).next_to(example_title, DOWN, buff=0.4)

        with self.voiceover(
            text="Let's see a concrete example. Take the permutation group S 5. Our generating set S consists of adjacent transpositions: swap 1 and 2, swap 2 and 3, swap 3 and 4, and swap 4 and 5."
        ) as tracker:
            self.play(Write(example_title), run_time=0.8)
            self.play(Write(gen_set), run_time=tracker.duration * 0.4)
            self.wait(tracker.duration * 0.2)

        # --- Sentence 3: Animated composition ---
        self.play(FadeOut(gen_set), run_time=0.3)

        nums = VGroup(*[
            MathTex(str(i), font_size=40, color=WHITE) for i in [1, 2, 3, 4, 5]
        ]).arrange(RIGHT, buff=0.7).next_to(example_title, DOWN, buff=0.6)

        step_label = Text("Start: [1, 2, 3, 4, 5]", font_size=22, color=GREEN).next_to(nums, DOWN, buff=0.5)
        positions = [nums[i].get_center().copy() for i in range(5)]

        with self.voiceover(
            text="Watch how we compose these simple swaps. Starting from 1 2 3 4 5, swapping positions 2 and 3 gives 1 3 2 4 5. Then swapping positions 1 and 2 gives 3 1 2 4 5. Just two generators, composed, already produce a new permutation!"
        ) as tracker:
            self.play(FadeIn(nums), FadeIn(step_label), run_time=0.6)

            # Swap (2,3)
            swap_arc1 = CurvedArrow(
                nums[1].get_top() + UP * 0.1, nums[2].get_top() + UP * 0.1,
                color=YELLOW, angle=-TAU / 5, tip_length=0.15
            )
            swap_label1 = MathTex(r"(2\,3)", font_size=24, color=YELLOW).next_to(swap_arc1, UP, buff=0.1)
            self.play(Create(swap_arc1), FadeIn(swap_label1), run_time=0.5)
            self.play(
                nums[1].animate.move_to(positions[2]),
                nums[2].animate.move_to(positions[1]),
                run_time=0.7
            )
            new_step = Text("After (2,3): [1, 3, 2, 4, 5]", font_size=22, color=YELLOW).next_to(nums, DOWN, buff=0.5)
            self.play(FadeOut(step_label), FadeIn(new_step), FadeOut(swap_arc1), FadeOut(swap_label1), run_time=0.5)

            # Swap (1,2)
            swap_arc2 = CurvedArrow(
                positions[0] + UP * 0.5, positions[1] + UP * 0.5,
                color=ORANGE, angle=-TAU / 5, tip_length=0.15
            )
            swap_label2 = MathTex(r"(1\,2)", font_size=24, color=ORANGE).next_to(swap_arc2, UP, buff=0.1)
            self.play(Create(swap_arc2), FadeIn(swap_label2), run_time=0.5)
            self.play(
                nums[0].animate.move_to(positions[1]),
                nums[2].animate.move_to(positions[0]),
                run_time=0.7
            )
            final_step = Text("After (1,2): [3, 1, 2, 4, 5]", font_size=22, color=ORANGE).next_to(nums, DOWN, buff=0.5)
            self.play(FadeOut(new_step), FadeIn(final_step), FadeOut(swap_arc2), FadeOut(swap_label2), run_time=0.5)

            compose_note = MathTex(
                r"(1\,2) \circ (2\,3) \;=\; \text{new permutation!}",
                font_size=28, color=GREEN
            ).next_to(final_step, DOWN, buff=0.4)
            self.play(Write(compose_note), run_time=0.8)
            self.wait(0.5)

        # --- Sentence 4: Implication for invariance ---
        self.play(*[FadeOut(m) for m in [nums, example_title, final_step, compose_note]], run_time=0.5)

        # S inside G diagram
        big_circle = Circle(radius=1.8, color=WHITE, stroke_width=2).shift(DOWN * 1.0)
        big_label = MathTex(r"G", font_size=36, color=WHITE).next_to(big_circle, UP, buff=0.1)
        small_circle = Circle(radius=0.6, color=YELLOW, stroke_width=2, fill_opacity=0.15, fill_color=YELLOW).shift(DOWN * 1.0 + LEFT * 0.5)
        small_label = MathTex(r"S", font_size=30, color=YELLOW).move_to(small_circle)
        gen_arrow = Arrow(small_circle.get_right(), big_circle.get_right() + LEFT * 0.3, color=GREEN, buff=0.1)
        gen_text = Text("generates", font_size=20, color=GREEN).next_to(gen_arrow, UP, buff=0.1)

        inv_s = MathTex(r"f^*(s \cdot x) = f^*(x), \; \forall s \in S", font_size=28, color=YELLOW).shift(DOWN * 3.0 + LEFT * 2)
        inv_arrow = MathTex(r"\Longrightarrow", font_size=36, color=GREEN).next_to(inv_s, RIGHT, buff=0.3)
        inv_g = MathTex(r"f^*(g \cdot x) = f^*(x), \; \forall g \in G", font_size=28, color=WHITE).next_to(inv_arrow, RIGHT, buff=0.3)

        with self.voiceover(
            text="This has a powerful implication: if a function f star is invariant under every element of S, then it is automatically invariant under the entire group G."
        ) as tracker:
            self.play(Create(big_circle), FadeIn(big_label), run_time=0.6)
            self.play(Create(small_circle), FadeIn(small_label), run_time=0.5)
            self.play(GrowArrow(gen_arrow), FadeIn(gen_text), run_time=0.5)
            self.play(Write(inv_s), run_time=0.8)
            self.play(FadeIn(inv_arrow), Write(inv_g), run_time=0.8)
            self.wait(tracker.duration * 0.1)

        # --- Sentence 5: Enforce on S only ---
        with self.voiceover(
            text="So instead of checking invariance over all of G, we only need to enforce it on the much smaller set S."
        ) as tracker:
            enforce_note = Text("→ Only enforce on S, not all of G!", font_size=24, color=GREEN).shift(DOWN * 3.5)
            self.play(FadeIn(enforce_note, shift=UP * 0.2), run_time=0.6)
            self.wait(tracker.duration * 0.4)

        # --- Sentence 6: NP-hard + adjacent transpositions ---
        self.play(*[FadeOut(m) for m in self.mobjects if m != s3_title], run_time=0.5)

        with self.voiceover(
            text="But here is the catch: finding a minimum size generating set is NP-hard in general. However, for S d, adjacent transpositions give us a generating set of size only d minus 1."
        ) as tracker:
            nphard = VGroup(
                Text("Minimum |S|?", font_size=28, color=WHITE),
                Text("→ NP-hard in general", font_size=26, color=RED),
            ).arrange(RIGHT, buff=0.4).next_to(s3_title, DOWN, buff=0.7)

            but_sd = VGroup(
                MathTex(r"\text{But for } S_d:", font_size=30, color=WHITE),
                MathTex(r"\sigma_i = (i,\; i\!+\!1), \quad i=1,\ldots,d\!-\!1", font_size=28, color=YELLOW),
                MathTex(r"\Rightarrow |S| = d - 1", font_size=30, color=GREEN),
            ).arrange(DOWN, buff=0.3, aligned_edge=LEFT).next_to(nphard, DOWN, buff=0.6)

            self.play(FadeIn(nphard), run_time=0.8)
            self.play(FadeIn(but_sd, shift=UP * 0.2), run_time=1.2)
            self.wait(tracker.duration * 0.5)

        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.2)
        self.wait(0.1)


class Scene04MinimumSize(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        s4_title = Text("How Small Can S Be?", font_size=36, color=WHITE).to_edge(UP, buff=0.5)

        # --- Sentence 0 ---
        with self.voiceover(
            text="Even though finding the exact minimum generating set is NP-hard, we can still ask: what is the smallest possible size?"
        ) as tracker:
            self.play(Write(s4_title), run_time=0.8)
            q_text = Text("What is the smallest possible generating set size?", font_size=26, color=WHITE).next_to(s4_title, DOWN, buff=0.5)
            self.play(FadeIn(q_text), run_time=0.6)
            self.wait(tracker.duration * 0.2)

        self.play(FadeOut(q_text), run_time=0.3)

        # --- Sentence 1: Theorem ---
        thm_label = Text("Theorem", font_size=24, color=YELLOW, weight=BOLD).next_to(s4_title, DOWN, buff=0.5)
        thm_text = MathTex(
            r"\forall \text{ finite group } G, \; \exists \; S \text{ with } |S| \leq \log_2 |G|",
            font_size=34, color=WHITE
        ).next_to(thm_label, DOWN, buff=0.3)
        thm_box = SurroundingRectangle(VGroup(thm_label, thm_text), color=YELLOW, buff=0.25, stroke_width=2)

        with self.voiceover(
            text="A classical theorem gives a beautiful answer: for any finite group G, there exists a generating set with at most log base 2 of the order of G elements."
        ) as tracker:
            self.play(Write(thm_label), run_time=0.4)
            self.play(Write(thm_text), run_time=tracker.duration * 0.5)
            self.play(Create(thm_box), run_time=0.5)
            self.wait(tracker.duration * 0.1)

        # --- Sentence 2: Exponential saving ---
        with self.voiceover(
            text="This is an exponential saving! The group has G elements, but we only need about log G generators."
        ) as tracker:
            bar_G = Rectangle(width=7, height=0.5, color=RED, fill_opacity=0.4, stroke_width=1.5).shift(DOWN * 0.8)
            bar_G_label = MathTex(r"|G| \text{ elements}", font_size=24, color=RED).next_to(bar_G, RIGHT, buff=0.2)
            bar_S = Rectangle(width=0.7, height=0.5, color=GREEN, fill_opacity=0.6, stroke_width=1.5).align_to(bar_G, LEFT).shift(DOWN * 1.5)
            bar_S_label = MathTex(r"\log_2|G| \text{ generators}", font_size=24, color=GREEN).next_to(bar_S, RIGHT, buff=0.2)
            exp_text = Text("Exponential Saving!", font_size=24, color=YELLOW, weight=BOLD).shift(DOWN * 2.3)

            self.play(FadeIn(bar_G), FadeIn(bar_G_label), run_time=0.6)
            self.play(FadeIn(bar_S), FadeIn(bar_S_label), run_time=0.6)
            self.play(FadeIn(exp_text), run_time=0.5)
            self.wait(tracker.duration * 0.1)

        # --- Sentence 3: Concrete number ---
        self.play(FadeOut(bar_G), FadeOut(bar_G_label), FadeOut(bar_S), FadeOut(bar_S_label), FadeOut(exp_text), run_time=0.4)

        concrete = VGroup(
            MathTex(r"|S_{50}| = 3 \times 10^{64}", font_size=34, color=RED),
            MathTex(r"\Downarrow", font_size=36, color=YELLOW),
            MathTex(r"\log_2(3 \times 10^{64}) \approx 215 \text{ generators}", font_size=34, color=GREEN),
        ).arrange(DOWN, buff=0.3).shift(DOWN * 0.7)

        wow = Text("215 generators → all 3×10⁶⁴ permutations!", font_size=22, color=YELLOW)

        with self.voiceover(
            text="For S 50, where the group has 3 times 10 to the 64 elements, log base 2 of that is roughly 215. Just 215 elements can generate all 3 times 10 to the 64 permutations."
        ) as tracker:
            self.play(Write(concrete[0]), run_time=0.8)
            self.play(FadeIn(concrete[1]), run_time=0.3)
            self.play(Write(concrete[2]), run_time=1.2)
            wow.next_to(concrete, DOWN, buff=0.4)
            self.play(FadeIn(wow), run_time=0.6)
            self.wait(tracker.duration * 0.1)

        # --- Sentence 4: Non-constructive ---
        self.play(FadeOut(concrete), FadeOut(wow), run_time=0.4)

        with self.voiceover(
            text="However, there is a significant limitation: the proof is non-constructive. It tells us the small set exists, but gives no algorithm to find one."
        ) as tracker:
            caveat = VGroup(
                Text("Limitation:", font_size=26, color=RED),
                Text("Proof is non-constructive", font_size=28, color=RED),
                Text("(existence proof, no algorithm)", font_size=22, color=WHITE),
            ).arrange(DOWN, buff=0.2).shift(DOWN * 0.5)
            self.play(FadeIn(caveat), run_time=tracker.duration * 0.4)
            self.wait(tracker.duration * 0.6)

        # --- Sentence 5: Can we find constructively? ---
        self.play(FadeOut(caveat), run_time=0.4)

        with self.voiceover(
            text="So the practical question becomes: can we find generating sets of size order log G, in polynomial time? The answer is yes — using randomness."
        ) as tracker:
            next_q = VGroup(
                MathTex(r"\text{Can we find } |S| = O(\log |G|) \text{ in poly-time?}", font_size=32, color=WHITE),
                Text("Answer: YES — using randomness!", font_size=28, color=GREEN, weight=BOLD),
            ).arrange(DOWN, buff=0.4).shift(DOWN * 0.5)
            self.play(Write(next_q[0]), run_time=1.0)
            self.wait(0.5)
            self.play(FadeIn(next_q[1], scale=1.2), run_time=0.8)
            self.wait(tracker.duration * 0.1)

        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.5)
        self.wait(0.3)


class Scene05CayleyExpander(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        s5_title = Text("Random Cayley Graphs are Expanders", font_size=36, color=WHITE).to_edge(UP, buff=0.5)

        # --- Sentence 0 ---
        with self.voiceover(
            text="Here is the elegant solution: instead of carefully constructing a generating set, we simply pick elements at random!"
        ) as tracker:
            self.play(Write(s5_title), run_time=0.8)
            random_idea = Text("Solution: pick generators at random!", font_size=30, color=GREEN).next_to(s5_title, DOWN, buff=0.5)
            self.play(FadeIn(random_idea, shift=UP * 0.2), run_time=0.8)
            self.wait(tracker.duration * 0.2)

        # --- Sentence 1: Alon-Roichman ---
        self.play(FadeOut(random_idea), run_time=0.3)

        thm_label = Text("Alon-Roichman Theorem", font_size=26, color=YELLOW, weight=BOLD).next_to(s5_title, DOWN, buff=0.4)
        thm_main = MathTex(
            r"\text{Random } S \text{ with } |S| = O(\log|G|)",
            font_size=32, color=WHITE
        ).next_to(thm_label, DOWN, buff=0.25)
        thm_result = Text("is a generating set with high probability", font_size=24, color=WHITE).next_to(thm_main, DOWN, buff=0.2)
        thm_box = SurroundingRectangle(VGroup(thm_label, thm_main, thm_result), color=YELLOW, buff=0, stroke_width=2)
        thm_box.scale([1.2, 0.7, 1])
        with self.voiceover(
            text="The Alon-Roichman theorem states: a randomly chosen subset S, with size proportional to log of the group size, is a generating set with high probability."
        ) as tracker:
            self.play(Write(thm_label), run_time=0.5)
            self.play(Write(thm_main), run_time=tracker.duration * 0.35)
            self.play(FadeIn(thm_result), run_time=0.6)
            self.play(Create(thm_box), run_time=0.5)
            self.wait(tracker.duration * 0.1)

        # --- Sentence 2: Size emphasis ---
        with self.voiceover(
            text="More precisely, the size of S is about 2.67 times log G. This grows only logarithmically — an enormous saving."
        ) as tracker:
            size_note = MathTex(
                r"|S| \approx 2.67 \times \log|G| \quad \text{(logarithmic!)}",
                font_size=28, color=GREEN
            ).next_to(thm_box, DOWN, buff=0.4)
            self.play(Write(size_note), run_time=tracker.duration * 0.5)
            self.wait(tracker.duration * 0.4)

        # --- Sentence 3: Cayley Graph is Expander ---
        self.play(FadeOut(thm_label), FadeOut(thm_main), FadeOut(thm_result), FadeOut(thm_box), FadeOut(size_note), run_time=0.5)

        with self.voiceover(
            text="But this result tells us something even deeper. The Cayley graph built from this random generating set is an expander graph."
        ) as tracker:
            cayley_title = Text("Cayley Graph → Expander Graph", font_size=32, color=YELLOW, weight=BOLD).next_to(s5_title, DOWN, buff=0.4)
            self.play(FadeIn(cayley_title), run_time=tracker.duration * 0.4)
            self.wait(tracker.duration * 0.5)

        # --- Sentence 4: Build graph ---
        self.play(cayley_title.animate.scale(0.8).to_edge(UP, buff=0.3), FadeOut(s5_title), run_time=0.4)

        n_nodes = 30
        vertices = list(range(n_nodes))
        edges = []
        for i in range(n_nodes):
            for offset in [1, 3, 7]:
                j = (i + offset) % n_nodes
                edge = (min(i, j), max(i, j))
                if edge not in edges:
                    edges.append(edge)

        graph = Graph(
            vertices, edges,
            layout="circular",
            layout_scale=2.5,
            vertex_config={"radius": 0.08, "fill_color": WHITE, "fill_opacity": 0.8},
            edge_config={"stroke_width": 0.8, "stroke_opacity": 0.3, "stroke_color": WHITE},
        ).shift(DOWN * 0.4 + LEFT * 1.5)

        node_ann = Text("Node = group element", font_size=18, color=BLUE).to_edge(DOWN, buff=0.8).shift(LEFT * 2)
        edge_ann = Text("Edge = apply generator", font_size=18, color=YELLOW).next_to(node_ann, RIGHT, buff=0.8)

        with self.voiceover(
            text="In a Cayley graph, each node represents one group element — think of it as a state of your data. Each edge corresponds to applying one generator from S."
        ) as tracker:
            self.play(Create(graph), run_time=tracker.duration * 0.5)
            self.play(FadeIn(node_ann), FadeIn(edge_ann), run_time=0.6)
            self.wait(tracker.duration * 0.15)

        # --- Sentence 5: BFS expansion ---
        self.play(FadeOut(node_ann), FadeOut(edge_ann), run_time=0.3)

        # BFS levels
        start = 0
        visited = {start}
        level_1 = set()
        for offset in [1, 3, 7, n_nodes - 1, n_nodes - 3, n_nodes - 7]:
            nb = (start + offset) % n_nodes
            if nb not in visited:
                level_1.add(nb)
                visited.add(nb)
        level_2 = set()
        for node in level_1:
            for offset in [1, 3, 7, n_nodes - 1, n_nodes - 3, n_nodes - 7]:
                nb = (node + offset) % n_nodes
                if nb not in visited:
                    level_2.add(nb)
                    visited.add(nb)
        level_3 = set(range(n_nodes)) - visited

        with self.voiceover(
            text="Now watch what happens when we do a breadth-first search. Starting from one node, level 1 reaches a few neighbors. Level 2 explodes outward. And by level 3, nearly the entire graph is covered!"
        ) as tracker:
            bfs_label = Text("BFS from node 0 → Exponential Expansion", font_size=22, color=GREEN).to_edge(DOWN, buff=0.5)
            self.play(FadeIn(bfs_label), run_time=0.4)

            # Level 0
            self.play(graph[start].animate.set_fill("#CD5C5C").scale(1.8), run_time=0.8)
            count_0 = Text("Level 0: 1 node", font_size=18, color="#CD5C5C").shift(RIGHT * 2.6 + UP * 1.2)
            self.play(FadeIn(count_0), run_time=0.3)

            # Level 1
            if level_1:
                self.play(*[graph[i].animate.set_fill("#F0E68C").scale(1.5) for i in level_1], run_time=1.2)
            count_1 = Text(f"Level 1: {len(level_1)} nodes", font_size=18, color="#F0E68C").next_to(count_0, DOWN, buff=0.25).align_to(count_0, LEFT)
            self.play(FadeIn(count_1), run_time=0.3)

            # Level 2
            if level_2:
                self.play(*[graph[i].animate.set_fill("#E67E22").scale(1.3) for i in level_2], run_time=1.2)
            count_2 = Text(f"Level 2: {len(level_2)} nodes", font_size=18, color="#E67E22").next_to(count_1, DOWN, buff=0.25).align_to(count_0, LEFT)
            self.play(FadeIn(count_2), run_time=0.3)

            # Level 3
            if level_3:
                self.play(*[graph[i].animate.set_fill("#2ECC71").scale(1.1) for i in level_3], run_time=1.2)
            count_3 = Text(f"Level 3: {len(level_3)} nodes", font_size=18, color="#2ECC71").next_to(count_2, DOWN, buff=0.25).align_to(count_0, LEFT)
            self.play(FadeIn(count_3), run_time=0.3)

            total_text = Text(f"→ All {n_nodes} nodes reached in 3 steps!", font_size=20, color=WHITE, weight=BOLD).next_to(count_3, DOWN, buff=0.4).align_to(count_0, LEFT)
            self.play(FadeIn(total_text), run_time=0.5)
            self.wait(0.5)

        # --- Sentence 6: Rapid mixing ---
        self.play(
            *[FadeOut(m) for m in [graph, bfs_label, count_0, count_1, count_2, count_3, total_text, cayley_title]],
            run_time=0.6
        )

        with self.voiceover(
            text="This exponential expansion is exactly the expander property. A short random walk on this graph can sample from the entire group efficiently."
        ) as tracker:
            mixing_text = VGroup(
                Text("Expander Property = Rapid Mixing", font_size=30, color=YELLOW, weight=BOLD),
                Text("Short random walk → sample from entire group", font_size=24, color=WHITE),
            ).arrange(DOWN, buff=0.3).shift(UP * 0.3)
            self.play(FadeIn(mixing_text), run_time=tracker.duration * 0.4)
            self.wait(tracker.duration * 0.5)

        self.play(FadeOut(mixing_text), run_time=0.5)
        self.wait(0.5)


print("All 5 VoiceoverScene classes defined")


In [ ]:
import subprocess, sys
from pathlib import Path

# ═══════════════════════════════════════════════════════════
# RENDER scenes 81–85
# ═══════════════════════════════════════════════════════════
scene_file = "/content/slides_81_85.py"
scenes_to_render = [
    "Scene01Title",
    "Scene02Groups",
    "Scene03GeneratingSets",
    "Scene04MinimumSize",
    "Scene05CayleyExpander",
]

output_paths_81_85 = []

for scene_name in scenes_to_render:
    print(f"Rendering {scene_name}...")
    result = subprocess.run(
        [
            sys.executable, "-m", "manim", "render",
            "-qh",                    # 1080p, 60fps
            "--disable_caching",
            scene_file,
            scene_name,
        ],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"  Error:\n{result.stderr[-800:]}")
    else:
        found = list(Path("media").rglob(f"{scene_name}.mp4"))
        if found:
            output_paths_81_85.append(str(found[0]))
            print(f"  {found[0]}")
        else:
            print(f"  Rendered OK but file not found")

print(f"\n Total: {len(output_paths_81_85)} videos")
for p in output_paths_81_85:
    print(f"  {p}")


## Display

In [ ]:
import subprocess
from pathlib import Path
from IPython.display import Video, display

list_file = "/content/concat_list_81_85.txt"
output_file = "/content/slide_81_85_voiceover.mp4"

with open(list_file, "w") as f:
    for p in output_paths_81_85:
        if Path(p).exists():
            f.write(f"file '{p}'\n")

cmd = [
    "ffmpeg", "-y",
    "-f", "concat",
    "-safe", "0",
    "-i", list_file,
    "-c:v", "libx264",
    "-preset", "medium",
    "-crf", "20",
    "-r", "60",
    "-pix_fmt", "yuv420p",
    "-c:a", "aac",
    "-b:a", "192k",
    "-movflags", "+faststart",
    output_file
]

result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode == 0 and Path(output_file).exists():
    probe = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries",
         "format=duration,bit_rate", "-show_entries",
         "stream=r_frame_rate,width,height,bit_rate",
         "-of", "default=noprint_wrappers=1",
         output_file],
        capture_output=True, text=True
    )
    print(f" Final: {output_file}")
    print(probe.stdout)
    display(Video(output_file, embed=True, width=854))
else:
    print(" Ghép thất bại")
    print(result.stderr[-500:])


# SLIDE 86–92

## Generative

In [ ]:
%%writefile /content/slides_86_92.py

# ══════════════════════════════════════════════════════════════
# FILE: slides_86_92.py — Self-contained with EdgeTTSService + VoiceoverScene
# ══════════════════════════════════════════════════════════════

import nest_asyncio
nest_asyncio.apply()

import asyncio
import hashlib
import random
import numpy as np
from pathlib import Path

from manim import *
from manim_voiceover import VoiceoverScene
from manim_voiceover.services.base import SpeechService

import edge_tts
C_PRIMARY = "#4FC3F7"      # Light blue - main concepts
C_SECONDARY = "#81C784"    # Green - solutions/good
C_ACCENT = "#FFD54F"       # Yellow - highlights
C_DANGER = "#E57373"       # Red - problems/hard
C_ORANGE = "#FFB74D"       # Orange - experiments
C_PURPLE = "#CE93D8"       # Purple - theory
C_BG_BOX = 0.08           # Fill opacity for boxes

np.random.seed(42)  # Reproducible visuals     # Purple - theory
# ─────────────────────────────────────────────
# EdgeTTSService
# ─────────────────────────────────────────────
VOICE = "en-US-GuyNeural"

def make_section_title(text, color=WHITE):
    title = Text(text, font_size=40, color=color, weight=BOLD)
    title.to_edge(UP, buff=0.6)
    underline = Line(
        title.get_left() + DOWN * 0.15,
        title.get_right() + DOWN * 0.15,
        color=color, stroke_width=2, stroke_opacity=0.6
    )
    return VGroup(title)


def make_badge(text, color, position):
    """Small rounded badge with text."""
    txt = Text(text, font_size=18, color=color, weight=BOLD)
    box = RoundedRectangle(
        width=txt.width + 0.4, height=txt.height + 0.25,
        color=color, fill_opacity=0.15, corner_radius=0.1, stroke_width=1.5
    )
    badge = VGroup(box, txt).move_to(position)
    return badge

class EdgeTTSService(SpeechService):
    def __init__(self, voice="en-US-GuyNeural", **kwargs):
        self.voice = voice
        super().__init__(**kwargs)

    def generate_from_text(self, text, cache_dir=None, path=None, **kwargs):
        if cache_dir is None:
            cache_dir = self.cache_dir
        Path(cache_dir).mkdir(parents=True, exist_ok=True)

        input_data = {"input_text": text, "service": "edge_tts", "voice": self.voice}

        cached = self.get_cached_result(input_data, cache_dir)
        if cached is not None:
            return cached

        audio_basename = hashlib.md5(text.encode()).hexdigest()
        audio_file = audio_basename + ".mp3"
        full_path = str(Path(cache_dir) / audio_file)

        communicate = edge_tts.Communicate(text, self.voice)
        loop = asyncio.get_event_loop()
        loop.run_until_complete(communicate.save(full_path))

        json_dict = {
            "input_text": text,
            "input_data": input_data,
            "original_audio": audio_file,
        }

        return json_dict


# ─────────────────────────────────────────────
# Scene 06: Applications Title
# ─────────────────────────────────────────────
class Scene06ApplicationsTitle(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        s6_title = Text("Applications of Expanders in ML", font_size=36, color=WHITE, weight=BOLD)
        s6_title.to_edge(UP, buff=0.8)

        with self.voiceover(
            text="Now that we understand expander graphs and random generating sets, let's explore how these tools are actually applied in geometric machine learning."
        ) as tracker:
            self.play(Write(s6_title), run_time=tracker.duration * 0.4)
            self.wait(tracker.duration * 0.5)

        # Animated roadmap: 3 boxes
        box_kw = {"width": 3.2, "height": 1.0, "stroke_width": 1.5, "corner_radius": 0.1, "fill_opacity": 0.12}

        box1 = RoundedRectangle(color=BLUE, **box_kw).shift(LEFT * 3.8 + DOWN * 0.8)
        txt1 = Text("Data\nAugmentation", font_size=22, color=BLUE).move_to(box1)

        box2 = RoundedRectangle(color=GREEN, **box_kw).shift(DOWN * 0.8)
        txt2 = Text("Approximate\nAveraging", font_size=22, color=GREEN).move_to(box2)

        box3 = RoundedRectangle(color=YELLOW, **box_kw).shift(RIGHT * 3.8 + DOWN * 0.8)
        txt3 = Text("Exact\nInvariance", font_size=22, color=YELLOW).move_to(box3)

        arr1 = Arrow(box1.get_right(), box2.get_left(), buff=0.1, color=WHITE, stroke_width=2, tip_length=0.15)
        arr2 = Arrow(box2.get_right(), box3.get_left(), buff=0.1, color=WHITE, stroke_width=2, tip_length=0.15)

        foundation = RoundedRectangle(width=9, height=0.7, color=RED, stroke_width=1.5, corner_radius=0.1, fill_opacity=0.1).shift(DOWN * 2.5)
        found_txt = Text("Foundation: Expander Graphs + O(log|G|) random generators", font_size=18, color=RED).move_to(foundation)
        connect_lines = VGroup(
            DashedLine(box1.get_bottom(), foundation.get_top() + LEFT * 3, color=WHITE, stroke_width=1),
            DashedLine(box2.get_bottom(), foundation.get_top(), color=WHITE, stroke_width=1),
            DashedLine(box3.get_bottom(), foundation.get_top() + RIGHT * 3, color=WHITE, stroke_width=1),
        )

        with self.voiceover(
            text="We will cover three major applications: data augmentation, approximate group averaging, and learning with exact invariances in polynomial time."
        ) as tracker:
            self.play(FadeIn(box1), FadeIn(txt1), run_time=0.5)
            self.play(GrowArrow(arr1), run_time=0.3)
            self.play(FadeIn(box2), FadeIn(txt2), run_time=0.5)
            self.play(GrowArrow(arr2), run_time=0.3)
            self.play(FadeIn(box3), FadeIn(txt3), run_time=0.5)
            self.play(Create(connect_lines), FadeIn(foundation), FadeIn(found_txt), run_time=0.8)
            self.wait(tracker.duration * 0.1)

        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.5)
        self.wait(0.3)


# ─────────────────────────────────────────────
# Scene 07: Computational Complexity
# ─────────────────────────────────────────────
class Scene07ComputationalComplexity(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        s7_title = Text("Learning with Exact Invariances", font_size=36, color=WHITE, weight=BOLD)
        s7_title.to_edge(UP, buff=0.5)

        with self.voiceover(
            text="A fundamental question in geometric machine learning is: can we learn an exactly invariant function efficiently?"
        ) as tracker:
            self.play(Write(s7_title), run_time=tracker.duration * 0.4)
            self.wait(tracker.duration * 0.5)

        # Constrained optimization
        with self.voiceover(
            text="In Sobolev regression, learning with invariances is equivalent to solving a constrained optimization problem."
        ) as tracker:
            s7_text1 = Text("Constrained Optimization", font_size=28, color=YELLOW).next_to(s7_title, DOWN, buff=0.5)
            self.play(Write(s7_text1), run_time=tracker.duration * 0.5)
            self.wait(tracker.duration * 0.4)

        self.play(FadeOut(s7_text1), run_time=0.3)

        # Show optimization problem
        s7_obj = MathTex(r"\arg\min_{\theta}", r"\Big\{", r"L(\theta)", r"+", r"\eta \|\theta\|^2", r"\Big\}", font_size=36, color=WHITE)
        s7_obj.next_to(s7_title, DOWN, buff=0.6)
        s7_constraint = MathTex(r"\text{s.t. } \rho(g) \theta = \theta ,", r"\;\; \forall g \in G", font_size=34, color=WHITE)
        s7_constraint.next_to(s7_obj, DOWN, buff=0.3)

        with self.voiceover(
            text="The optimization takes the form: argmin over theta of L of theta plus eta times the squared norm of theta, subject to the constraint that rho of g times theta equals theta for all g in G."
        ) as tracker:
            self.play(Write(s7_obj), run_time=tracker.duration * 0.4)
            self.play(Write(s7_constraint), run_time=tracker.duration * 0.4)
            self.wait(tracker.duration * 0.15)

        # Highlight problematic constraint
        with self.voiceover(
            text="This is convex, which is good. But the constraint must hold for every element g in the entire group G, which can be astronomically large."
        ) as tracker:
            highlight_G = SurroundingRectangle(s7_constraint[1], color=RED, buff=0.08, stroke_width=2)
            size_note = MathTex(r"|G| = 3 \times 10^{64} \text{ constraints!}", font_size=24, color=RED).next_to(highlight_G, DOWN, buff=0.3)
            self.play(Create(highlight_G), FadeIn(size_note), run_time=tracker.duration * 0.4)
            self.wait(tracker.duration * 0.5)

        # Representation property
        self.play(FadeOut(size_note), FadeOut(highlight_G), run_time=0.3)

        rep_eq = MathTex(r"\rho(gh) = \rho(g)\rho(h)", font_size=34, color=GREEN).next_to(s7_constraint, DOWN, buff=0.5)
        chain_visual = VGroup(
            MathTex(r"s_1", font_size=28, color=YELLOW),
            MathTex(r"\cdot", font_size=28, color=WHITE),
            MathTex(r"s_2", font_size=28, color=YELLOW),
            MathTex(r"\cdot", font_size=28, color=WHITE),
            MathTex(r"s_3", font_size=28, color=YELLOW),
            MathTex(r"=", font_size=28, color=WHITE),
            MathTex(r"g", font_size=28, color=GREEN),
        ).arrange(RIGHT, buff=0.15).next_to(rep_eq, DOWN, buff=0.3)

        with self.voiceover(
            text="The crucial observation: since rho is a group representation, meaning rho of g h equals rho of g times rho of h, we can replace the full group constraint with a constraint over just a generating set S."
        ) as tracker:
            self.play(Write(rep_eq), run_time=tracker.duration * 0.4)
            self.play(FadeIn(chain_visual), run_time=tracker.duration * 0.3)
            self.wait(tracker.duration * 0.2)

        # Transform constraint
        self.play(FadeOut(rep_eq), FadeOut(chain_visual), run_time=0.3)

        s7_new_constraint = MathTex(r"\text{s.t. } \rho(g) \theta = \theta", r",\;\; \forall g \in S", font_size=34, color=GREEN)
        s7_new_constraint.move_to(s7_constraint)

        with self.voiceover(
            text="Same objective, but now subject only to rho of g theta equals theta for g in S. This replacement is lossless."
        ) as tracker:
            self.play(Transform(s7_constraint, s7_new_constraint), run_time=tracker.duration * 0.4)
            free_badge = VGroup(
                RoundedRectangle(width=2.5, height=0.6, color=GREEN, fill_opacity=0.2, corner_radius=0.1),
                Text("Lossless!", font_size=22, color=GREEN),
            )
            free_badge[1].move_to(free_badge[0])
            free_badge.next_to(s7_constraint, RIGHT, buff=0.3)
            self.play(FadeIn(free_badge, scale=1.3), run_time=0.5)
            self.wait(tracker.duration * 0.3)

        # Final theorem
        self.play(*[FadeOut(m) for m in [s7_obj, s7_constraint, free_badge]], run_time=0.4)

        thm_box_content = VGroup(
            Text("ICML 2025 Spotlight", font_size=18, color=WHITE),
            MathTex(r"\text{Exactly invariant estimator in time } O\!\left((\log|G|)^3\right)", font_size=30, color=WHITE),
        ).arrange(DOWN, buff=0.2).next_to(s7_title, DOWN, buff=0.8)
        thm_box = SurroundingRectangle(thm_box_content, color=YELLOW, stroke_width=2, buff=0.25)
        thm_box.scale([1, 0.8, 1])
        bar_full = Rectangle(width=7, height=0.4, color=RED, fill_opacity=0.4, stroke_width=1).shift(DOWN * 1.2)
        bar_full_l = MathTex(r"|G| \text{ (naive)}", font_size=20, color=RED).next_to(bar_full, RIGHT, buff=0.15)
        bar_log = Rectangle(width=0.3, height=0.4, color=GREEN, fill_opacity=0.6, stroke_width=1).align_to(bar_full, LEFT).shift(DOWN * 1.8)
        bar_log_l = MathTex(r"(\log|G|)^3 \text{ (ours)}", font_size=20, color=GREEN).next_to(bar_log, RIGHT, buff=0.15)

        with self.voiceover(
            text="The theorem from ICML 2025 states: one can find an exactly invariant estimator in time big-O of log of |G|, cubed."
        ) as tracker:
            self.play(Write(thm_box_content[0]), Write(thm_box_content[1]), run_time=tracker.duration * 0.35)
            self.play(Create(thm_box), run_time=0.3)
            self.play(FadeIn(bar_full), FadeIn(bar_full_l), run_time=0.5)
            self.play(FadeIn(bar_log), FadeIn(bar_log_l), run_time=0.5)
            self.wait(tracker.duration * 0.1)

        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.5)
        self.wait(0.3)


# ─────────────────────────────────────────────
# Scene 08: Data Augmentation
# ─────────────────────────────────────────────
class Scene08DataAugmentation(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        s8_title = Text("Application 1: Data Augmentation", font_size=36, color=WHITE, weight=BOLD)
        s8_title.to_edge(UP, buff=0.5)

        # Sentence 0: standard augmentation
        orig_dot = Dot(ORIGIN, radius=0.15, color=WHITE)
        orig_label = MathTex(r"x", font_size=28, color=WHITE).next_to(orig_dot, DOWN, buff=0.15)

        with self.voiceover(
            text="The first application is data augmentation. Normally we augment by applying all g in G to each data point."
        ) as tracker:
            self.play(Write(s8_title), run_time=0.8)
            self.play(FadeIn(orig_dot), FadeIn(orig_label), run_time=0.5)
            self.wait(tracker.duration * 0.3)

        # Sentence 1: full augmentation is prohibitive
        n_full = 40
        full_dots = VGroup()
        for i in range(n_full):
            angle = i * TAU / n_full
            r = 1.5 + 0.3 * np.random.random()
            pos = np.array([r * np.cos(angle), r * np.sin(angle), 0])
            full_dots.add(Dot(pos, radius=0.05, color=RED, fill_opacity=0.6))

        full_label = MathTex(r"\{g \cdot x : g \in G\}", font_size=24, color=RED).shift(DOWN * 2.5)
        too_many = Text("|G| = d! augmented copies!", font_size=20, color=RED).next_to(full_label, DOWN, buff=0.2)

        with self.voiceover(
            text="But full augmentation is prohibitive. For S d, the group has d factorial elements."
        ) as tracker:
            self.play(*[FadeIn(d, scale=0.3) for d in full_dots], FadeIn(full_label), run_time=tracker.duration * 0.5)
            self.play(FadeIn(too_many), run_time=0.4)
            self.wait(tracker.duration * 0.2)

        # Sentence 2: Can small subset suffice?
        cross = Cross(VGroup(full_dots), color=RED, stroke_width=3)

        with self.voiceover(
            text="Can a small subset S achieve the same statistical benefits?"
        ) as tracker:
            self.play(Create(cross), run_time=0.5)
            self.wait(tracker.duration * 0.5)

        self.play(FadeOut(full_dots), FadeOut(cross), FadeOut(full_label), FadeOut(too_many), run_time=0.4)

        # Sentence 3: kernel methods setting
        with self.voiceover(
            text="In classical density estimation using kernel methods, the answer is yes."
        ) as tracker:
            yes_text = Text("YES — in kernel methods / Sobolev spaces", font_size=26, color=GREEN)
            yes_text.next_to(orig_dot, DOWN, buff=1.0)
            self.play(FadeIn(yes_text, scale=1.1), run_time=tracker.duration * 0.5)
            self.wait(tracker.duration * 0.4)

        self.play(FadeOut(yes_text), FadeOut(orig_dot), FadeOut(orig_label), run_time=0.3)

        # Sentence 4: theorem
        small_dots = VGroup()
        n_small = 8
        for i in range(n_small):
            angle = i * TAU / n_small + 0.2
            r = 1.5
            pos = np.array([r * np.cos(angle), r * np.sin(angle), 0])
            small_dots.add(Dot(pos, radius=0.1, color=GREEN))

        small_label = MathTex(r"|S| = O(\log|G|) \text{ suffices!}", font_size=26, color=GREEN).shift(DOWN * 2.5)

        with self.voiceover(
            text="The theorem proves: only O of log |G| random elements suffice for the full statistical gain."
        ) as tracker:
            new_orig = Dot(ORIGIN, radius=0.12, color=WHITE)
            self.play(FadeIn(new_orig), run_time=0.3)
            self.play(*[GrowFromCenter(d) for d in small_dots], run_time=tracker.duration * 0.35)
            arrows_small = VGroup(*[
                Arrow(ORIGIN, d.get_center(), buff=0.2, color=GREEN, stroke_width=1.5, tip_length=0.1)
                for d in small_dots
            ])
            self.play(Create(arrows_small), FadeIn(small_label), run_time=tracker.duration * 0.35)
            self.wait(tracker.duration * 0.2)

        # Sentence 5: exponential improvement
        self.play(FadeOut(new_orig), FadeOut(small_dots), FadeOut(arrows_small), FadeOut(small_label), run_time=0.4)

        thm_line1 = MathTex(r"|S_d| \approx \exp(d \log d)", font_size=32, color=RED).shift(UP * 0.3)
        thm_line2 = MathTex(r"\text{Needed: } |S| = O(d \log d)", font_size=32, color=GREEN).next_to(thm_line1, DOWN, buff=0.3)

        bar_exp = Rectangle(width=7, height=0.45, color=RED, fill_opacity=0.35, stroke_width=1.5).next_to(thm_line2, DOWN, buff=0.5)
        bar_exp_l = Text("exp(d log d)", font_size=18, color=RED).move_to(bar_exp)
        bar_log = Rectangle(width=0.5, height=0.45, color=GREEN, fill_opacity=0.5, stroke_width=1.5).align_to(bar_exp, LEFT).next_to(bar_exp, DOWN, buff=0.2)
        bar_log_l = Text("d log d", font_size=18, color=GREEN).next_to(bar_log, RIGHT, buff=0.15)
        exp_imp = Text("Exponential Improvement!", font_size=26, color=YELLOW, weight=BOLD).next_to(bar_log, DOWN, buff=0.4)

        with self.voiceover(
            text="For S d, group size is about exp of d log d, but only O of d log d random augmentations are needed. An exponential improvement."
        ) as tracker:
            self.play(Write(thm_line1), run_time=0.8)
            self.play(Write(thm_line2), run_time=0.8)
            self.play(FadeIn(bar_exp), FadeIn(bar_exp_l), run_time=0.5)
            self.play(FadeIn(bar_log), FadeIn(bar_log_l), run_time=0.5)
            self.play(FadeIn(exp_imp, scale=1.2), run_time=0.5)
            self.wait(tracker.duration * 0.1)

        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.5)
        self.wait(0.3)


# ─────────────────────────────────────────────
# Scene 09: Approximate Averaging
# ─────────────────────────────────────────────
class Scene09ApproxAveraging(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        s9_title = Text("Application 2: Approximate Averaging", font_size=36, color=WHITE, weight=BOLD)
        s9_title.to_edge(UP, buff=0.5)

        # Sentence 0: full vs partial visual
        wheel_center = LEFT * 3 + DOWN * 0.3
        wheel_radius = 1.2
        wheel_circle = Circle(radius=wheel_radius, color=RED, stroke_width=2).move_to(wheel_center)
        wheel_label = Text("Full Avg", font_size=20, color=RED).next_to(wheel_circle, DOWN, buff=0.2)
        wheel_sublabel = Text("(|G| terms)", font_size=16, color=RED).next_to(wheel_label, DOWN, buff=0.05)

        ticks_full = VGroup()
        for i in range(24):
            angle = i * TAU / 24
            start = wheel_center + wheel_radius * 0.85 * np.array([np.cos(angle), np.sin(angle), 0])
            end = wheel_center + wheel_radius * np.array([np.cos(angle), np.sin(angle), 0])
            ticks_full.add(Line(start, end, color=RED, stroke_width=1.5))

        partial_center = RIGHT * 3 + DOWN * 0.3
        partial_circle = Circle(radius=wheel_radius, color=GREEN, stroke_width=2).move_to(partial_center)
        partial_label = Text("Partial Avg", font_size=20, color=GREEN).next_to(partial_circle, DOWN, buff=0.2)
        partial_sublabel = Text("(|S| terms)", font_size=16, color=GREEN).next_to(partial_label, DOWN, buff=0.05)

        ticks_partial = VGroup()
        for i in range(6):
            angle = i * TAU / 6
            start = partial_center + wheel_radius * 0.8 * np.array([np.cos(angle), np.sin(angle), 0])
            end = partial_center + wheel_radius * np.array([np.cos(angle), np.sin(angle), 0])
            ticks_partial.add(Line(start, end, color=GREEN, stroke_width=2.5))

        approx_sign = MathTex(r"\approx", font_size=44, color=YELLOW)
        approx_sign.move_to((wheel_center + partial_center) / 2)

        with self.voiceover(
            text="The second application: approximate group averaging. We replace the full average over G with a partial average over S."
        ) as tracker:
            self.play(Write(s9_title), run_time=0.6)
            self.play(Create(wheel_circle), Create(ticks_full), FadeIn(wheel_label), FadeIn(wheel_sublabel), run_time=0.8)
            self.play(FadeIn(approx_sign), run_time=0.3)
            self.play(Create(partial_circle), Create(ticks_partial), FadeIn(partial_label), FadeIn(partial_sublabel), run_time=0.8)
            self.wait(tracker.duration * 0.1)

        # Sentence 1: how large must S be?
        with self.voiceover(
            text="The full average, sum over all g in G, is replaced by sum over just g in S. The key question: how large must S be?"
        ) as tracker:
            q_text = Text("How large must |S| be?", font_size=24, color=WHITE).to_edge(DOWN, buff=0.5)
            self.play(FadeIn(q_text), run_time=0.6)
            self.wait(tracker.duration * 0.5)

        # Sentence 2: uniform bound needed
        self.play(
            FadeOut(wheel_circle), FadeOut(ticks_full), FadeOut(wheel_label), FadeOut(wheel_sublabel),
            FadeOut(partial_circle), FadeOut(ticks_partial), FadeOut(partial_label), FadeOut(partial_sublabel),
            FadeOut(approx_sign), FadeOut(q_text),
            run_time=0.4
        )

        with self.voiceover(
            text="We need a uniform bound because f changes during training."
        ) as tracker:
            uniform_note = VGroup(
                Text("f changes during training", font_size=26, color=WHITE),
                MathTex(r"\Rightarrow", font_size=30, color=YELLOW),
                Text("need uniform bound over all f", font_size=26, color=YELLOW),
            ).arrange(RIGHT, buff=0.3).next_to(s9_title, DOWN, buff=0.8)
            self.play(FadeIn(uniform_note), run_time=tracker.duration * 0.5)
            self.wait(tracker.duration * 0.4)

        # Sentence 3: Theorem
        self.play(FadeOut(uniform_note), run_time=0.3)

        thm_label = Text("Theorem", font_size=28, color=YELLOW, weight=BOLD).next_to(s9_title, DOWN, buff=0.6)
        lhs = MathTex(
            r"\sup_{\|f\| \leq 1}",
            r"\left\|",
            r"\text{FullAvg} - \text{PartialAvg}",
            r"\right\|^2",
            font_size=30, color=WHITE
        ).next_to(thm_label, DOWN, buff=0.5)
        rhs = MathTex(
            r"\leq \; O\!\left(\frac{\log|G|}{|S|}\right)",
            font_size=36, color=GREEN
        ).next_to(lhs, DOWN, buff=0.5)

        thm_content = VGroup(thm_label, lhs, rhs)

        with self.voiceover(
            text="The theorem gives: the worst-case error is at most O of log |G| divided by |S|. Error shrinks as S grows."
        ) as tracker:
            self.play(Write(thm_label), run_time=0.4)
            self.play(Write(lhs), run_time=tracker.duration * 0.3)
            self.play(Write(rhs), run_time=tracker.duration * 0.25)
            thm_box = SurroundingRectangle(thm_content, color=YELLOW, stroke_width=1.5, buff=0.4, corner_radius=0.1)
            self.play(Create(thm_box), run_time=0.4)

            rhs_annotation = VGroup(
                MathTex(r"|S| \uparrow", font_size=26, color=GREEN),
                MathTex(r"\Rightarrow", font_size=24, color=WHITE),
                MathTex(r"\text{error} \downarrow", font_size=26, color=GREEN),
            ).arrange(RIGHT, buff=0.15).next_to(thm_box, RIGHT, buff=0.25)
            self.play(FadeIn(rhs_annotation), run_time=0.5)
            self.wait(tracker.duration * 0.1)

        # Sentence 4: Proof technique
        with self.voiceover(
            text="The proof requires Fourier analysis on groups — classical large deviation theory is insufficient here."
        ) as tracker:
            self.play(FadeOut(rhs_annotation), run_time=0.3)
            proof_note = VGroup(
                Text("Proof: Fourier analysis on groups", font_size=22, color=BLUE),
                Text("(classical large deviation ≠ sufficient)", font_size=18, color=WHITE),
            ).arrange(DOWN, buff=0.1).next_to(thm_box, DOWN, buff=0.4)
            self.play(FadeIn(proof_note), run_time=tracker.duration * 0.5)
            self.wait(tracker.duration * 0.4)

        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.6)
        self.wait(0.3)


# ─────────────────────────────────────────────
# Scene 10: Toy Experiment
# ─────────────────────────────────────────────
class Scene10ToyExperiment(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        s10_title = Text("Experiment: Sign-Flip Group", font_size=36, color=WHITE, weight=BOLD)
        s10_title.to_edge(UP, buff=0.5)

        # Sentence 0: setup
        with self.voiceover(
            text="Let's see a concrete experiment. Task: predict absolute value using a 3-layer ReLU net. The sign-flip group has 2 to the 20 elements, about one million."
        ) as tracker:
            self.play(Write(s10_title), run_time=0.6)
            vec_orig = MathTex(r"x = [+, +, -, +, -, \ldots]", font_size=28, color=WHITE).next_to(s10_title, DOWN, buff=0.4)
            vec_flip = MathTex(r"g \cdot x = [-, +, +, -, +, \ldots]", font_size=28, color=YELLOW).next_to(vec_orig, DOWN, buff=0.2)
            group_size = MathTex(r"|G| = 2^{20} \approx 10^6", font_size=30, color=RED).next_to(vec_flip, DOWN, buff=0.3)
            self.play(Write(vec_orig), run_time=0.5)
            self.play(Write(vec_flip), run_time=0.5)
            self.play(Write(group_size), run_time=0.6)
            self.wait(tracker.duration * 0.1)

        self.play(FadeOut(vec_orig), FadeOut(vec_flip), FadeOut(group_size), run_time=0.3)

        # Sentence 1 & 2: Left plot
        ax_left = Axes(
            x_range=[0, 7, 1], y_range=[0.014, 0.028, 0.004],
            x_length=4.0, y_length=2.6,
            axis_config={"color": WHITE, "font_size": 16, "include_ticks": True},
            tips=False
        ).shift(LEFT * 3 + DOWN * 0.8)

        x_tick_vals = [1, 2, 4, 8, 16, 32, 64]
        x_tick_labels = VGroup()
        for i, val in enumerate(x_tick_vals):
            lbl = Text(str(val), font_size=12, color=WHITE)
            lbl.next_to(ax_left.c2p(i + 0.5, 0.014), DOWN, buff=0.08)
            x_tick_labels.add(lbl)

        x_title_l = Text("|S|", font_size=16, color=WHITE).next_to(ax_left, DOWN, buff=0.35)
        y_title_l = Text("Loss", font_size=16, color=WHITE).next_to(ax_left, LEFT, buff=0.1).rotate(PI / 2)
        plot_title_l = Text("Final Test Loss vs |S|", font_size=18, color=WHITE).next_to(ax_left, UP, buff=0.1)

        y_vals = [0.027, 0.0235, 0.020, 0.018, 0.0165, 0.016, 0.0155]
        points_l = [ax_left.c2p(i + 0.5, y) for i, y in enumerate(y_vals)]
        curve_l = VMobject(color=BLUE, stroke_width=2.5)
        curve_l.set_points_smoothly(points_l)

        dot_32 = Dot(ax_left.c2p(4.5, 0.0165), color=RED, radius=0.1)
        label_32 = Text("32", font_size=14, color=RED, weight=BOLD).next_to(dot_32, UR, buff=0.05)
        plateau = DashedLine(ax_left.c2p(3.5, 0.016), ax_left.c2p(6.5, 0.016), color=WHITE, stroke_width=1.5)

        with self.voiceover(
            text="The left plot shows test loss versus number of random sign flips used."
        ) as tracker:
            self.play(Create(ax_left), FadeIn(x_tick_labels), FadeIn(x_title_l), FadeIn(y_title_l), FadeIn(plot_title_l), run_time=tracker.duration * 0.6)
            self.play(Create(curve_l), run_time=tracker.duration * 0.3)

        with self.voiceover(
            text="Notice how loss drops sharply and plateaus around |S| equals 32."
        ) as tracker:
            self.play(FadeIn(dot_32), FadeIn(label_32), Create(plateau), run_time=tracker.duration * 0.5)
            self.wait(tracker.duration * 0.4)

        # Sentence 3: Right plot
        ax_right = Axes(
            x_range=[0, 500, 100], y_range=[0, 0.6, 0.1],
            x_length=4.0, y_length=2.6,
            axis_config={"color": WHITE, "font_size": 16},
            tips=False
        ).shift(RIGHT * 3 + DOWN * 0.8)

        x_title_r = Text("Epoch", font_size=16, color=WHITE).next_to(ax_right, DOWN, buff=0.3)
        y_title_r = Text("Loss", font_size=16, color=WHITE).next_to(ax_right, LEFT, buff=0.1).rotate(PI / 2)
        plot_title_r = Text("Training Curve Comparison", font_size=18, color=WHITE).next_to(ax_right, UP, buff=0.1)

        curve_full = ax_right.plot(lambda t: 0.55 * np.exp(-0.008 * t) + 0.018, x_range=[1, 500], color=BLUE, stroke_width=2.5)
        curve_s32 = ax_right.plot(lambda t: 0.50 * np.exp(-0.007 * t) + 0.020, x_range=[1, 500], color=ORANGE, stroke_width=2.5)

        legend = VGroup(
            VGroup(Line(ORIGIN, RIGHT * 0.5, color=BLUE, stroke_width=3), Text("Full |G|", font_size=14, color=BLUE)).arrange(RIGHT, buff=0.1),
            VGroup(Line(ORIGIN, RIGHT * 0.5, color=ORANGE, stroke_width=3), Text("|S|=32", font_size=14, color=ORANGE)).arrange(RIGHT, buff=0.1),
        ).arrange(DOWN, aligned_edge=LEFT, buff=0.1).next_to(ax_right, RIGHT, buff=0.2).shift(UP * 0.5)

        with self.voiceover(
            text="The right plot shows loss over training epochs. Orange, using only 32 flips, closely tracks the blue full-group baseline across all epochs."
        ) as tracker:
            self.play(Create(ax_right), FadeIn(x_title_r), FadeIn(y_title_r), FadeIn(plot_title_r), run_time=0.6)
            self.play(Create(curve_full), run_time=tracker.duration * 0.3)
            self.play(Create(curve_s32), FadeIn(legend), run_time=tracker.duration * 0.3)
            self.wait(tracker.duration * 0.15)

        # Sentence 4: takeaway
        with self.voiceover(
            text="Key takeaway: just 32 elements from a group of one million gives excellent approximation, with a single random draw."
        ) as tracker:
            takeaway = VGroup(
                MathTex(r"|S| = 32", font_size=28, color=ORANGE),
                Text("from", font_size=20, color=WHITE),
                MathTex(r"|G| = 10^6", font_size=28, color=BLUE),
                MathTex(r"\checkmark", font_size=36, color=GREEN),
            ).arrange(RIGHT, buff=0.2).to_edge(DOWN, buff=0.4)
            self.play(FadeIn(takeaway, shift=UP * 0.2), run_time=tracker.duration * 0.4)
            self.wait(tracker.duration * 0.5)

        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.5)
        self.wait(0.3)


# ─────────────────────────────────────────────
# Scene 11: Exponential Separation
# ─────────────────────────────────────────────
class Scene11ExponentialSeparation(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        s11_title = Text("Exact vs Approximate Symmetry", font_size=36, color=WHITE, weight=BOLD)
        s11_title.to_edge(UP, buff=0.5)

        with self.voiceover(
            text="Now a remarkable negative result. "
                 "What if we demand exact equality instead of just approximation?"
        ) as tracker:
            self.play(Write(s11_title), run_time=0.8)
            self.wait(tracker.duration - 1.0)

        # --- The exact condition ---
        exact_eq = MathTex(
            r"\frac{1}{|G|}\sum_{g \in G} f(g \cdot x)",
            r"\;=\;",
            r"\frac{1}{|S|}\sum_{g \in S} \alpha_g\, f(g \cdot x)",
            font_size=26, color=WHITE
        ).next_to(s11_title, DOWN, buff=0.5)

        forall_note = MathTex(
            r"\forall\, f \in \mathcal{F}",
            font_size=28, color=YELLOW
        ).next_to(exact_eq, DOWN, buff=0.2)

        with self.voiceover(
            text="The condition: the full group average must EXACTLY equal a weighted partial average, "
                 "for ALL functions in a given class F."
        ) as tracker:
            self.play(Write(exact_eq), run_time=1.0)
            self.play(FadeIn(forall_note), run_time=0.4)
            self.wait(tracker.duration - 1.6)

        self.play(FadeOut(exact_eq), FadeOut(forall_note), run_time=0.3)

        # --- Separation diagram: two paths diverging ---
        start_dot = Dot(UP * 1.5, radius=0.12, color=WHITE)
        start_label = Text("Symmetrize f", font_size=20, color=WHITE).next_to(start_dot, UP, buff=0.15)

        # Left path: EXACT (hard)
        left_path = Line(start_dot.get_center(), LEFT * 3.5 + DOWN * 1.0, color=RED, stroke_width=3)
        left_box = RoundedRectangle(
            width=3.2, height=2.0, color=RED,
            fill_opacity=0.08, corner_radius=0.15, stroke_width=2
        ).move_to(LEFT * 3.5 + DOWN * 2.2)
        left_content = VGroup(
            Text("EXACT", font_size=22, color=RED, weight=BOLD),
            MathTex(r"|S| = \Omega(|G|)", font_size=26, color=RED),
            Text("Linear in group size", font_size=16, color=WHITE),
            Text("HARD", font_size=20, color=RED, weight=BOLD),
        ).arrange(DOWN, buff=0.12).move_to(left_box)

        # Right path: APPROXIMATE (easy)
        right_path = Line(start_dot.get_center(), RIGHT * 3.5 + DOWN * 1.0, color=GREEN, stroke_width=3)
        right_box = RoundedRectangle(
            width=3.2, height=2.0, color=GREEN,
            fill_opacity=0.08, corner_radius=0.15, stroke_width=2
        ).move_to(RIGHT * 3.5 + DOWN * 2.2)
        right_content = VGroup(
            Text("APPROXIMATE", font_size=22, color=GREEN, weight=BOLD),
            MathTex(r"|S| = O(\log|G|)", font_size=26, color=GREEN),
            Text("Logarithmic", font_size=16, color=WHITE),
            Text("EASY", font_size=20, color=GREEN, weight=BOLD),
        ).arrange(DOWN, buff=0.12).move_to(right_box)

        # Gap indicator
        gap_arrow = DoubleArrow(
            left_box.get_right() + RIGHT * 0.1,
            right_box.get_left() + LEFT * 0.1,
            color=YELLOW, stroke_width=2.5, tip_length=0.15
        )
        gap_label = Text("EXPONENTIAL gap", font_size=18, color=YELLOW, weight=BOLD)
        gap_label.move_to(gap_arrow.get_center() + DOWN * 0.5)

        with self.voiceover(
            text="The theorem proves an exponential separation. "
                 "For polynomial function spaces, exact symmetry via averaging requires a subset of size LINEAR in the group — "
                 "essentially as hard as using the full group. "
                 "But approximate symmetry only needs logarithmic size. "
                 "The gap between them is exponential."
        ) as tracker:
            self.play(FadeIn(start_dot), FadeIn(start_label), run_time=0.4)
            self.play(Create(left_path), run_time=0.8)
            self.play(FadeIn(left_box), FadeIn(left_content), run_time=0.8)
            self.play(Create(right_path), run_time=0.8)
            self.play(FadeIn(right_box), FadeIn(right_content), run_time=0.8)
            self.play(GrowArrow(gap_arrow), FadeIn(gap_label), run_time=0.6)
            self.wait(max(0.1, tracker.duration - 4.4))

        # --- Punchline ---
        self.play(
            *[FadeOut(m) for m in [
                start_dot, start_label, left_path, left_box, left_content,
                right_path, right_box, right_content, gap_arrow, gap_label
            ]],
            run_time=0.5
        )

        # Three-row summary
        summary_title = Text("Three-Part Corollary", font_size=26, color=YELLOW, weight=BOLD)
        summary_title.next_to(s11_title, DOWN, buff=0.5)

        rows_data = [
            ("Exact symmetry (averaging)", r"|S| = \Omega(|G|)", "HARD", RED),
            ("Approximate symmetry", r"|S| = O(\log|G|)", "EASY", GREEN),
            ("Full statistical gain", r"|S| = O(\log|G|)", "EASY", GREEN),
        ]

        rows = VGroup()
        for i, (desc, formula, difficulty, color) in enumerate(rows_data):
            badge_txt = Text(difficulty, font_size=16, color=color, weight=BOLD)
            badge_box = RoundedRectangle(
                width=badge_txt.width + 0.3, height=badge_txt.height + 0.2,
                color=color, fill_opacity=0.15, corner_radius=0.08, stroke_width=1.5
            )
            badge_txt.move_to(badge_box)
            badge = VGroup(badge_box, badge_txt)

            row = VGroup(
                Text(desc, font_size=20, color=WHITE),
                MathTex(formula, font_size=24, color=color),
                badge,
            ).arrange(RIGHT, buff=0.4)
            rows.add(row)
        rows.arrange(DOWN, buff=0.25).next_to(summary_title, DOWN, buff=0.4)

        with self.voiceover(
            text="The three-part corollary: "
                 "exact symmetry is hard, needing linear complexity. "
                 "Approximate symmetry is easy, needing only log. "
                 "And crucially, the full statistical benefit is also achievable with log complexity."
        ) as tracker:
            self.play(Write(summary_title), run_time=0.4)
            for row in rows:
                self.play(FadeIn(row, shift=RIGHT * 0.2), run_time=0.7)
            self.wait(max(0.1, tracker.duration - 2.8))

        # Final punchline
        with self.voiceover(
            text="The punchline: you lose almost nothing by relaxing from exact to approximate symmetry, "
                 "but you gain an exponential speedup."
        ) as tracker:
            punchline = MathTex(
                r"\text{Exact} \;\to\; \text{Approximate: zero cost, exponential gain}",
                font_size=28, color=YELLOW
            ).next_to(rows, DOWN, buff=0.5)
            self.play(FadeIn(punchline, scale=1.1), run_time=0.8)
            self.wait(tracker.duration - 1.0)

        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.6)
        self.wait(0.3)


# ─────────────────────────────────────────────
# Scene 12: Summary
# ─────────────────────────────────────────────
class Scene12Summary(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        s12_title = Text("Summary: Expanders in GeoML", font_size=36, color=WHITE, weight=BOLD)
        s12_title.to_edge(UP, buff=0.5)

        # Sentence 0
        with self.voiceover(
            text="Let's summarize. Expander graphs and random Cayley graphs provide powerful tools directly relevant to machine learning."
        ) as tracker:
            self.play(Write(s12_title), run_time=tracker.duration * 0.3)
            self.wait(tracker.duration * 0.6)

        # Sentence 1: key message
        key_msg = VGroup(
            Text("Random subsets of size", font_size=26, color=WHITE),
            MathTex(r"O(\log|G|)", font_size=36, color=GREEN),
            Text("are almost as good as the full group", font_size=26, color=WHITE),
        ).arrange(RIGHT, buff=0.2).next_to(s12_title, DOWN, buff=0.5)
        tight_note = Text("— and this is provably tight.", font_size=22, color=YELLOW).next_to(key_msg, DOWN, buff=0.2)

        with self.voiceover(
            text="The key message: random subsets of size O of log |G| are almost as good as the full group, and this is provably tight."
        ) as tracker:
            self.play(FadeIn(key_msg, shift=UP * 0.2), run_time=tracker.duration * 0.4)
            self.play(FadeIn(tight_note), run_time=0.5)
            self.wait(tracker.duration * 0.3)

        # Sentence 2: hub and spokes + references
        self.play(FadeOut(key_msg), FadeOut(tight_note), run_time=0.4)

        center = ORIGIN
        hub = Circle(radius=0.85, color=YELLOW, fill_opacity=0.15, stroke_width=2.5).move_to(center)
        hub_text = Text("Expander\nGraphs", font_size=22, color=YELLOW).move_to(hub)

        spokes_data = [
            ("Fourier Analysis\non Groups", BLUE, 2.3),
            ("Representation\nTheory", GREEN, 0.85),
            ("Optimization\nunder Symmetry", RED, 3.9),
            ("ML\nApplications", ORANGE, 5.4),
        ]

        spoke_distance = 3.0  # distance from center to box center
        spoke_nodes = VGroup()
        spoke_lines = VGroup()

        for txt, col, angle in spokes_data:
            # Box position
            box_pos = center + spoke_distance * np.array([np.cos(angle), np.sin(angle), 0])

            node_box = RoundedRectangle(
                width=2.6, height=1.0, color=col,
                fill_opacity=0.08, corner_radius=0.12, stroke_width=2
            ).move_to(box_pos)
            node_text = Text(txt, font_size=18, color=col, line_spacing=0.9).move_to(node_box)

            # Line START: point on circle boundary at this angle
            start_p = hub.point_at_angle(angle)

            # Line END: closest point on box edge toward hub center
            # Use the direction from box center toward hub center
            direction_to_hub = center - box_pos
            direction_to_hub_norm = direction_to_hub / np.linalg.norm(direction_to_hub)

            # Find intersection with box boundary
            # Approximate: use half-width and half-height to find edge point
            hw = node_box.width / 2
            hh = node_box.height / 2
            dx, dy = direction_to_hub_norm[0], direction_to_hub_norm[1]

            # Scale factor to reach edge
            if abs(dx) < 1e-6:
                t = hh / abs(dy) if abs(dy) > 1e-6 else 1.0
            elif abs(dy) < 1e-6:
                t = hw / abs(dx)
            else:
                tx = hw / abs(dx)
                ty = hh / abs(dy)
                t = min(tx, ty)

            end_p = box_pos + t * direction_to_hub_norm

            line = Line(start_p, end_p, color=col, stroke_width=2, stroke_opacity=0.7)

            spoke_nodes.add(VGroup(node_box, node_text))
            spoke_lines.add(line)

        with self.voiceover(
            text="The techniques involve Fourier analysis on groups, expander graph theory, and optimization under symmetry constraints."
        ) as tracker:
            self.play(FadeIn(hub), FadeIn(hub_text), run_time=0.5)
            for i in range(4):
                self.play(Create(spoke_lines[i]), FadeIn(spoke_nodes[i]), run_time=tracker.duration * 0.15)
            self.wait(tracker.duration * 0.1)

        # Fade out diagram, show references (silent)
        self.play(FadeOut(hub), FadeOut(hub_text), FadeOut(spoke_nodes), FadeOut(spoke_lines), run_time=0.5)

        # References — displayed silently
        ref_title = Text("References", font_size=18, color=WHITE)
        refs = VGroup(
            Text("[1] Soleymani et al. — Learning with Exact Invariances", font_size=17, color=WHITE),
            Text("    in Polynomial Time. ICML 2025 (Spotlight)", font_size=17, color=WHITE),
            Text("[2] Tahmasebi, Weber, Jegelka — Data Augmentation:", font_size=17, color=WHITE),
            Text("    A Fourier Analysis Perspective. COLT 2026", font_size=17, color=WHITE),
            Text("[3] Tahmasebi, Weber — Approximate Symmetry Is Exp.", font_size=17, color=WHITE),
            Text("    Easier than Exact Symmetry. ICLR 2025", font_size=17, color=WHITE),
            Text("[4] Alon, Roichman — Random Cayley Graphs. 1994", font_size=17, color=WHITE),
        ).arrange(DOWN, aligned_edge=LEFT, buff=0.08)
        ref_group = VGroup(ref_title, refs).arrange(DOWN, buff=0.15).move_to(ORIGIN)
        if ref_group.width > 7.5:
            ref_group.scale_to_fit_width(7.5)

        with self.voiceover(
            text="Here are the key references for further reading."
        ) as tracker:
            self.play(FadeIn(ref_group, shift=UP * 0.15), run_time=1.0)
            self.wait(2.0)

        self.play(*[FadeOut(m) for m in self.mobjects], run_time=1.0)
        self.wait(0.5)


In [ ]:
import subprocess, sys
from pathlib import Path

scene_file = "/content/slides_86_92.py"
scenes_to_render = [
    "Scene06ApplicationsTitle",
    "Scene07ComputationalComplexity",
    "Scene08DataAugmentation",
    "Scene09ApproxAveraging",
    "Scene10ToyExperiment",
    "Scene11ExponentialSeparation",
    "Scene12Summary",
]

output_paths_86_92 = []

for scene_name in scenes_to_render:
    print(f"Rendering {scene_name}...")
    result = subprocess.run(
        [
            sys.executable, "-m", "manim", "render",
            "-qh",                    # 480p, 15fps
            # BỎ --disable_caching để dùng cache TTS!
            scene_file,
            scene_name,
        ],
        capture_output=True, text=True,
        timeout=300  # 5 phút timeout mỗi scene
    )
    if result.returncode != 0:
        print(f"  Error:\n{result.stderr[-800:]}")
    else:
        found = list(Path("media").rglob(f"{scene_name}.mp4"))
        if found:
            output_paths_86_92.append(str(found[0]))
            print(f"  {found[0]}")
        else:
            print(f"  Rendered OK but file not found")

print(f"\nTotal: {len(output_paths_86_92)} videos")
for p in output_paths_86_92:
    print(f"  {p}")


## Display

In [ ]:
import subprocess
from pathlib import Path
from IPython.display import Video, display

list_file = "/content/concat_list_86_92.txt"
output_file = "/content/slide_86_92_voiceover.mp4"

with open(list_file, "w") as f:
    for p in output_paths_86_92:
        if Path(p).exists():
            f.write(f"file '{p}'\n")

cmd = [
    "ffmpeg", "-y",
    "-f", "concat",
    "-safe", "0",
    "-i", list_file,
    "-c:v", "libx264",
    "-preset", "medium",
    "-crf", "20",
    "-r", "60",
    "-pix_fmt", "yuv420p",
    "-c:a", "aac",
    "-b:a", "192k",
    "-movflags", "+faststart",
    output_file
]

result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode == 0 and Path(output_file).exists():
    probe = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries",
         "format=duration,bit_rate", "-show_entries",
         "stream=r_frame_rate,width,height,bit_rate",
         "-of", "default=noprint_wrappers=1",
         output_file],
        capture_output=True, text=True
    )
    print(f"Final: {output_file}")
    print(probe.stdout)
    display(Video(output_file, embed=True, width=854))
else:
    print("Ghép thất bại")
    print(result.stderr[-500:])


---

# SLIDE 93–101

## Generative

In [ ]:
%%writefile /content/slides_93_101.py

# ══════════════════════════════════════════════════════════════
# FILE: slides_93_101.py — Recent Results II: Symmetry Validation
# Style: VoiceoverScene + EdgeTTSService (same as slides_86_92.py)
# ══════════════════════════════════════════════════════════════

import nest_asyncio
nest_asyncio.apply()

import asyncio
import hashlib
import numpy as np
from pathlib import Path

from manim import *
from manim_voiceover import VoiceoverScene
from manim_voiceover.services.base import SpeechService

import edge_tts

# ─────────────────────────────────────────────
# EdgeTTSService (identical to slides_86_92.py)
# ─────────────────────────────────────────────
VOICE = "en-US-GuyNeural"


class EdgeTTSService(SpeechService):
    def __init__(self, voice="en-US-GuyNeural", **kwargs):
        self.voice = voice
        super().__init__(**kwargs)

    def generate_from_text(self, text, cache_dir=None, path=None, **kwargs):
        if cache_dir is None:
            cache_dir = self.cache_dir
        Path(cache_dir).mkdir(parents=True, exist_ok=True)

        input_data = {"input_text": text, "service": "edge_tts", "voice": self.voice}

        cached = self.get_cached_result(input_data, cache_dir)
        if cached is not None:
            return cached

        audio_basename = hashlib.md5(text.encode()).hexdigest()
        audio_file = audio_basename + ".mp3"
        full_path = str(Path(cache_dir) / audio_file)

        communicate = edge_tts.Communicate(text, self.voice)
        loop = asyncio.get_event_loop()
        loop.run_until_complete(communicate.save(full_path))

        json_dict = {
            "input_text": text,
            "input_data": input_data,
            "original_audio": audio_file,
        }

        return json_dict


# ═══════════════════════════════════════════════════════════════
# SCENE 13: Title — Symmetry Validation Introduction
# ═══════════════════════════════════════════════════════════════
class Scene13SymmetryValidationTitle(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title1 = Text("Recent Results II:", font_size=44, color=WHITE, weight=BOLD)
        title2 = Text("Symmetry Validation (Testing)", font_size=38, color=WHITE, weight=BOLD)
        title_group = VGroup(title1, title2).arrange(DOWN, buff=0.3).move_to(UP * 1.0)

        # Visual teaser: two distributions side by side (silhouettes)
        np.random.seed(42)
        ellipse_dots = VGroup()
        for _ in range(60):
            x = np.random.normal(0, 1.0)
            y = np.random.normal(0, 0.3)
            ellipse_dots.add(Dot([x, y, 0], radius=0.03, color=RED, fill_opacity=0.5))
        ellipse_dots.scale(0.6).move_to(LEFT * 2.5 + DOWN * 2.0)

        np.random.seed(7)
        ring_dots = VGroup()
        for _ in range(60):
            angle = np.random.uniform(0, TAU)
            r = np.random.normal(0.7, 0.08)
            ring_dots.add(Dot([r * np.cos(angle), r * np.sin(angle), 0], radius=0.03, color=GREEN, fill_opacity=0.5))
        ring_dots.scale(0.6).move_to(RIGHT * 2.5 + DOWN * 2.0)

        q_mark = MathTex(r"\stackrel{?}{=}", font_size=40, color=YELLOW).move_to(DOWN * 2.0)

        with self.voiceover(
            text="We now move to the second set of recent results, focusing on symmetry validation — specifically, testing whether a data distribution respects a given group symmetry."
        ) as tracker:
            self.play(Write(title1), run_time=tracker.duration * 0.25)
            self.play(Write(title2), run_time=tracker.duration * 0.2)
            self.play(
                FadeIn(ellipse_dots, scale=0.5),
                FadeIn(ring_dots, scale=0.5),
                FadeIn(q_mark),
                run_time=tracker.duration * 0.3
            )
            self.wait(tracker.duration * 0.2)

        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.5)
        self.wait(0.3)


# ═══════════════════════════════════════════════════════════════
# SCENE 14: Problem Setup — Distributional Symmetry
# ═══════════════════════════════════════════════════════════════
class Scene14DistributionalSymmetry(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title = Text("Testing Distributional Symmetry", font_size=36, color=WHITE, weight=BOLD)
        title.to_edge(UP, buff=0.5)

        # ── Sentence 1: Setup + rotation demo ──
        np.random.seed(42)
        left_center = LEFT * 3.0 + DOWN * 0.5
        right_center = RIGHT * 3.0 + DOWN * 0.5

        # Non-invariant: elliptical
        ellipse_dots = VGroup()
        for _ in range(120):
            x = np.random.normal(0, 1.2)
            y = np.random.normal(0, 0.35)
            ellipse_dots.add(Dot([x, y, 0], radius=0.028, color=RED, fill_opacity=0.6))
        ellipse_dots.move_to(left_center)

        # Invariant: ring
        np.random.seed(7)
        ring_dots = VGroup()
        for _ in range(120):
            angle = np.random.uniform(0, TAU)
            r = np.random.normal(1.0, 0.12)
            ring_dots.add(Dot([r * np.cos(angle), r * np.sin(angle), 0], radius=0.028, color=GREEN, fill_opacity=0.6))
        ring_dots.move_to(right_center)

        l_label = Text("Non-Invariant (elliptical)", font_size=18, color=RED).next_to(left_center, UP, buff=1.6)
        r_label = Text("Invariant (ring)", font_size=18, color=GREEN).next_to(right_center, UP, buff=1.6)

        with self.voiceover(
            text="Given i.i.d. data drawn from a distribution mu, we want to decide whether mu is invariant under a group, or not invariant. Consider 2D rotational invariance as an example. A non-invariant distribution looks elliptical, while an invariant one forms a perfect ring."
        ) as tracker:
            self.play(Write(title), run_time=tracker.duration * 0.1)
            self.play(
                FadeIn(ellipse_dots, scale=0.3), FadeIn(l_label),
                FadeIn(ring_dots, scale=0.3), FadeIn(r_label),
                run_time=tracker.duration * 0.2
            )
            # Rotate both to demonstrate
            rot_label = Text("Apply rotation g", font_size=22, color=YELLOW).to_edge(DOWN, buff=0.5)
            self.play(FadeIn(rot_label), run_time=0.4)
            self.play(
                Rotate(ellipse_dots, angle=PI / 3, about_point=left_center),
                Rotate(ring_dots, angle=PI / 3, about_point=right_center),
                run_time=tracker.duration * 0.25
            )
            # Annotations
            changed_txt = Text("Shape changed!", font_size=16, color=RED).next_to(left_center, DOWN, buff=1.6)
            same_txt = Text("Same shape!", font_size=16, color=GREEN).next_to(right_center, DOWN, buff=1.6)
            self.play(FadeIn(changed_txt), FadeIn(same_txt), run_time=0.5)
            self.wait(tracker.duration * 0.15)

        # ── Sentence 2: Applications ──
        self.play(*[FadeOut(m) for m in self.mobjects if m != title], run_time=0.4)

        app_title = Text("Why test symmetry?", font_size=28, color=YELLOW, weight=BOLD).next_to(title, DOWN, buff=0.5)

        box_data = [
            ("Exchangeability\nTesting", BLUE, LEFT * 3.5 + DOWN * 1.5),
            ("Verify ML\nAssumptions", GREEN, LEFT * 0.0 + DOWN * 1.5),
            ("Detect Subtle\nAsymmetries", ORANGE, RIGHT * 3.5 + DOWN * 1.5),
        ]
        boxes = VGroup()
        for txt, col, pos in box_data:
            box = RoundedRectangle(width=2.8, height=1.2, color=col, fill_opacity=0.1, corner_radius=0.1, stroke_width=1.5).move_to(pos)
            label = Text(txt, font_size=18, color=col).move_to(box)
            boxes.add(VGroup(box, label))

        example_note = Text("e.g. Are MRI images truly rotationally symmetric?", font_size=18, color=WHITE).shift(DOWN * 3.0)

        with self.voiceover(
            text="Applications include testing exchangeability, verifying symmetry assumptions in geometric machine learning — such as whether MRI images are truly rotationally symmetric — detecting subtle asymmetries that could bias models, and improving robustness and interpretability."
        ) as tracker:
            self.play(Write(app_title), run_time=0.5)
            for box in boxes:
                self.play(FadeIn(box, scale=0.9), run_time=tracker.duration * 0.15)
            self.play(FadeIn(example_note), run_time=0.5)
            self.wait(tracker.duration * 0.15)

        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.5)
        self.wait(0.3)


# ═══════════════════════════════════════════════════════════════
# SCENE 15: Null Hypothesis — H₀: μ is invariant
# ═══════════════════════════════════════════════════════════════
class Scene15NullHypothesis(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title = Text("Null Hypothesis H₀: μ is G-invariant", font_size=34, color=WHITE, weight=BOLD)
        title.to_edge(UP, buff=0.5)

        # ── Sentence 1: H0 definition + visual ──
        h0_eq = MathTex(r"H_0:", r"\;\mu = g \cdot \mu", r"\quad \forall\, g \in G", font_size=36, color=WHITE)
        h0_eq[0].set_color(YELLOW)
        h0_eq.next_to(title, DOWN, buff=0.5)

        # Visual: ring of dots + rotation that preserves it
        np.random.seed(7)
        demo_center = DOWN * 1.5
        ring = VGroup()
        for _ in range(80):
            angle = np.random.uniform(0, TAU)
            r = np.random.normal(1.3, 0.1)
            ring.add(Dot([r * np.cos(angle), r * np.sin(angle), 0], radius=0.03, color=GREEN, fill_opacity=0.6))
        ring.move_to(demo_center)

        inv_label = Text("f(x,y) = φ(‖(x,y)‖₂)", font_size=20, color=GREEN).next_to(demo_center, DOWN, buff=1.8)
        explanation = Text("Density depends only on distance from origin", font_size=18, color=WHITE).next_to(inv_label, DOWN, buff=0.15)

        with self.voiceover(
            text="Under the null hypothesis H zero, the distribution mu is invariant: mu equals g dot mu for all g in G. For example, if G is the group of 2D rotations, invariance of the density f means f depends only on the norm of x."
        ) as tracker:
            self.play(Write(title), run_time=tracker.duration * 0.1)
            self.play(Write(h0_eq), run_time=tracker.duration * 0.15)
            self.play(FadeIn(ring, scale=0.5), run_time=tracker.duration * 0.15)
            # Rotate to show invariance
            self.play(Rotate(ring, angle=PI / 2, about_point=demo_center), run_time=tracker.duration * 0.15)
            self.play(Rotate(ring, angle=PI / 2, about_point=demo_center), run_time=tracker.duration * 0.15)
            self.play(FadeIn(inv_label), FadeIn(explanation), run_time=0.5)
            self.wait(tracker.duration * 0.1)

        # ── Sentence 2: Rotation matrix ──
        self.play(FadeOut(ring), FadeOut(inv_label), FadeOut(explanation), run_time=0.3)

        rot_matrix = MathTex(
            r"\begin{pmatrix} x \\ y \end{pmatrix}",
            r"\;\sim\;",
            r"\begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix}",
            r"\begin{pmatrix} x \\ y \end{pmatrix}",
            font_size=30, color=WHITE
        ).next_to(h0_eq, DOWN, buff=0.6)
        rot_matrix[2].set_color(YELLOW)

        forall = MathTex(r"\forall\, \theta \in [0, 2\pi]", font_size=28, color=YELLOW).next_to(rot_matrix, DOWN, buff=0.3)

        # Animated rotation arrow on circle
        circle_vis = Circle(radius=1.0, color=WHITE, stroke_width=1.5, stroke_opacity=0.4).shift(DOWN * 2.5)
        pointer = Arrow(
            circle_vis.get_center(),
            circle_vis.get_center() + RIGHT * 1.0,
            buff=0, color=BLUE, stroke_width=3, tip_length=0.15
        )

        with self.voiceover(
            text="Specifically, the vector x, y has the exact same distribution when multiplied by the 2D rotation matrix, consisting of cosine theta and negative sine theta in the first row, and sine theta and cosine theta in the second row, for any angle theta from zero to two pi."
        ) as tracker:
            self.play(Write(rot_matrix), run_time=tracker.duration * 0.3)
            self.play(Write(forall), run_time=tracker.duration * 0.1)
            self.play(Create(circle_vis), GrowArrow(pointer), run_time=0.5)
            self.play(
                Rotate(pointer, angle=TAU, about_point=circle_vis.get_center()),
                run_time=tracker.duration * 0.4,
                rate_func=linear
            )
            self.wait(tracker.duration * 0.1)

        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.5)
        self.wait(0.3)


# ═══════════════════════════════════════════════════════════════
# SCENE 16: Alternative Hypothesis — H₁ + Divergence D
# ═══════════════════════════════════════════════════════════════
class Scene16AlternativeHypothesis(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title = Text("Alternative Hypothesis H₁", font_size=34, color=WHITE, weight=BOLD)
        title.to_edge(UP, buff=0.5)

        # ── Sentence 1: H1 definition ──
        h1_eq = MathTex(
            r"H_1:", r"\;\sup_{g \in G}\,", r"D(g \cdot \mu,\, \mu)", r"\;\geq\; \epsilon",
            font_size=34, color=WHITE
        )
        h1_eq[0].set_color(RED)
        h1_eq[3].set_color(RED)
        h1_eq.next_to(title, DOWN, buff=0.5)

        # Visual: two overlapping distributions with shaded area = divergence
        ax_div = Axes(
            x_range=[-3, 3, 1], y_range=[0, 0.5, 0.1],
            x_length=5, y_length=2.0,
            axis_config={"color": WHITE, "include_ticks": False},
            tips=False
        ).shift(DOWN * 1.2)

        curve_mu = ax_div.plot(lambda x: 0.4 * np.exp(-0.5 * x**2), x_range=[-3, 3], color=BLUE, stroke_width=2.5)
        curve_g_mu = ax_div.plot(lambda x: 0.35 * np.exp(-0.5 * (x - 0.8)**2), x_range=[-3, 3], color=ORANGE, stroke_width=2.5)

        # Shade the difference area
        shade = ax_div.get_area(curve_mu, x_range=[-0.5, 2.0], bounded_graph=curve_g_mu, color=YELLOW, opacity=0.3)

        label_mu = MathTex(r"\mu", font_size=24, color=BLUE).next_to(curve_mu, UP, buff=0.1).shift(LEFT * 0.5)
        label_gmu = MathTex(r"g \cdot \mu", font_size=24, color=ORANGE).next_to(curve_g_mu, UP, buff=0.1).shift(RIGHT * 0.8)
        d_label = MathTex(r"D(g\cdot\mu, \mu) \geq \epsilon", font_size=22, color=YELLOW).next_to(ax_div, DOWN, buff=0.3)

        with self.voiceover(
            text="Under the alternative hypothesis H one, mu is not invariant — there exists some g in G such that the divergence D between g dot mu and mu is at least epsilon. Here D is a probability pseudometric satisfying the triangle inequality."
        ) as tracker:
            self.play(Write(title), run_time=tracker.duration * 0.08)
            self.play(Write(h1_eq), run_time=tracker.duration * 0.15)
            self.play(Create(ax_div), Create(curve_mu), FadeIn(label_mu), run_time=tracker.duration * 0.15)
            self.play(Create(curve_g_mu), FadeIn(label_gmu), run_time=tracker.duration * 0.15)
            self.play(FadeIn(shade), FadeIn(d_label), run_time=tracker.duration * 0.15)
            self.wait(tracker.duration * 0.25)

        # ── Sentence 2: Examples of D ──
        self.play(FadeOut(ax_div), FadeOut(curve_mu), FadeOut(curve_g_mu), FadeOut(shade),
                  FadeOut(label_mu), FadeOut(label_gmu), FadeOut(d_label), run_time=0.4)

        d_title = Text("Examples of D:", font_size=24, color=YELLOW, weight=BOLD).next_to(h1_eq, DOWN, buff=0.5)
        d_items = [
            (r"\text{Optimal Transport}", BLUE),
            (r"\text{Kernel MMD}", GREEN),
            (r"\text{Total Variation}", ORANGE),
            (r"\ell_p\text{ / Hellinger}", TEAL),  # Proper LaTeX math notation
            (r"\text{Neural Net Features}", PURPLE),
        ]
        d_boxes = VGroup()
        for i, (txt, col) in enumerate(d_items):
            box = RoundedRectangle(width=3.0, height=0.7, color=col, fill_opacity=0.1, corner_radius=0.08, stroke_width=1.5)
            label = MathTex(txt, font_size=32, color=col).move_to(box)
            d_boxes.add(VGroup(box, label))
        d_boxes.arrange_in_grid(rows=2, cols=3, buff=0.2).next_to(d_title, DOWN, buff=0.3)

        inv_note = Text("Assumption: D is G-invariant", font_size=18, color=WHITE).next_to(d_boxes, DOWN, buff=0.3)

        with self.voiceover(
            text="Examples include optimal transport, kernel MMD, total variation, ell-p distance, Hellinger, and neural network features. We assume D is itself invariant under group actions."
        ) as tracker:
            self.play(Write(d_title), run_time=0.4)
            self.play(*[FadeIn(b, scale=0.8) for b in d_boxes], run_time=tracker.duration * 0.4)
            self.play(FadeIn(inv_note), run_time=0.5)
            self.wait(tracker.duration * 0.2)

        # ── Sentence 3: Goal ──
        self.play(FadeOut(d_title), FadeOut(d_boxes), FadeOut(inv_note), run_time=0.4)

        goal_box = RoundedRectangle(width=10, height=2.0, color=YELLOW, fill_opacity=0.05, corner_radius=0.15, stroke_width=2)
        goal_box.next_to(h1_eq, DOWN, buff=0.5)
        goal_title = Text("DECISION GOAL", font_size=22, color=YELLOW, weight=BOLD).move_to(goal_box.get_top() + DOWN * 0.35)
        goal_left = VGroup(
            MathTex(r"\checkmark", font_size=30, color=GREEN),
            Text("μ is G-invariant", font_size=20, color=GREEN),
        ).arrange(RIGHT, buff=0.15)
        goal_right = VGroup(
            MathTex(r"\times", font_size=30, color=RED),
            Text("Find g ∈ G certifying asymmetry", font_size=20, color=RED),
        ).arrange(RIGHT, buff=0.15)
        goal_content = VGroup(goal_left, Text("OR", font_size=18, color=WHITE), goal_right).arrange(RIGHT, buff=0.5)
        goal_content.next_to(goal_title, DOWN, buff=0.3)

        with self.voiceover(
            text="So the goal is a decision: either conclude mu is invariant, or certify it is not by finding a specific g in G that violates the symmetry."
        ) as tracker:
            self.play(Create(goal_box), Write(goal_title), run_time=tracker.duration * 0.3)
            self.play(FadeIn(goal_content), run_time=tracker.duration * 0.4)
            self.wait(tracker.duration * 0.25)

        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.5)
        self.wait(0.3)


# ═══════════════════════════════════════════════════════════════
# SCENE 17: Computational Hardness & Randomized Solution
# ═══════════════════════════════════════════════════════════════
class Scene17HardnessAndRandomness(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title = Text("Computational Hardness & Randomized Solution", font_size=32, color=WHITE, weight=BOLD)
        title.to_edge(UP, buff=0.5)

        # ── Sentence 1: NP-hardness ──
        hard_box = RoundedRectangle(width=8, height=1.4, color=RED, fill_opacity=0.1, corner_radius=0.12, stroke_width=2)
        hard_box.next_to(title, DOWN, buff=0.5)
        hard_text = VGroup(
            Text("Even with infinite data:", font_size=20, color=WHITE),
            MathTex(r"\arg\sup_{g \in G} D(g \cdot \mu, \mu)", font_size=28, color=RED),
            Text("is NP-hard!", font_size=22, color=RED, weight=BOLD),
        ).arrange(DOWN, buff=0.1).move_to(hard_box)

        reduction_note = Text("(Reduction from Traveling Salesman Problem)", font_size=16, color=WHITE).next_to(hard_box, DOWN, buff=0.2)

        with self.voiceover(
            text="Even with infinite data, finding the worst-case g in G can be computationally challenging. There exists a setting where this arg sup problem is NP-hard — the proof reduces from a variant of the Traveling Salesman Problem."
        ) as tracker:
            self.play(Write(title), run_time=tracker.duration * 0.1)
            self.play(FadeIn(hard_box), Write(hard_text), run_time=tracker.duration * 0.35)
            self.play(FadeIn(reduction_note), run_time=tracker.duration * 0.15)
            self.wait(tracker.duration * 0.3)

        # ── Sentence 2: Randomness helps — sandwich theorem ──
        self.play(FadeOut(hard_text), FadeOut(hard_box), FadeOut(reduction_note), run_time=0.3)

        rand_title = Text("But randomness helps!", font_size=26, color=GREEN, weight=BOLD).next_to(title, DOWN, buff=0.5)

        # Sandwich inequality
        sandwich = MathTex(
            r"\mathbb{E}_{g \sim G}[D(g \cdot \mu, \mu)]",
            r"\;\leq\;",
            r"\sup_{g \in G} D(g \cdot \mu, \mu)",
            r"\;\leq\;",
            r"2\,\mathbb{E}_{g \sim G}[D(g \cdot \mu, \mu)]",
            font_size=26, color=WHITE
        ).next_to(rand_title, DOWN, buff=0.5)
        sandwich[0].set_color(GREEN)
        sandwich[2].set_color(RED)
        sandwich[4].set_color(GREEN)

        # Visual: number line with sampled D values, mean, and sup
        nline = NumberLine(x_range=[0, 1, 0.2], length=8, include_numbers=True, font_size=16, color=WHITE).shift(DOWN * 1.5)
        # Sample several D(g·μ, μ) values
        d_vals = [0.15, 0.25, 0.35, 0.42, 0.55, 0.3, 0.2, 0.48]
        sample_dots = VGroup()
        for v in d_vals:
            dot = Dot(nline.n2p(v), radius=0.08, color=BLUE, fill_opacity=0.7)
            sample_dots.add(dot)

        mean_val = np.mean(d_vals)
        sup_val = max(d_vals)
        mean_line = DashedLine(
            nline.n2p(mean_val) + UP * 0.5,
            nline.n2p(mean_val) + DOWN * 0.3,
            color=GREEN, stroke_width=2.5
        )
        mean_label = MathTex(r"\mathbb{E}", font_size=22, color=GREEN).next_to(mean_line, UP, buff=0.1)

        sup_line = DashedLine(
            nline.n2p(sup_val) + UP * 0.5,
            nline.n2p(sup_val) + DOWN * 0.3,
            color=RED, stroke_width=2.5
        )
        sup_label = MathTex(r"\sup", font_size=22, color=RED).next_to(sup_line, UP, buff=0.1)

        # Show 2×E marker
        two_e_val = min(2 * mean_val, 1.0)
        two_e_line = DashedLine(
            nline.n2p(two_e_val) + UP * 0.5,
            nline.n2p(two_e_val) + DOWN * 0.3,
            color=YELLOW, stroke_width=2
        )
        two_e_label = MathTex(r"2\mathbb{E}", font_size=22, color=YELLOW).next_to(two_e_line, UP, buff=0.1)

        # Bracket showing sup ≤ 2E
        bracket = BraceBetweenPoints(nline.n2p(mean_val) + DOWN * 0.4, nline.n2p(two_e_val) + DOWN * 0.4, direction=DOWN, color=YELLOW)
        bracket_label = Text("factor of 2", font_size=16, color=YELLOW).next_to(bracket, DOWN, buff=0.1)

        with self.voiceover(
            text="But randomness can help. A key theorem shows that the expected divergence over a random g is always within a factor of two of the supremum. The proof uses second-order probability and group theory."
        ) as tracker:
            self.play(Write(rand_title), run_time=tracker.duration * 0.1)
            self.play(Write(sandwich), run_time=tracker.duration * 0.2)
            self.play(Create(nline), run_time=0.5)
            self.play(*[GrowFromCenter(d) for d in sample_dots], run_time=tracker.duration * 0.15)
            self.play(Create(mean_line), FadeIn(mean_label), run_time=tracker.duration * 0.1)
            self.play(Create(sup_line), FadeIn(sup_label), run_time=tracker.duration * 0.1)
            self.play(Create(two_e_line), FadeIn(two_e_label), Create(bracket), FadeIn(bracket_label), run_time=tracker.duration * 0.15)
            self.wait(tracker.duration * 0.1)

        # ── Sentence 3: Corollary — sample then test ──
        self.play(
            FadeOut(nline), FadeOut(sample_dots), FadeOut(mean_line), FadeOut(mean_label),
            FadeOut(sup_line), FadeOut(sup_label), FadeOut(two_e_line), FadeOut(two_e_label),
            FadeOut(bracket), FadeOut(bracket_label), FadeOut(sandwich), FadeOut(rand_title),
            run_time=0.4
        )

        # Recipe visual: 3-step pipeline
        step1_box = RoundedRectangle(width=2.5, height=1.0, color=BLUE, fill_opacity=0.12, corner_radius=0.1, stroke_width=1.5).shift(LEFT * 3.5 + DOWN * 0.5)
        step1_txt = VGroup(
            Text("Step 1", font_size=14, color=BLUE),
            Text("Sample g ~ G", font_size=18, color=WHITE),
        ).arrange(DOWN, buff=0.05).move_to(step1_box)

        step2_box = RoundedRectangle(width=2.5, height=1.0, color=ORANGE, fill_opacity=0.12, corner_radius=0.1, stroke_width=1.5).shift(DOWN * 0.5)
        step2_txt = VGroup(
            Text("Step 2", font_size=14, color=ORANGE),
            Text("Compute D(g·μ, μ)", font_size=18, color=WHITE),
        ).arrange(DOWN, buff=0.05).move_to(step2_box)

        step3_box = RoundedRectangle(width=2.5, height=1.0, color=GREEN, fill_opacity=0.12, corner_radius=0.1, stroke_width=1.5).shift(RIGHT * 3.5 + DOWN * 0.5)
        step3_txt = VGroup(
            Text("Step 3", font_size=14, color=GREEN),
            Text("Threshold test", font_size=18, color=WHITE),
        ).arrange(DOWN, buff=0.05).move_to(step3_box)

        arr1 = Arrow(step1_box.get_right(), step2_box.get_left(), buff=0.1, color=WHITE, stroke_width=2, tip_length=0.15)
        arr2 = Arrow(step2_box.get_right(), step3_box.get_left(), buff=0.1, color=WHITE, stroke_width=2, tip_length=0.15)

        recipe_label = Text("Recipe: Sample from G, then test!", font_size=18, color=WHITE, weight=BOLD).next_to(title, DOWN, buff=0.5)

        with self.voiceover(
            text="The corollary is powerful: a random g from G certificates symmetry or asymmetry. So the recipe is simple — sample from the group, then test!"
        ) as tracker:
            self.play(Write(recipe_label), run_time=tracker.duration * 0.15)
            self.play(FadeIn(step1_box), FadeIn(step1_txt), run_time=tracker.duration * 0.15)
            self.play(GrowArrow(arr1), run_time=0.3)
            self.play(FadeIn(step2_box), FadeIn(step2_txt), run_time=tracker.duration * 0.15)
            self.play(GrowArrow(arr2), run_time=0.3)
            self.play(FadeIn(step3_box), FadeIn(step3_txt), run_time=tracker.duration * 0.15)
            self.wait(tracker.duration * 0.15)

        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.5)
        self.wait(0.3)


# ═══════════════════════════════════════════════════════════════
# SCENE 18: Deterministic Algorithms — Covering via Group Structure
# ═══════════════════════════════════════════════════════════════
class Scene18DeterministicCovering(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title = Text("Deterministic Algorithms: Exploiting Group Structure", font_size=30, color=WHITE, weight=BOLD)
        title.to_edge(UP, buff=0.5)

        # ── Sentence 1: Classical covering ──
        classical_label = Text("Classical Approach: ε-covering of G", font_size=18, color=WHITE).next_to(title, DOWN, buff=0.4)

        # Visual: circle (SO(2)) packed with many points
        circle_left = Circle(radius=1.3, color=RED, stroke_width=2).shift(LEFT * 3 + DOWN * 1.2)
        np.random.seed(99)
        many_points = VGroup()
        for i in range(40):
            angle = i * TAU / 40
            many_points.add(Dot(circle_left.get_center() + 1.3 * np.array([np.cos(angle), np.sin(angle), 0]), radius=0.04, color=RED, fill_opacity=0.7))

        classical_size = MathTex(r"|\mathcal{S}| = O(\epsilon^{-d^2})", font_size=26, color=RED).next_to(circle_left, DOWN, buff=0.3)
        classical_note = Text("Exponential in dimension!", font_size=16, color=RED).next_to(classical_size, DOWN, buff=0.1)

        with self.voiceover(
            text="How can we avoid sampling g from the group? The classical idea is covering: approximate the supremum over G by the maximum over a finite set S. But the group is large, the geometry is non-standard, and covering is non-constructive."
        ) as tracker:
            self.play(Write(title), run_time=tracker.duration * 0.1)
            self.play(Write(classical_label), run_time=tracker.duration * 0.1)
            self.play(Create(circle_left), run_time=tracker.duration * 0.1)
            self.play(*[GrowFromCenter(p) for p in many_points], run_time=tracker.duration * 0.2)
            self.play(Write(classical_size), FadeIn(classical_note), run_time=tracker.duration * 0.2)
            self.wait(tracker.duration * 0.25)

        # ── Sentence 2: New idea — triangle inequality ──
        new_label = Text("New Idea: Exploit Group Structure", font_size=18, color=WHITE, weight=BOLD).next_to(title, DOWN, buff=0.4)

        # Triangle inequality diagram
        tri_center = RIGHT * 2.5 + DOWN * 0.8
        mu_pos = tri_center + LEFT * 2.0 + DOWN * 1.2
        g1_pos = tri_center + UP * 0.8
        g1g2_pos = tri_center + RIGHT * 2.0 + DOWN * 1.2

        mu_dot = Dot(mu_pos, radius=0.12, color=BLUE)
        g1_dot = Dot(g1_pos, radius=0.12, color=ORANGE)
        g1g2_dot = Dot(g1g2_pos, radius=0.12, color=PURPLE)

        mu_lbl = MathTex(r"\mu", font_size=24, color=BLUE).next_to(mu_dot, DOWN, buff=0.15)
        g1_lbl = MathTex(r"g_1 \cdot \mu", font_size=24, color=ORANGE).next_to(g1_dot, UP, buff=0.15)
        g1g2_lbl = MathTex(r"g_1 g_2 \cdot \mu", font_size=24, color=PURPLE).next_to(g1g2_dot, DOWN, buff=0.15)

        edge1 = Arrow(mu_pos, g1_pos, buff=0.15, color=ORANGE, stroke_width=2.5, tip_length=0.12)
        edge2 = Arrow(g1_pos, g1g2_pos, buff=0.15, color=PURPLE, stroke_width=2.5, tip_length=0.12)
        edge3 = Arrow(mu_pos, g1g2_pos, buff=0.15, color=YELLOW, stroke_width=3, tip_length=0.12)

        edge1_lbl = MathTex(r"D(g_1\cdot\mu, \mu)", font_size=16, color=ORANGE).move_to((mu_pos + g1_pos) / 2 + LEFT * 0.8)
        edge2_lbl = MathTex(r"D(g_2\cdot\mu, \mu)", font_size=16, color=PURPLE).move_to((g1_pos + g1g2_pos) / 2 + RIGHT * 0.8)
        edge3_lbl = MathTex(r"\leq \text{sum}", font_size=16, color=YELLOW).move_to((mu_pos + g1g2_pos) / 2 + DOWN * 0.4)

        tri_ineq = MathTex(
            r"D(g_1 g_2 \cdot \mu,\, \mu)",
            r"\;\leq\;",
            r"D(g_1 \cdot \mu,\, \mu) + D(g_2 \cdot \mu,\, \mu)",
            font_size=24, color=WHITE
        ).to_edge(DOWN, buff=0.5)
        tri_ineq[0].set_color(YELLOW)

        with self.voiceover(
            text="The new idea is to exploit the group structure and its redundancies. By the triangle inequality for D, the divergence under composition g one times g two is at most the sum of individual divergences — this is not captured by classical covering arguments."
        ) as tracker:
            self.play(
                ReplacementTransform(classical_label, new_label),
                FadeOut(many_points), FadeOut(classical_size), FadeOut(classical_note),
                run_time=tracker.duration * 0.1
            )
            # Build triangle diagram step by step
            self.play(FadeIn(mu_dot), FadeIn(mu_lbl), run_time=tracker.duration * 0.08)
            self.play(GrowArrow(edge1), FadeIn(g1_dot), FadeIn(g1_lbl), FadeIn(edge1_lbl), run_time=tracker.duration * 0.15)
            self.play(GrowArrow(edge2), FadeIn(g1g2_dot), FadeIn(g1g2_lbl), FadeIn(edge2_lbl), run_time=tracker.duration * 0.15)
            self.play(GrowArrow(edge3), FadeIn(edge3_lbl), run_time=tracker.duration * 0.15)
            self.play(Write(tri_ineq), run_time=tracker.duration * 0.2)
            self.wait(tracker.duration * 0.1)

        # ── Sentence 3: Result — exponential improvement in covering ──
        self.play(
            *[FadeOut(m) for m in [mu_dot, g1_dot, g1g2_dot, mu_lbl, g1_lbl, g1g2_lbl,
                                    edge1, edge2, edge3, edge1_lbl, edge2_lbl, edge3_lbl,
                                    tri_ineq, circle_left, new_label]],
            run_time=0.4
        )

        result_title = Text("Result for SO(d):", font_size=18, color=WHITE, weight=BOLD).next_to(title, DOWN, buff=0.5)

        # Side by side comparison: classical vs new
        comp_left = VGroup(
            Text("Classical", font_size=20, color=RED, weight=BOLD),
            MathTex(r"O\!\left(\epsilon^{-d^2}\right)", font_size=30, color=RED),
        ).arrange(DOWN, buff=0.15).shift(LEFT * 2.5 + DOWN * 1.0)

        comp_right = VGroup(
            Text("Group-Aware (Ours)", font_size=20, color=GREEN, weight=BOLD),
            MathTex(r"O\!\left(\frac{d^2}{\epsilon}\right)", font_size=30, color=GREEN),
        ).arrange(DOWN, buff=0.15).shift(RIGHT * 2.5 + DOWN * 1.0)

        vs_text = Text("vs", font_size=24, color=WHITE).move_to(DOWN * 1.0)

        # Bar chart comparison
        bar_classical = Rectangle(width=6, height=0.5, color=RED, fill_opacity=0.35, stroke_width=1.5).shift(DOWN * 2.3)
        bar_classical_l = MathTex(r"\epsilon^{-d^2}", font_size=20, color=RED).move_to(bar_classical)
        bar_new = Rectangle(width=0.6, height=0.5, color=GREEN, fill_opacity=0.5, stroke_width=1.5).align_to(bar_classical, LEFT).shift(DOWN * 3.0)
        bar_new_l = MathTex(r"d^2/\epsilon", font_size=20, color=GREEN).next_to(bar_new, RIGHT, buff=0.15)

        exp_badge = Text("Exponential Improvement!", font_size=22, color=YELLOW, weight=BOLD)
        exp_badge.next_to(bar_new, DOWN, buff=0.4).set_x(0)

        with self.voiceover(
            text="For the rotation group SO of d, this yields dramatically reduced coverings: size order d squared over epsilon instead of epsilon to the minus d squared. This is an exponential improvement."
        ) as tracker:
            self.play(Write(result_title), run_time=tracker.duration * 0.1)
            self.play(FadeIn(comp_left), FadeIn(vs_text), FadeIn(comp_right), run_time=tracker.duration * 0.2)
            self.play(FadeIn(bar_classical), FadeIn(bar_classical_l), run_time=tracker.duration * 0.15)
            self.play(FadeIn(bar_new), FadeIn(bar_new_l), run_time=tracker.duration * 0.15)
            self.play(FadeIn(exp_badge, scale=1.2), run_time=tracker.duration * 0.15)
            self.wait(tracker.duration * 0.15)

        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.5)
        self.wait(0.3)


# ═══════════════════════════════════════════════════════════════
# SCENE 19: Open Directions & Summary
# ═══════════════════════════════════════════════════════════════
class Scene19OpenDirections(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title = Text("Open Directions & Summary", font_size=38, color=WHITE, weight=BOLD)
        title.to_edge(UP, buff=0.5)

        # ── Sentence 1: Open questions ──
        open_qs = VGroup(
            Text("Theory beyond classical i.i.d. statistics", font_size=22, color=WHITE),
            Text("Optimal reduced coverings for complex groups", font_size=22, color=WHITE),
            Text("Coverings for optimization over groups", font_size=22, color=WHITE),
        )
        icons = VGroup(
            MathTex(r"?", font_size=30, color=BLUE),
            MathTex(r"?", font_size=30, color=GREEN),
            MathTex(r"?", font_size=30, color=ORANGE),
        )
        q_rows = VGroup()
        for icon, q in zip(icons, open_qs):
            row = VGroup(icon, q).arrange(RIGHT, buff=0.3)
            q_rows.add(row)
        q_rows.arrange(DOWN, aligned_edge=LEFT, buff=0.35).next_to(title, DOWN, buff=0.6)

        with self.voiceover(
            text="Finally, open directions. Statistics under symmetries need theory beyond classical statistics. Can we obtain optimal reduced coverings for groups? Can we develop coverings for optimization over groups?"
        ) as tracker:
            self.play(Write(title), run_time=tracker.duration * 0.1)
            for row in q_rows:
                self.play(FadeIn(row, shift=RIGHT * 0.3), run_time=tracker.duration * 0.2)
            self.wait(tracker.duration * 0.2)

        # ── Sentence 2: Exciting applied question + references ──
        self.play(FadeOut(q_rows), run_time=0.3)

        applied_box = RoundedRectangle(width=10, height=1.6, color=YELLOW, fill_opacity=0.08, corner_radius=0.12, stroke_width=2)
        applied_box.next_to(title, DOWN, buff=0.5)
        applied_q = Text("Can we integrate symmetry testing with\nsymmetry-aware architecture design?", font_size=24, color=YELLOW, line_spacing=1.2).move_to(applied_box)

        # Hub connecting testing ↔ architecture
        hub_test = RoundedRectangle(width=4.0, height=0.8, color=BLUE, fill_opacity=0.1, corner_radius=0.08, stroke_width=1.5).shift(LEFT * 3 + DOWN * 2.5)
        hub_test_txt = Tex("Symmetry\nTesting", font_size=32, color=BLUE).move_to(hub_test)
        hub_arch = RoundedRectangle(width=4.0, height=0.8, color=GREEN, fill_opacity=0.1, corner_radius=0.08, stroke_width=1.5).shift(RIGHT * 3 + DOWN * 2.5)
        hub_arch_txt = Tex("Equivariant\nArchitecture", font_size=32, color=GREEN).move_to(hub_arch)
        connect_arrow = DoubleArrow(hub_test.get_right(), hub_arch.get_left(), buff=0.1, color=YELLOW, stroke_width=2.5, tip_length=0.15)
        connect_label = Text("Integrate?", font_size=18, color=YELLOW).next_to(connect_arrow, DOWN, buff=0.1)

        # Reference
        ref = VGroup(
            Tex("Soleymani, Tahmasebi, Jegelka, Jaillet", font_size=28, color=WHITE),
            Tex("AISTATS 2025 / Annals of Statistics", font_size=28, color=WHITE),
        ).arrange(DOWN, buff=0.05).to_edge(DOWN, buff=0.3)

        with self.voiceover(
            text=(
              "And an exciting applied question: can we integrate symmetry testing "
              "with symmetry-aware architecture design? "
              "These remain open problems for future work. "
              "This concludes our section on symmetry validation."
          )
        ) as tracker:
            self.play(Create(applied_box), Write(applied_q), run_time=tracker.duration * 0.3)
            self.play(
                FadeIn(hub_test), FadeIn(hub_test_txt),
                FadeIn(hub_arch), FadeIn(hub_arch_txt),
                run_time=tracker.duration * 0.2
            )
            self.play(GrowArrow(connect_arrow), FadeIn(connect_label), run_time=tracker.duration * 0.2)
            self.play(FadeIn(ref), run_time=0.5)
            self.wait(tracker.duration * 0.15)

        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.8)
        self.wait(0.5)


In [ ]:
import subprocess, sys
from pathlib import Path
from IPython.display import Video, display

# ═══════════════════════════════════════════════════════════
# RENDER scenes 93–101
# ═══════════════════════════════════════════════════════════
scene_file = "/content/slides_93_101.py"
scenes_to_render = [
    "Scene13SymmetryValidationTitle",
    "Scene14DistributionalSymmetry",
    "Scene15NullHypothesis",
    "Scene16AlternativeHypothesis",
    "Scene17HardnessAndRandomness",
    "Scene18DeterministicCovering",
    "Scene19OpenDirections",
]

output_paths_93_101 = []

for scene_name in scenes_to_render:
    print(f"Rendering {scene_name}...")
    result = subprocess.run(
        [
            sys.executable, "-m", "manim", "render",
            "-qh",                    # 1080p, 60fps
            "--disable_caching",
            scene_file,
            scene_name,
        ],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"  Error:\n{result.stderr[-800:]}")
    else:
        found = list(Path("media").rglob(f"{scene_name}.mp4"))
        if found:
            output_paths_93_101.append(str(found[0]))
            print(f"  {found[0]}")
        else:
            print(f"  Rendered OK but file not found")

print(f"\nTotal: {len(output_paths_93_101)} videos")
for p in output_paths_93_101:
    print(f"  {p}")

# ═══════════════════════════════════════════════════════════
# CONCAT — giữ 60fps, CRF 20, thống nhất với các segment khác
# ═══════════════════════════════════════════════════════════
list_file = "/content/concat_list_93_101.txt"
with open(list_file, "w") as f:
    for p in output_paths_93_101:
        if Path(p).exists():
            f.write(f"file '{p}'\n")

output_file = "/content/slide_93_101_voiceover.mp4"

cmd = [
    "ffmpeg", "-y",
    "-f", "concat",
    "-safe", "0",
    "-i", list_file,
    "-c:v", "libx264",
    "-preset", "medium",
    "-crf", "20",
    "-r", "60",                # thống nhất 60fps với các segment khác
    "-pix_fmt", "yuv420p",
    "-c:a", "aac",
    "-b:a", "192k",
    "-movflags", "+faststart",
    output_file
]

result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode == 0 and Path(output_file).exists():
    probe = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries",
         "format=duration,bit_rate", "-show_entries",
         "stream=r_frame_rate,width,height,bit_rate",
         "-of", "default=noprint_wrappers=1",
         output_file],
        capture_output=True, text=True
    )
    print(f"\nFinal: {output_file}")
    print(probe.stdout)
    display(Video(output_file, embed=True, width=854))
else:
    print(f"FFmpeg error:\n{result.stderr[-500:]}")


## Display

In [ ]:
import subprocess
from pathlib import Path
from IPython.display import Video, display

list_file = "/content/concat_list_93_101.txt"
output_file = "/content/slide_93_101_voiceover.mp4"

with open(list_file, "w") as f:
    for p in output_paths_93_101:
        if Path(p).exists():
            f.write(f"file '{p}'\n")

cmd = [
    "ffmpeg", "-y",
    "-f", "concat",
    "-safe", "0",
    "-i", list_file,
    "-c:v", "libx264",
    "-preset", "medium",
    "-crf", "20",
    "-r", "60",
    "-pix_fmt", "yuv420p",
    "-c:a", "aac",
    "-b:a", "192k",
    "-movflags", "+faststart",
    output_file
]

result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode == 0 and Path(output_file).exists():
    probe = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries",
         "format=duration,bit_rate", "-show_entries",
         "stream=r_frame_rate,width,height,bit_rate",
         "-of", "default=noprint_wrappers=1",
         output_file],
        capture_output=True, text=True
    )
    print(f"Final: {output_file}")
    print(probe.stdout)
    display(Video(output_file, embed=True, width=854))
else:
    print(f"FFmpeg error:\n{result.stderr[-500:]}")


# SLIDE 102–110

## Generative

In [ ]:
%%writefile /content/slides_102_110.py

# ══════════════════════════════════════════════════════════════
# FILE: slides_102_110.py — Recent Results III: Sample Complexity
# Style: VoiceoverScene + EdgeTTSService
# ══════════════════════════════════════════════════════════════

import nest_asyncio
nest_asyncio.apply()

import asyncio
import hashlib
import numpy as np
from pathlib import Path

from manim import *
from manim_voiceover import VoiceoverScene
from manim_voiceover.services.base import SpeechService

import edge_tts

# ─────────────────────────────────────────────
# EdgeTTSService
# ─────────────────────────────────────────────
VOICE = "en-US-GuyNeural"


class EdgeTTSService(SpeechService):
    def __init__(self, voice="en-US-GuyNeural", **kwargs):
        self.voice = voice
        super().__init__(**kwargs)

    def generate_from_text(self, text, cache_dir=None, path=None, **kwargs):
        if cache_dir is None:
            cache_dir = self.cache_dir
        Path(cache_dir).mkdir(parents=True, exist_ok=True)

        input_data = {"input_text": text, "service": "edge_tts", "voice": self.voice}

        cached = self.get_cached_result(input_data, cache_dir)
        if cached is not None:
            return cached

        audio_basename = hashlib.md5(text.encode()).hexdigest()
        audio_file = audio_basename + ".mp3"
        full_path = str(Path(cache_dir) / audio_file)

        communicate = edge_tts.Communicate(text, self.voice)
        loop = asyncio.get_event_loop()
        loop.run_until_complete(communicate.save(full_path))

        json_dict = {
            "input_text": text,
            "input_data": input_data,
            "original_audio": audio_file,
        }

        return json_dict


# ═══════════════════════════════════════════════════════════════
# SCENE 20: Title — Sample Complexity Benefits
# ═══════════════════════════════════════════════════════════════
class Scene20SampleComplexityTitle(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title1 = Text("Recent Results III:", font_size=44, color=WHITE, weight=BOLD)
        title2 = Text("Sample Complexity Benefits of Symmetry", font_size=38, color=WHITE, weight=BOLD)
        title_group = VGroup(title1, title2).arrange(DOWN, buff=0.3).move_to(UP * 1.0)

        # Visual teaser: two learning curves showing the correct rates
        # n^{-s/(s+d)} vs n^{-s/(s+d_eff)} with s=2, d=4, d_eff=2
        ax_teaser = Axes(
            x_range=[0, 100, 20], y_range=[0, 1, 0.2],
            x_length=8, y_length=3.0,  # tăng từ 5×2.2
            axis_config={"color": WHITE, "include_ticks": False},
            tips=False,
        ).shift(DOWN * 1.5)

        # Rate n^{-s/(s+d)}: s=2, d=4 → n^{-1/3}; with symmetry d_eff=2 → n^{-1/2}
        curve_no_sym = ax_teaser.plot(
            lambda x: min(1.0, max(0.05, 2.0 * (max(x, 1) ** (-1.0 / 3.0)))),
            x_range=[1, 100], color=RED, stroke_width=2.5
        )
        curve_sym = ax_teaser.plot(
            lambda x: min(1.0, max(0.03, 1.5 * (max(x, 1) ** (-1.0 / 2.0)))),
            x_range=[1, 100], color=GREEN, stroke_width=2.5
        )

        label_no = MathTex(r"n^{-\frac{s}{s+d}}", font_size=28, color=RED).next_to(ax_teaser.c2p(85, 0.45), RIGHT, buff=0.1)
        label_yes = MathTex(r"n^{-\frac{s}{s+d_{\text{eff}}}}", font_size=28, color=GREEN).next_to(ax_teaser.c2p(60, 0.12), RIGHT, buff=0.1)
        x_lab = Text("Samples n", font_size=20, color=WHITE).next_to(ax_teaser, DOWN, buff=0.15)
        y_lab = Text("Error", font_size=20, color=WHITE).next_to(ax_teaser, LEFT, buff=0.1).rotate(PI / 2)

        with self.voiceover(
                text=(
              "Now, moving to a different topic entirely. "
              "The third set of recent results focuses on "
              "the provable sample complexity benefits of incorporating symmetries into learning."
          )
        ) as tracker:
            self.play(Write(title1), run_time=tracker.duration * 0.3)
            self.play(Write(title2), run_time=tracker.duration * 0.3)
            # self.play(Create(ax_teaser), FadeIn(x_lab), FadeIn(y_lab), run_time=tracker.duration * 0.15)
            # self.play(Create(curve_no_sym), FadeIn(label_no), run_time=tracker.duration * 0.15)
            # self.play(Create(curve_sym), FadeIn(label_yes), run_time=tracker.duration * 0.15)
            self.wait(max(0, tracker.duration * 0.4))

        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.5)
        self.wait(0.3)


# ═══════════════════════════════════════════════════════════════
# SCENE 21: Motivation — Why Provable Gains?
# ═══════════════════════════════════════════════════════════════
class Scene21ProvableGains(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title = Text("Do Symmetries Provably Reduce Sample Complexity?", font_size=30, color=WHITE, weight=BOLD)
        title.to_edge(UP, buff=0.5)

        # ── Sentence 1: Practice vs Theory ──
        practice_box = RoundedRectangle(width=4.5, height=1.2, color=GREEN, fill_opacity=0.1, corner_radius=0.1, stroke_width=1.5).shift(LEFT * 2.8 + UP * 1.2)
        practice_txt = VGroup(
            Tex("Practice", font_size=36, color=GREEN),
            Tex("Invariances help!", font_size=32, color=WHITE),
        ).arrange(DOWN, buff=0.05).move_to(practice_box)

        theory_box = RoundedRectangle(width=4.5, height=1.2, color=YELLOW, fill_opacity=0.1, corner_radius=0.1, stroke_width=1.5).shift(RIGHT * 2.8 + UP * 1.2)
        theory_txt = VGroup(
            Tex("Theory", font_size=36, color=YELLOW),
            Tex("Provable gains?", font_size=32, color=WHITE),
        ).arrange(DOWN, buff=0.05).move_to(theory_box)

        with self.voiceover(
            text="In practice, learning with invariances works better. But can we actually prove that symmetry reduces sample complexity?"
        ) as tracker:
            self.play(Write(title), run_time=tracker.duration * 0.15)
            self.play(FadeIn(practice_box), FadeIn(practice_txt), run_time=tracker.duration * 0.25)
            self.play(FadeIn(theory_box), FadeIn(theory_txt), run_time=tracker.duration * 0.25)
            self.wait(max(0, tracker.duration * 0.3))

        # ── Sentence 2: Minimal assumptions ──
        with self.voiceover(
            text="We need a theory that works under minimal assumptions — any compact Riemannian manifold as data domain, any compact Lie group acting by isometries."
        ) as tracker:
            goal_items = VGroup(
                Text("• Any compact manifold M (not just spheres)", font_size=18, color=YELLOW),
                Text("• Any compact Lie group G (not just permutations)", font_size=18, color=YELLOW),
                Text("• Any smooth group action (isometric)", font_size=18, color=YELLOW),
            ).arrange(DOWN, aligned_edge=LEFT, buff=0.12).next_to(theory_box, DOWN, buff=0.5)
            goal_items.set_x(0)
            self.play(
                LaggedStart(*[FadeIn(item, shift=RIGHT * 0.3) for item in goal_items], lag_ratio=0.3),
                run_time=tracker.duration * 0.7
            )
            self.wait(max(0, tracker.duration * 0.25))

        # ── Sentence 3: Prior work limitations ──
        self.play(FadeOut(practice_box), FadeOut(practice_txt), FadeOut(theory_box), FadeOut(theory_txt), FadeOut(goal_items), run_time=0.3)

        # Prior work: sphere + specific groups
        prior_title = Text("Prior Work (Restrictive)", font_size=22, color=RED, weight=BOLD).shift(LEFT * 3 + UP * 1.0)
        sphere = Circle(radius=0.8, color=RED, stroke_width=2).next_to(prior_title, DOWN, buff=0.3)
        sphere_label = Tex(r"Unit Sphere only", font_size=25, color=RED).next_to(sphere, DOWN, buff=0.1)
        groups_label = Tex(r"$G = S_n$ or $SO(2)$ only", font_size=25, color=RED).next_to(sphere_label, DOWN, buff=0.1)

        with self.voiceover(
            text="Prior work often assumes the data lives on the unit sphere, and the group is a specific one like permutations or planar rotations. This is too restrictive for real applications."
        ) as tracker:
            self.play(Write(prior_title), Create(sphere), FadeIn(sphere_label), FadeIn(groups_label), run_time=tracker.duration * 0.5)
            # Cross it out
            cross = Cross(VGroup(sphere, sphere_label, groups_label), color=RED, stroke_width=3)
            self.play(Create(cross), run_time=tracker.duration * 0.2)
            self.wait(max(0, tracker.duration * 0.25))

        # ── Sentence 4: Our approach ──
        our_title = Text("Our Approach (General)", font_size=22, color=GREEN, weight=BOLD).shift(RIGHT * 3 + UP * 1.0)

        # Generic manifold: closed curve representing arbitrary shape
        manifold = ParametricFunction(
            lambda t: np.array([1.5 * np.cos(t) + 0.3 * np.cos(3 * t), 0.8 * np.sin(t) + 0.2 * np.sin(2 * t), 0]),
            t_range=[0, TAU], color=GREEN, stroke_width=2.5
        ).next_to(our_title, DOWN, buff=0.3).scale(0.7)
        manifold_label = Tex("Any compact manifold M", font_size=25, color=GREEN).next_to(manifold, DOWN, buff=0.1)
        group_label2 = Tex("Any compact Lie group G acting on M", font_size=25, color=GREEN).next_to(manifold_label, DOWN, buff=0.1)

        with self.voiceover(
            text="Our result works for any compact Riemannian manifold M and any compact Lie group G acting on M by isometries."
        ) as tracker:
            self.play(Write(our_title), Create(manifold), FadeIn(manifold_label), FadeIn(group_label2), run_time=tracker.duration * 0.6)
            self.wait(max(0, tracker.duration * 0.3))

        # ── Sentence 5: Concrete example ──
        self.play(*[FadeOut(m) for m in self.mobjects if m != title], run_time=0.4)

        # 3D molecular example
        example_title = Text("Example: Molecular conformations", font_size=18, color=WHITE, weight=BOLD).next_to(title, DOWN, buff=0.5)

        np.random.seed(42)
        points_3d = VGroup()
        positions = [[-1.5, 0.5, 0], [-0.5, -0.3, 0], [0.5, 0.7, 0], [1.2, -0.2, 0], [0, -1.0, 0]]
        for pos in positions:
            points_3d.add(Dot(pos, radius=0.12, color=BLUE, fill_opacity=0.8))
        # bonds
        bonds = VGroup()
        for i in range(len(positions) - 1):
            bonds.add(Line(positions[i], positions[i + 1], color=WHITE, stroke_width=1.5))
        molecule = VGroup(bonds, points_3d).shift(DOWN * 1.0)

        action_label = MathTex(r"G = SE(3)", font_size=22, color=YELLOW).shift(DOWN * 2.8 + LEFT * 2)
        action_desc = Tex("rotations and translations", font_size=29, color=YELLOW).next_to(action_label, RIGHT, buff=0.2)

        with self.voiceover(
            text="For example, molecular conformations live on a non-spherical manifold, and the symmetry group is S E 3 — rotations and translations in 3D."
        ) as tracker:
            self.play(FadeIn(example_title), run_time=0.3)
            self.play(FadeIn(molecule, scale=0.5), run_time=tracker.duration * 0.25)
            self.play(FadeIn(action_label), FadeIn(action_desc), run_time=tracker.duration * 0.2)
            # Animate rotation
            self.play(Rotate(molecule, angle=PI / 3, about_point=molecule.get_center()), run_time=tracker.duration * 0.25)
            self.wait(max(0, tracker.duration * 0.2))

        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.5)
        self.wait(0.3)


# ═══════════════════════════════════════════════════════════════
# SCENE 22: General Result — The Theorem
# ═══════════════════════════════════════════════════════════════
# ═══════════════════════════════════════════════════════════════
# SCENE 22: General Result — The Theorem (FIXED)
# ═══════════════════════════════════════════════════════════════
class Scene22GeneralResult(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title = Text("General Theorem: Sample Complexity with Symmetry", font_size=28, color=WHITE, weight=BOLD)
        title.to_edge(UP, buff=0.4)

        # ── Setting up assumptions ──
        with self.voiceover(
            text="Here is our general result. The setup: data points x one through x n are sampled i.i.d. uniformly from a compact Riemannian manifold M. Observations are y i equals f star of x i plus Gaussian noise."
        ) as tracker:
            self.play(Write(title), run_time=tracker.duration * 0.1)

            # Visual: manifold with scattered data points
            manifold_curve = ParametricFunction(
                lambda t: np.array([2.5 * np.cos(t), 1.2 * np.sin(t), 0]),
                t_range=[0, TAU], color=BLUE, stroke_width=2
            ).shift(DOWN * 1.5)

            np.random.seed(12)
            data_dots = VGroup()
            for _ in range(15):
                t = np.random.uniform(0, TAU)
                noise = np.random.normal(0, 0.05)
                x = (2.5 + noise) * np.cos(t)
                y = (1.2 + noise) * np.sin(t)
                data_dots.add(Dot([x, y, 0], radius=0.06, color=YELLOW, fill_opacity=0.8))
            data_dots.shift(DOWN * 1.5)

            m_label = MathTex(r"x_i \sim \text{Unif}(M)", font_size=24, color=YELLOW).shift( UP * 2.8)
            noise_label = MathTex(r"y_i = f^*(x_i) + \varepsilon_i,\;\; \varepsilon_i \sim \mathcal{N}(0,\sigma^2)", font_size=22, color=WHITE).next_to(m_label, DOWN, buff=0.2)

            self.play(Create(manifold_curve), run_time=tracker.duration * 0.2)
            self.play(*[GrowFromCenter(d) for d in data_dots], run_time=tracker.duration * 0.25)
            self.play(FadeIn(m_label), FadeIn(noise_label), run_time=tracker.duration * 0.2)
            self.wait(max(0, tracker.duration * 0.15))

        with self.voiceover(
            text="The target function f star is invariant under G, meaning f star of g dot x equals f star of x for all g in G. And f star belongs to the Sobolev space H s of M — meaning it has s square-integrable derivatives."
        ) as tracker:
            inv_eq = MathTex(r"f^*(g \cdot x) = f^*(x) \;\; \forall g \in G", font_size=26, color=GREEN).next_to(noise_label, DOWN, buff=0.3)
            sob_eq = MathTex(r"f^* \in H^s(M)", font_size=26, color=TEAL).next_to(inv_eq, DOWN, buff=0.2)
            # Show symmetry action on manifold
            sym_arrow = CurvedArrow(
                manifold_curve.point_from_proportion(0.2),
                manifold_curve.point_from_proportion(0.8),
                color=GREEN, stroke_width=2, tip_length=0.12
            )
            g_label = MathTex(r"g", font_size=20, color=GREEN).next_to(sym_arrow, UP, buff=0.05)
            self.play(Write(inv_eq), Create(sym_arrow), FadeIn(g_label), run_time=tracker.duration * 0.4)
            self.play(Write(sob_eq), run_time=tracker.duration * 0.3)
            self.wait(max(0, tracker.duration * 0.2))

        # ── Estimator ──
        self.play(
            FadeOut(manifold_curve), FadeOut(data_dots), FadeOut(m_label), FadeOut(noise_label),
            FadeOut(inv_eq), FadeOut(sym_arrow), FadeOut(g_label), FadeOut(sob_eq),
            run_time=0.4
        )

        with self.voiceover(
            text="The estimator is kernel ridge regression restricted to the invariant function class H s G — minimizing empirical loss plus a Sobolev norm penalty, only over G-invariant functions."
        ) as tracker:
            est_box = RoundedRectangle(width=10, height=1.6, color=BLUE, fill_opacity=0.08, corner_radius=0.1, stroke_width=1.5)
            est_box.next_to(title, DOWN, buff=0.4)
            est_eq = MathTex(
                r"\hat{f} = \arg\min_{f \in H^s_G(M)}",
                r"\left\{ \frac{1}{n}\sum_{i=1}^n (y_i - f(x_i))^2 + \eta\|f\|_{H^s}^2 \right\}",
                font_size=24, color=WHITE
            ).move_to(est_box)
            est_label = Tex("Kernel Ridge Regression over G-invariant Sobolev class", font_size=28, color=BLUE).next_to(est_box, DOWN, buff=0.15)

            self.play(Create(est_box), Write(est_eq), run_time=tracker.duration * 0.5)
            self.play(FadeIn(est_label), run_time=0.5)
            self.wait(max(0, tracker.duration * 0.2))

        # ── Theorem statement ──
        self.play(FadeOut(est_box), FadeOut(est_eq), FadeOut(est_label), run_time=0.3)

        with self.voiceover(
            text="The main theorem. The expected L2 error satisfies: E of the squared L2 norm of f hat minus f star is bounded by C d times vol of M mod G over n, raised to the power s over s plus d effective, where d effective equals dim M minus dim G."
        ) as tracker:
            thm_label = Text("Theorem (Main Result)", font_size=24, color=YELLOW, weight=BOLD).next_to(title, DOWN, buff=0.4)
            thm_eq = MathTex(
                r"\mathbb{E}\|\hat{f} - f^*\|_{L^2}^2",
                r"\;\leq\;",
                r"\left(\frac{C_d \cdot \text{vol}(M/G)}{n}\right)^{\!\frac{s}{s+d_{\text{eff}}}}",
                font_size=33, color=WHITE
            ).next_to(thm_label, DOWN, buff=0.4)
            thm_eq[2].set_color(WHITE)

            # Key: define d_eff immediately
            d_eff_def = MathTex(
                r"d_{\text{eff}} = \dim(M) - \dim(G)",
                font_size=24, color=WHITE
            ).next_to(thm_eq, DOWN, buff=0.3)
            d_eff_note = Tex("(effective dimension of the quotient M/G)", font_size=28, color=WHITE).next_to(d_eff_def, DOWN, buff=0.4)

            thm_box = SurroundingRectangle(VGroup(thm_label, thm_eq, d_eff_def), color=YELLOW, stroke_width=2, buff=0.2, corner_radius=0.1)
            thm_box.scale([1.2, 0.85, 1])
            self.play(Write(thm_label), run_time=0.4)
            self.play(Write(thm_eq), run_time=tracker.duration * 0.3)
            self.play(Write(d_eff_def), FadeIn(d_eff_note), run_time=tracker.duration * 0.2)
            self.play(Create(thm_box), run_time=0.4)
            self.wait(max(0, tracker.duration * 0.15))

        # ── Two types of gains ──
        self.play(FadeOut(thm_box), FadeOut(d_eff_note),
                  VGroup(thm_eq, d_eff_def).animate.scale(0.8).to_edge(UP, buff=1.3),
                  FadeOut(thm_label), run_time=0.4)

        with self.voiceover(
            text="This reveals two distinct gains. First, a multiplicative gain: the volume of the quotient space M mod G is smaller than the volume of M itself. The larger the group orbits, the smaller this volume."
        ) as tracker:
            gain1_title = Text("Gain 1: Multiplicative (volume reduction)", font_size=24, color=YELLOW, weight=BOLD).shift(UP * 0.1)

            # Visual: full circle vs smaller sector (using Arc + Lines instead of Sector)
            full_circle = Circle(radius=1.0, color=RED, stroke_width=2.5, fill_opacity=0.15, fill_color=RED).shift(LEFT * 3 + DOWN * 1.3)
            full_label = MathTex(r"\text{vol}(M)", font_size=20, color=RED).next_to(full_circle, DOWN, buff=0.15)

            # Quotient: use Arc + two lines to form a sector shape (avoiding Sector class issue)
            sector_arc = Arc(radius=1.0, start_angle=0, angle=PI / 2, color=GREEN, stroke_width=2.5).shift(RIGHT * 3 + DOWN * 1.3)
            sector_line1 = Line(
                RIGHT * 3 + DOWN * 1.3,
                RIGHT * 3 + DOWN * 1.3 + RIGHT * 1.0,
                color=GREEN, stroke_width=2.5
            )
            sector_line2 = Line(
                RIGHT * 3 + DOWN * 1.3,
                RIGHT * 3 + DOWN * 1.3 + UP * 1.0,
                color=GREEN, stroke_width=2.5
            )
            sector_fill = Polygon(
                RIGHT * 3 + DOWN * 1.3,
                RIGHT * 3 + DOWN * 1.3 + RIGHT * 1.0,
                RIGHT * 3 + DOWN * 1.3 + 0.707 * RIGHT + 0.707 * UP,
                RIGHT * 3 + DOWN * 1.3 + UP * 1.0,
                color=GREEN, fill_opacity=0.2, stroke_width=0
            )
            sector_group = VGroup(sector_fill, sector_arc, sector_line1, sector_line2)
            sector_label = MathTex(r"\text{vol}(M/G)", font_size=20, color=GREEN).next_to(sector_group, DOWN, buff=0.15)
            ratio_text = MathTex(r"= \frac{\text{vol}(M)}{|G|}", font_size=18, color=GREEN).next_to(sector_label, DOWN, buff=0.08)

            arrow_mult = Arrow(LEFT * 1.0, RIGHT * 1.0, color=YELLOW, stroke_width=2, tip_length=0.12).shift(DOWN * 1.3)
            arrow_label = MathTex(r"G\text{-action}", font_size=16, color=YELLOW).next_to(arrow_mult, UP, buff=0.05)

            self.play(Write(gain1_title), run_time=0.4)
            self.play(Create(full_circle), FadeIn(full_label), run_time=tracker.duration * 0.15)
            self.play(GrowArrow(arrow_mult), FadeIn(arrow_label), run_time=0.3)
            self.play(FadeIn(sector_group), FadeIn(sector_label), FadeIn(ratio_text), run_time=tracker.duration * 0.2)
            self.wait(max(0, tracker.duration * 0.2))

        with self.voiceover(
            text="Second, an exponential gain in the exponent: the rate becomes n to the power minus s over s plus d effective. Since d effective equals dim M minus dim G is smaller than dim M, the exponent is larger — meaning faster convergence. This is an exponential improvement in the sample complexity."
        ) as tracker:
            self.play(
                FadeOut(full_circle), FadeOut(full_label), FadeOut(sector_group), FadeOut(sector_label),
                FadeOut(ratio_text), FadeOut(gain1_title), FadeOut(arrow_mult), FadeOut(arrow_label),
                run_time=0.3
            )

            gain2_title = Text("Gain 2: Exponential (dimension reduction in exponent)", font_size=24, color=YELLOW, weight=BOLD).shift(UP * 0.1)

            # Comparison table
            without_sym = MathTex(r"\text{Without symmetry: rate} = n^{-\frac{s}{s + \dim(M)}}", font_size=27, color=WHITE).shift(DOWN * 0.6)
            with_sym = MathTex(r"\text{With symmetry: rate} = n^{-\frac{s}{s + \dim(M) - \dim(G)}}", font_size=27, color=WHITE).shift(DOWN * 1.3)

            # Arrow showing "larger exponent = faster"
            faster_brace = Brace(VGroup(without_sym, with_sym), direction=RIGHT, color=YELLOW)
            faster_label = Text("Larger exponent\n= Faster!", font_size=16, color=YELLOW, weight=BOLD).next_to(faster_brace, RIGHT, buff=0.15)

            # Concrete example
            example_text = MathTex(
                r"\text{E.g. } \dim(M)=6,\; \dim(G)=3 \;\Rightarrow\; d_{\text{eff}}=3",
                font_size=27, color=WHITE
            ).shift(DOWN * 2.5)
            example_rates = MathTex(
                r"n^{-s/(s+6)} \;\;\text{vs}\;\; n^{-s/(s+3)}",
                font_size=27, color=WHITE
            ).next_to(example_text, DOWN, buff=0.1)

            self.play(Write(gain2_title), run_time=0.4)
            self.play(FadeIn(without_sym), run_time=tracker.duration * 0.12)
            self.play(FadeIn(with_sym), run_time=tracker.duration * 0.12)
            self.play(Create(faster_brace), FadeIn(faster_label), run_time=tracker.duration * 0.12)
            self.play(FadeIn(example_text), FadeIn(example_rates), run_time=tracker.duration * 0.12)
            self.wait(max(0, tracker.duration * 0.2))

        # ── Minimax optimal ──
        with self.voiceover(
            text="Moreover, this rate is minimax optimal — meaning our upper bound matches the information-theoretic lower bound. No estimator, no matter how clever, can do better."
        ) as tracker:
            self.play(
                FadeOut(gain2_title), FadeOut(without_sym), FadeOut(with_sym),
                FadeOut(faster_brace), FadeOut(faster_label),
                FadeOut(example_text), FadeOut(example_rates),
                run_time=0.3
            )

            # Upper bound = Lower bound visual
            ub_bar = Rectangle(width=5, height=0.6, color=GREEN, fill_opacity=0, stroke_width=1.5).shift(UP * 0.0)
            ub_label = MathTex(r"\text{Upper Bound (our estimator achieves)}", font_size=18, color=WHITE).move_to(ub_bar)
            lb_bar = Rectangle(width=5, height=0.6, color=ORANGE, fill_opacity=0, stroke_width=1.5).shift(DOWN * 0.8)
            lb_label = MathTex(r"\text{Lower Bound (no estimator can beat)}", font_size=18, color=WHITE).move_to(lb_bar)

            match_brace = Brace(VGroup(ub_bar, lb_bar), direction=RIGHT, color=YELLOW)
            match_label = Text("Match!\n(Tight)", font_size=18, color=YELLOW, weight=BOLD).next_to(match_brace, RIGHT, buff=0.15)

            minimax_badge = VGroup(
                RoundedRectangle(width=4.0, height=0.8, color=YELLOW, fill_opacity=0, corner_radius=0.1, stroke_width=2),
                Text("MINIMAX OPTIMAL", font_size=22, color=YELLOW, weight=BOLD),
            )
            minimax_badge[1].move_to(minimax_badge[0])
            minimax_badge.shift(DOWN * 2.3)

            self.play(FadeIn(ub_bar), FadeIn(ub_label), run_time=tracker.duration * 0.2)
            self.play(FadeIn(lb_bar), FadeIn(lb_label), run_time=tracker.duration * 0.2)
            self.play(Create(match_brace), FadeIn(match_label), run_time=tracker.duration * 0.2)
            self.play(FadeIn(minimax_badge, scale=1.2), run_time=tracker.duration * 0.2)
            self.wait(max(0, tracker.duration * 0.1))

        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.5)
        self.wait(0.3)


# ═══════════════════════════════════════════════════════════════
# SCENE 23: Proof Idea — Spectral Methods & Fourier Sparsity
# ═══════════════════════════════════════════════════════════════
class Scene23ProofIdea(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title = Text("Proof Idea: Spectral Methods on Manifolds", font_size=30, color=WHITE, weight=BOLD)
        title.to_edge(UP, buff=0.4)

        # ══════════════════════════════════════════════════════
        # PART 1: Simple example — Circle with reflection θ → −θ
        # ══════════════════════════════════════════════════════
        with self.voiceover(
            text="Let's build intuition with a simple example. Take M equals the unit circle S one, parameterized by angle theta from 0 to 2 pi. Let G be the two-element group consisting of the identity and the reflection theta maps to minus theta."
        ) as tracker:
            self.play(Write(title), run_time=tracker.duration * 0.1)

            circle = Circle(radius=1.3, color=WHITE, stroke_width=2).shift(LEFT * 3.0 + DOWN * 0.8)
            # Mark angle theta and -theta
            theta_val = PI / 3
            dot_theta = Dot(circle.point_at_angle(theta_val), radius=0.08, color=YELLOW)
            dot_neg_theta = Dot(circle.point_at_angle(-theta_val), radius=0.08, color=YELLOW)
            label_theta = MathTex(r"\theta", font_size=18, color=YELLOW).next_to(dot_theta, UR, buff=0.05)
            label_neg_theta = MathTex(r"-\theta", font_size=18, color=YELLOW).next_to(dot_neg_theta, DR, buff=0.05)

            # x-axis as mirror line
            x_axis = DashedLine(
                circle.get_center() + LEFT * 1.7,
                circle.get_center() + RIGHT * 1.7,
                color=BLUE, stroke_width=2
            )
            mirror_label = Text("mirror", font_size=12, color=BLUE).next_to(x_axis, DOWN, buff=0.05)

            m_label = MathTex(r"M = S^1", font_size=20, color=WHITE).next_to(circle, DOWN, buff=0.35)
            g_label = MathTex(r"G = \{e, \;\theta \mapsto -\theta\}", font_size=18, color=BLUE).next_to(m_label, DOWN, buff=0.1)

            # Reflection arrow
            refl_arrow = CurvedArrow(
                dot_theta.get_center() + UP * 0.1,
                dot_neg_theta.get_center() + DOWN * 0.1,
                color=GREEN, stroke_width=2, tip_length=0.1, angle=-PI / 2
            )

            self.play(Create(circle), Create(x_axis), FadeIn(mirror_label), run_time=tracker.duration * 0.2)
            self.play(FadeIn(dot_theta), FadeIn(dot_neg_theta), FadeIn(label_theta), FadeIn(label_neg_theta), run_time=tracker.duration * 0.15)
            self.play(FadeIn(m_label), FadeIn(g_label), Create(refl_arrow), run_time=tracker.duration * 0.25)
            self.wait(max(0, tracker.duration * 0.2))

        # ══════════════════════════════════════════════════════
        # PART 2: Fourier basis + invariance kills sin modes
        # ══════════════════════════════════════════════════════
        with self.voiceover(
            text="The Fourier basis on the circle consists of 1, cos k theta, and sin k theta for k equals 1, 2, 3, and so on. Now, an invariant function satisfies f of minus theta equals f of theta. Since cosine of minus k theta equals cosine of k theta, all cosine modes are invariant. But since sine of minus k theta equals negative sine of k theta, all sine modes are anti-symmetric and must have zero coefficient."
        ) as tracker:
            fourier_text = VGroup(
                MathTex(r"\text{Fourier basis on } S^1:", font_size=20, color=WHITE),
                MathTex(r"1,\; \cos(k\theta),\; \sin(k\theta),\;\; k=1,2,3,\ldots", font_size=20, color=WHITE),
            ).arrange(DOWN, buff=0.1).shift(RIGHT * 2.5 + UP * 0.8)

            inv_cond = MathTex(r"f(-\theta) = f(\theta)", font_size=22, color=YELLOW).next_to(fourier_text, DOWN, buff=0.3)

            cos_check = MathTex(r"\cos(k(-\theta)) = \cos(k\theta) \;\checkmark", font_size=20, color=GREEN).next_to(inv_cond, DOWN, buff=0.25)
            sin_cross = MathTex(r"\sin(k(-\theta)) = -\sin(k\theta) \;\times", font_size=20, color=RED).next_to(cos_check, DOWN, buff=0.15)

            self.play(Write(fourier_text), run_time=tracker.duration * 0.2)
            self.play(Write(inv_cond), run_time=tracker.duration * 0.15)
            self.play(Write(cos_check), run_time=tracker.duration * 0.2)
            self.play(Write(sin_cross), run_time=tracker.duration * 0.2)
            self.wait(max(0, tracker.duration * 0.15))

        # ══════════════════════════════════════════════════════
        # PART 3: Fourier bar chart — sin modes killed
        # ══════════════════════════════════════════════════════
        with self.voiceover(
            text="So only cosine modes survive — exactly half the Fourier basis. This means the invariant function space has half the effective degrees of freedom. In our framework, this corresponds to vol of M mod G equals one half times vol of M."
        ) as tracker:
            self.play(
                FadeOut(fourier_text), FadeOut(inv_cond), FadeOut(cos_check), FadeOut(sin_cross),
                FadeOut(circle), FadeOut(x_axis), FadeOut(mirror_label),
                FadeOut(dot_theta), FadeOut(dot_neg_theta), FadeOut(label_theta), FadeOut(label_neg_theta),
                FadeOut(m_label), FadeOut(g_label), FadeOut(refl_arrow),
                run_time=0.3
            )

            # Fourier bar chart — clearly labeled sin and cos
            bar_ax = Axes(
                x_range=[0, 11, 1], y_range=[0, 1.2, 0.5],
                x_length=8, y_length=2.0,
                axis_config={"color": WHITE, "include_ticks": False},
                tips=False,
            ).shift(DOWN * 0.3)

            bar_label_x = Text("Fourier mode", font_size=14, color=WHITE).next_to(bar_ax, DOWN, buff=0.3)
            bar_label_y = Text("Coefficient", font_size=14, color=WHITE).next_to(bar_ax, LEFT, buff=0.05).rotate(PI / 2)

            # Create bars: cos(1), sin(1), cos(2), sin(2), cos(3), sin(3), cos(4), sin(4), cos(5), sin(5)
            mode_labels_text = [
                r"\cos 1", r"\sin 1", r"\cos 2", r"\sin 2", r"\cos 3",
                r"\sin 3", r"\cos 4", r"\sin 4", r"\cos 5", r"\sin 5"
            ]
            np.random.seed(7)
            bars = VGroup()
            bar_is_cos = []
            for i in range(10):
                is_cos = (i % 2 == 0)
                bar_is_cos.append(is_cos)
                height = 0.4 + 0.5 * np.random.random()
                color = GREEN if is_cos else RED
                bar = Rectangle(
                    width=0.55, height=height * 2,
                    color=color, fill_opacity=0.6, stroke_width=1
                )
                bar.move_to(bar_ax.c2p(i + 1, 0))
                bar.align_to(bar_ax.c2p(i + 1, 0), DOWN)
                bars.add(bar)

            # Mode labels below each bar
            k_labels = VGroup()
            for i, lbl_text in enumerate(mode_labels_text):
                lbl = MathTex(lbl_text, font_size=12, color=WHITE).next_to(bar_ax.c2p(i + 1, 0), DOWN, buff=0.05)
                # lbl.rotate(-PI / 6)
                k_labels.add(lbl)

            self.play(Create(bar_ax), FadeIn(bar_label_x), FadeIn(bar_label_y), run_time=0.4)
            self.play(*[GrowFromEdge(b, DOWN) for b in bars], FadeIn(k_labels), run_time=tracker.duration * 0.15)

            # Animate: sin bars (red, odd indices) get crossed out and fade
            sin_bars = [bars[i] for i in range(10) if not bar_is_cos[i]]
            cross_marks = VGroup()
            for bar in sin_bars:
                cm = Cross(bar, color=RED, stroke_width=2.5)
                cross_marks.add(cm)

            killed_label = Text("Killed by invariance!", font_size=16, color=RED, weight=BOLD).shift(RIGHT * 4 + UP * 1.0)
            survive_label = Text("Survive (invariant)", font_size=16, color=GREEN, weight=BOLD).shift(LEFT * 4 + UP * 1.0)

            self.play(*[Create(cm) for cm in cross_marks], FadeIn(killed_label), run_time=tracker.duration * 0.15)
            self.play(*[bar.animate.set_opacity(0.1) for bar in sin_bars], run_time=tracker.duration * 0.1)
            self.play(FadeIn(survive_label), run_time=0.3)

            vol_note = MathTex(r"\text{vol}(M/G) = \tfrac{1}{2}\,\text{vol}(M)", font_size=24, color=GREEN).shift(DOWN * 2.5)
            sparsity_note = Tex(
                r"Half the modes $\rightarrow$ half the effective complexity",
                font_size=29,
                color=WHITE
            ).next_to(vol_note, DOWN, buff=0.1)
            self.play(Write(vol_note), FadeIn(sparsity_note), run_time=tracker.duration * 0.15)
            self.wait(max(0, tracker.duration * 0.1))

        # ══════════════════════════════════════════════════════
        # PART 4: Key insight — Fourier sparsity → sample gain
        # ══════════════════════════════════════════════════════
        with self.voiceover(
            text="The key insight: fewer surviving Fourier modes means the invariant function class is smaller. Kernel ridge regression over a smaller class needs fewer samples to achieve the same error. This is where the sample complexity gain comes from."
        ) as tracker:
            self.play(
                FadeOut(bar_ax), FadeOut(bars), FadeOut(cross_marks), FadeOut(k_labels),
                FadeOut(bar_label_x), FadeOut(bar_label_y),
                FadeOut(killed_label), FadeOut(survive_label), FadeOut(sparsity_note),
                vol_note.animate.shift(UP * 2.5).scale(1.2),
                run_time=0.4
            )

            # Chain of logic
            chain = VGroup(
                MathTex(r"\text{Invariance}", font_size=26, color=GREEN),
                MathTex(r"\Rightarrow", font_size=26, color=WHITE),
                MathTex(r"\text{Fewer eigen-modes}", font_size=26, color=TEAL),
                MathTex(r"\Rightarrow", font_size=26, color=WHITE),
                MathTex(r"\text{Smaller function class}", font_size=26, color=BLUE),
                MathTex(r"\Rightarrow", font_size=26, color=WHITE),
                MathTex(r"\text{Fewer samples needed}", font_size=26, color=YELLOW),
            ).arrange(RIGHT, buff=0.15).shift(DOWN * 0.5)

            # Scale to fit
            if chain.width > 12:
                chain.scale(12 / chain.width)

            self.play(LaggedStart(*[FadeIn(c, shift=RIGHT * 0.2) for c in chain], lag_ratio=0.2), run_time=tracker.duration * 0.6)
            self.wait(max(0, tracker.duration * 0.3))

        # ══════════════════════════════════════════════════════
        # PART 5: Extended Weyl's Law (general manifold)
        # ══════════════════════════════════════════════════════
        self.play(*[FadeOut(m) for m in self.mobjects if m != title], run_time=0.4)

        with self.voiceover(
            text="For general manifolds, we generalize this via the Extended Weyl's Law. On a compact manifold M, the Laplace-Beltrami operator has discrete eigenvalues. The counting function N of lambda counts eigenvalues up to lambda."
        ) as tracker:
            weyl_title = Text("General Case: Extended Weyl's Law", font_size=24, color=YELLOW, weight=BOLD).next_to(title, DOWN, buff=0.4)
            self.play(Write(weyl_title), run_time=tracker.duration * 0.2)

            # Eigenvalue line with dots
            eig_line = NumberLine(x_range=[0, 20, 5], length=8, include_numbers=False, color=WHITE).shift(DOWN * 0.3)
            eig_label = MathTex(r"\lambda", font_size=20, color=WHITE).next_to(eig_line, RIGHT, buff=0.1)
            eig_title_label = Tex("Eigenvalues of Laplacian on $M$", font_size=29, color=WHITE).next_to(eig_line, UP, buff=0.15)

            # Eigenvalues
            eig_positions = [1, 2.5, 4, 5.5, 7, 9, 11, 13, 15.5, 18]
            eig_dots = VGroup()
            for pos in eig_positions:
                eig_dots.add(Dot(eig_line.n2p(pos), radius=0.08, color=TEAL))

            self.play(Create(eig_line), FadeIn(eig_label), FadeIn(eig_title_label), run_time=0.5)
            self.play(LaggedStart(*[GrowFromCenter(d) for d in eig_dots], lag_ratio=0.12), run_time=tracker.duration * 0.4)
            self.wait(max(0, tracker.duration * 0.15))

        with self.voiceover(
            text="Weyl's Law for G-invariant eigenfunctions, extended by Brüning, Heintze, and Donnelly, states: N of lambda is approximately omega d over 2 pi to the d, times vol of M mod G, times lambda to the d over 2. Here d equals dim of M mod G, and omega d is the volume of the unit d-ball."
        ) as tracker:
            # Count bracket showing N(lambda)
            cutoff_pos = eig_line.n2p(12)
            cutoff_line = DashedLine(cutoff_pos + UP * 0.5, cutoff_pos + DOWN * 0.5, color=YELLOW, stroke_width=2)
            cutoff_label = MathTex(r"\lambda", font_size=16, color=YELLOW).next_to(cutoff_line, DOWN, buff=0.1)

            counted_dots = VGroup(*[d for d in eig_dots if d.get_center()[0] < cutoff_pos[0]])
            count_bracket = Brace(counted_dots, direction=DOWN, color=YELLOW)
            count_label = MathTex(r"N(\lambda)", font_size=20, color=YELLOW).next_to(count_bracket, DOWN, buff=0.1)

            self.play(Create(cutoff_line), FadeIn(cutoff_label), run_time=0.3)
            self.play(Create(count_bracket), FadeIn(count_label), run_time=tracker.duration * 0.15)

            weyl_eq = MathTex(
                r"N(\lambda) \;\approx\; \frac{\omega_d}{(2\pi)^d} \cdot \text{vol}(M/G) \cdot \lambda^{d/2}",
                font_size=26, color=WHITE
            ).shift(DOWN * 2.2)
            weyl_note = MathTex(r"d = \dim(M/G) = \dim(M) - \dim(G)", font_size=20, color=GREEN).next_to(weyl_eq, DOWN, buff=0.15)
            weyl_eq_box = SurroundingRectangle(VGroup(weyl_eq, weyl_note), color=YELLOW, stroke_width=1.5, buff=0.15, corner_radius=0.08)

            self.play(Write(weyl_eq), run_time=tracker.duration * 0.3)
            self.play(Write(weyl_note), Create(weyl_eq_box), run_time=tracker.duration * 0.2)
            self.wait(max(0, tracker.duration * 0.15))

        # ══════════════════════════════════════════════════════
        # PART 6: Proof challenge — boundary condition
        # ══════════════════════════════════════════════════════
        self.play(
            FadeOut(eig_line), FadeOut(eig_label), FadeOut(eig_title_label), FadeOut(eig_dots),
            FadeOut(count_bracket), FadeOut(count_label), FadeOut(cutoff_line), FadeOut(cutoff_label),
            FadeOut(weyl_eq), FadeOut(weyl_note), FadeOut(weyl_eq_box), FadeOut(weyl_title),
            run_time=0.4
        )

        with self.voiceover(
            text="A key proof challenge: the quotient space M mod G often has a boundary, even when M itself does not. For our circle example, M mod G is the half-circle from 0 to pi — it has endpoints as boundary."
        ) as tracker:
            challenge_title = Text("Proof Challenge: Boundary Emerges", font_size=18, color=WHITE, weight=BOLD).next_to(title, DOWN, buff=0.4)

            # Full circle → half arc
            full_c = Circle(radius=1.0, color=WHITE, stroke_width=2, fill_opacity=0.1).shift(LEFT * 3 )
            full_c_label = MathTex(r"M = S^1", font_size=18, color=WHITE).next_to(full_c, DOWN, buff=0.15)
            full_c_note = Tex("(no boundary)", font_size=25, color=WHITE).next_to(full_c_label, DOWN, buff=0.05)

            cut_arrow = Arrow(LEFT * 1.0, RIGHT * 1.0, color=YELLOW, stroke_width=2.5, tip_length=0.12)
            cut_label = MathTex(r"M/G", font_size=18, color=YELLOW).next_to(cut_arrow, UP, buff=0.05)

            half_c = Arc(radius=1.0, start_angle=0, angle=PI, color=GREEN, stroke_width=3).shift(RIGHT * 3 )
            # Boundary dots at endpoints
            bd_dot1 = Dot(half_c.get_start(), radius=0.1, color=RED)
            bd_dot2 = Dot(half_c.get_end(), radius=0.1, color=RED)
            boundary_label = Text("Boundary!", font_size=16, color=RED, weight=BOLD).next_to(VGroup(bd_dot1, bd_dot2), DOWN, buff=0.3)
            half_c_label = MathTex(r"M/G = [0, \pi]", font_size=18, color=GREEN).next_to(boundary_label, DOWN, buff=0.1)

            self.play(Write(challenge_title), run_time=0.3)
            self.play(Create(full_c), FadeIn(full_c_label), FadeIn(full_c_note), run_time=tracker.duration * 0.2)
            self.play(GrowArrow(cut_arrow), FadeIn(cut_label), run_time=tracker.duration * 0.1)
            self.play(Create(half_c), FadeIn(bd_dot1), FadeIn(bd_dot2), FadeIn(boundary_label), FadeIn(half_c_label), run_time=tracker.duration * 0.3)
            self.wait(max(0, tracker.duration * 0.2))

        with self.voiceover(
            text="The Neumann boundary condition emerges naturally from the invariance: the derivative of f at the boundary must be zero. This is because invariance under reflection forces the function to have a flat slope at the mirror line. Also, one must carefully define vol of M mod G, since the quotient may not always be a smooth manifold — it can be an orbifold."
        ) as tracker:
            neumann_eq = MathTex(r"\frac{df}{d\theta}\bigg|_{\theta=0,\pi} = 0", font_size=26, color=YELLOW)
            neumann_eq.shift(DOWN * 2.8)
            neumann_box = SurroundingRectangle(neumann_eq, color=YELLOW, stroke_width=1.5, buff=0.12, corner_radius=0.08)
            neumann_label = Text("Neumann BC (from invariance!)", font_size=16, color=YELLOW).next_to(neumann_box, DOWN, buff=0.1)

            # Visual: flat function at boundary
            mini_ax = Axes(x_range=[0, PI, 1], y_range=[-1, 1, 0.5], x_length=2.5, y_length=1.2,
                          axis_config={"color": GREY, "include_ticks": False}, tips=False).shift(RIGHT * 3.2 + DOWN * 2.8)
            cos_curve = mini_ax.plot(lambda x: np.cos(x), x_range=[0, PI], color=GREEN, stroke_width=2)
            flat_dot1 = Dot(mini_ax.c2p(0, 1), radius=0.06, color=RED)
            flat_dot2 = Dot(mini_ax.c2p(PI, -1), radius=0.06, color=RED)
            flat_note = Text("flat at endpoints", font_size=11, color=RED).next_to(mini_ax, RIGHT, buff=0.05)

            self.play(Write(neumann_eq), Create(neumann_box), run_time=tracker.duration * 0.25)
            self.play(FadeIn(neumann_label), run_time=0.3)
            self.play(Create(mini_ax), Create(cos_curve), FadeIn(flat_dot1), FadeIn(flat_dot2), FadeIn(flat_note), run_time=tracker.duration * 0.25)
            self.wait(max(0, tracker.duration * 0.2))

        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.5)
        self.wait(0.3)


# ═══════════════════════════════════════════════════════════════
# SCENE 24: Beyond Learning — Extensions & Summary
# ═══════════════════════════════════════════════════════════════
# ═══════════════════════════════════════════════════════════════
# SCENE 24: Beyond Learning — Extensions & Summary + References
# ═══════════════════════════════════════════════════════════════
class Scene24BeyondLearning(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title = Text("Beyond Regression: Extensions", font_size=36, color=WHITE, weight=BOLD)
        title.to_edge(UP, buff=0.5)

        # ── Part 1: Extensions overview ──
        with self.voiceover(
            text="These sample complexity results extend beyond regression to several other settings. Let me highlight the key extensions."
        ) as tracker:
            self.play(Write(title), run_time=tracker.duration * 0.3)
            self.wait(max(0, tracker.duration * 0.5))

        # ── Part 2: Optimal Transport ──
        with self.voiceover(
            text="First, optimal transport and Wasserstein distances. When comparing two distributions that are both invariant under G, the sample complexity of estimating their Wasserstein distance also improves — with the same vol of M mod G factor appearing."
        ) as tracker:
            ot_title = Text("1. Optimal Transport", font_size=18, color=WHITE, weight=BOLD).next_to(title, DOWN, buff=0.5).align_to(title, LEFT).shift(RIGHT * 0.5)

            mu_curve = ParametricFunction(
                lambda t: np.array([0.8 * np.cos(t) - 2, 0.5 * np.sin(t) - 1.5, 0]),
                t_range=[0, TAU], color=BLUE, stroke_width=2
            )
            nu_curve = ParametricFunction(
                lambda t: np.array([0.8 * np.cos(t) + 1, 0.5 * np.sin(t) - 1.5, 0]),
                t_range=[0, TAU], color=ORANGE, stroke_width=2
            )
            mu_label = MathTex(r"\mu", font_size=24, color=BLUE).next_to(mu_curve, DOWN, buff=0.1)
            nu_label = MathTex(r"\nu", font_size=24, color=ORANGE).next_to(nu_curve, DOWN, buff=0.1)
            wass_arrow = Arrow(mu_curve.get_right(), nu_curve.get_left(), color=YELLOW, stroke_width=2, tip_length=0.1)
            wass_label = MathTex(r"W_p(\mu, \nu)", font_size=20, color=YELLOW).next_to(wass_arrow, UP, buff=0.05)

            ot_gain = MathTex(
                r"\text{Gain: vol}(M/G) \text{ replaces vol}(M)",
                font_size=24, color=GREEN
            ).shift(DOWN * 2.8)

            self.play(Write(ot_title), run_time=0.3)
            self.play(Create(mu_curve), Create(nu_curve), FadeIn(mu_label), FadeIn(nu_label), run_time=tracker.duration * 0.2)
            self.play(GrowArrow(wass_arrow), FadeIn(wass_label), run_time=tracker.duration * 0.15)
            self.play(FadeIn(ot_gain), run_time=tracker.duration * 0.15)
            self.wait(max(0, tracker.duration * 0.2))

        # ── Part 3: Kernel MMD ──
        self.play(FadeOut(mu_curve), FadeOut(nu_curve), FadeOut(mu_label), FadeOut(nu_label),
                  FadeOut(wass_arrow), FadeOut(wass_label), FadeOut(ot_gain), FadeOut(ot_title), run_time=0.3)

        with self.voiceover(
            text="Second, kernel Maximum Mean Discrepancy, or MMD. Here the story is more subtle: the gain depends not just on the volume ratio but also on the spectral zeta function of the manifold — which encodes how fast eigenvalues of the Laplacian grow. Different kernels lead to different gains depending on their smoothness."
        ) as tracker:
            mmd_title = Text("2. Kernel MMD (more subtle)", font_size=18, color=WHITE, weight=BOLD).next_to(title, DOWN, buff=0.5).align_to(title, LEFT).shift(RIGHT * 0.5)

            mmd_formula = MathTex(
                r"\text{MMD gain} \sim \frac{\zeta_{M/G}(s)}{\zeta_M(s)}",
                font_size=30, color=WHITE
            )
            mmd_note = VGroup(
                Tex(r"$\zeta$ = spectral zeta function", font_size=25, color=WHITE),
                MathTex(r"\zeta_M(s) = \sum_k \lambda_k^{-s}", font_size=23, color=WHITE),
                Tex(r"Encodes eigenvalue growth rate", font_size=25, color=WHITE),
            ).arrange(DOWN, buff=0.1).next_to(mmd_formula, DOWN, buff=0.3)

            self.play(Write(mmd_title), run_time=0.3)
            self.play(Write(mmd_formula), run_time=tracker.duration * 0.3)
            self.play(FadeIn(mmd_note), run_time=tracker.duration * 0.3)
            self.wait(max(0, tracker.duration * 0.2))

        # ── Part 4: Other extensions ──
        self.play(FadeOut(mmd_title), FadeOut(mmd_formula), FadeOut(mmd_note), run_time=0.3)

        with self.voiceover(
            text="Third, these ideas extend to reinforcement learning and MDPs — where invariance of the transition kernel leads to sample complexity gains for policy evaluation. Fourth, to diffusion models, where equivariant score functions learn faster. And fifth, to geometric stability results — how robust is the gain when symmetry is only approximate?"
        ) as tracker:
            ext_title = Text("3-5. Further Extensions", font_size=18, color=WHITE, weight=BOLD).next_to(title, DOWN, buff=0.5).align_to(title, LEFT).shift(RIGHT * 0.5)

            extensions = VGroup(
                VGroup(
                    Text("3. RL / MDPs:", font_size=18, color=WHITE, weight=BOLD),
                    Tex(r"Invariant transitions $\rightarrow$ faster policy evaluation", font_size=24, color=WHITE),
                ).arrange(RIGHT, buff=0.2),
                VGroup(
                    Text("4. Diffusion Models:", font_size=18, color=WHITE, weight=BOLD),
                    Tex(r"Equivariant score $\rightarrow$ faster generation", font_size=24, color=WHITE),
                ).arrange(RIGHT, buff=0.2),
                VGroup(
                    Text("5. Geometric Stability:", font_size=18, color=WHITE, weight=BOLD),
                    Tex(r"Robustness under approximate symmetry", font_size=24, color=WHITE),
                ).arrange(RIGHT, buff=0.2),
            ).arrange(DOWN, aligned_edge=LEFT, buff=0.25).shift(DOWN * 0.8)

            self.play(Write(ext_title), run_time=0.3)
            self.play(
                LaggedStart(*[FadeIn(ext, shift=RIGHT * 0.3) for ext in extensions], lag_ratio=0.3),
                run_time=tracker.duration * 0.5
            )
            self.wait(max(0, tracker.duration * 0.3))

        # ── Part 5: Summary & takeaway ──
        self.play(FadeOut(ext_title), FadeOut(extensions), run_time=0.3)

        with self.voiceover(
            text="To summarize: symmetry provably reduces sample complexity. The gain is both multiplicative through volume reduction, and exponential through dimension reduction. The result is minimax optimal, and the proof uses spectral geometry — specifically an extended Weyl's law on the quotient space."
        ) as tracker:
            summary_title = Text("Summary", font_size=25, color=WHITE, weight=BOLD).next_to(title, DOWN, buff=0.5)

            summary_box = RoundedRectangle(width=10, height=3.5, color=YELLOW, fill_opacity=0, corner_radius=0.15, stroke_width=2).shift(DOWN * 1.0)

            summary_items = VGroup(
                MathTex(r"\text{1. Multiplicative gain: vol}(M/G) < \text{vol}(M)", font_size=26, color=WHITE),
                MathTex(r"\text{2. Exponential gain: } d_{\text{eff}} = \dim M - \dim G < \dim M", font_size=26, color=WHITE),
                MathTex(r"\text{3. Minimax optimal (tight upper + lower bounds)}", font_size=26, color=WHITE),
                MathTex(r"\text{4. Proof: Extended Weyl's Law on } M/G", font_size=26, color=WHITE),
                MathTex(r"\text{5. Extends to OT, MMD, RL, diffusion models}", font_size=26, color=WHITE),
            ).arrange(DOWN, aligned_edge=LEFT, buff=0.2).move_to(summary_box)

            self.play(Write(summary_title), Create(summary_box), run_time=0.5)
            self.play(
                LaggedStart(*[FadeIn(item, shift=RIGHT * 0.2) for item in summary_items], lag_ratio=0.25),
                run_time=tracker.duration * 0.6
            )
            self.wait(max(0, tracker.duration * 0.25))

        # ── Part 6: References ──
        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.5)

        with self.voiceover(
            text="Here are the key references for further reading."
        ) as tracker:
            ref_title = Text("References", font_size=18, color=WHITE, weight=BOLD).to_edge(UP, buff=0.5)

            ref1 = Tex(
                r"[1] Tahmasebi \& Jegelka, ``The Exact Sample Complexity Gain",
                r" from Invariances for Kernel Regression'', NeurIPS 2023",
                font_size=22, color=WHITE
            )
            ref2 = Tex(
                r"[2] Tahmasebi \& Jegelka, ``Sample Complexity Bounds for Estimating",
                r" Probability Divergences under Invariances'', ICML 2024",
                font_size=22, color=WHITE
            )
            ref3 = Tex(
                r"[3] Elesedy \& Zaidi, ``Provably Strict Generalisation Benefit",
                r" for Equivariant Models'', ICML 2021",
                font_size=22, color=WHITE
            )
            ref4 = Tex(
                r"[4] Lin, Mendes \& Radeschi, ``A Weyl's Law for Singular",
                r" Riemannian Foliations'', Geometry \& Topology, 2025",
                font_size=22, color=WHITE
            )
            ref5 = Tex(
                r"[5] Bietti, Venturi \& Bruna, ``On the Sample Complexity of",
                r" Learning under Geometric Stability'', NeurIPS 2021",
                font_size=22, color=WHITE
            )

            refs = VGroup(ref1, ref2, ref3, ref4, ref5).arrange(
                DOWN, aligned_edge=LEFT, buff=0.25
            ).next_to(ref_title, DOWN, buff=0.4)

            ref_group = VGroup(ref_title, refs)
            if ref_group.height > 6.0:
                ref_group.scale(6.0 / ref_group.height)
            ref_group.center()

            self.play(FadeIn(ref_group, shift=UP * 0.2), run_time=1.0)
            self.wait(max(0, tracker.duration - 1.0))

        self.wait(2.0)
        self.play(FadeOut(ref_group), run_time=0.8)
        self.wait(0.5)




In [ ]:
import subprocess, sys
from pathlib import Path
from IPython.display import Video, display

# ═══════════════════════════════════════════════════════════
# RENDER scenes 102–110
# ═══════════════════════════════════════════════════════════
scene_file = "/content/slides_102_110.py"
scenes_to_render = [
    "Scene20SampleComplexityTitle",
    "Scene21ProvableGains",
    "Scene22GeneralResult",
    "Scene23ProofIdea",
    "Scene24BeyondLearning",
]

output_paths_102_110 = []

for scene_name in scenes_to_render:
    print(f"Rendering {scene_name} with High Quality (-qh)...")
    result = subprocess.run(
        [
            sys.executable, "-m", "manim", "render",
            "-qh",                    # 1080p, 60fps
            "--disable_caching",
            scene_file,
            scene_name,
        ],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"  Error in {scene_name}:\n{result.stderr[-800:]}")
    else:
        found = list(Path("media").rglob(f"{scene_name}.mp4"))
        if found:
            output_paths_102_110.append(str(found[0]))
            print(f"  {found[0]}")
        else:
            print(f"  Rendered OK but file not found")

print(f"\nTotal: {len(output_paths_102_110)} videos")
for p in output_paths_102_110:
    print(f"  {p}")

# ═══════════════════════════════════════════════════════════
# CONCAT — thống nhất 60fps, CRF 20
# ═══════════════════════════════════════════════════════════
list_file = "/content/concat_list_102_110.txt"
with open(list_file, "w") as f:
    for p in output_paths_102_110:
        if Path(p).exists():
            f.write(f"file '{p}'\n")

output_file = "/content/slide_102_110_voiceover.mp4"

cmd = [
    "ffmpeg", "-y",
    "-f", "concat",
    "-safe", "0",
    "-i", list_file,
    "-c:v", "libx264",
    "-preset", "medium",
    "-crf", "20",
    "-r", "60",
    "-pix_fmt", "yuv420p",
    "-c:a", "aac",
    "-b:a", "192k",
    "-movflags", "+faststart",
    output_file
]

result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode == 0 and Path(output_file).exists():
    probe = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries",
         "format=duration,bit_rate", "-show_entries",
         "stream=r_frame_rate,width,height,bit_rate",
         "-of", "default=noprint_wrappers=1",
         output_file],
        capture_output=True, text=True
    )
    print(f"\nFinal: {output_file}")
    print(probe.stdout)
    display(Video(output_file, embed=True, width=854))
else:
    print(f"FFmpeg error:\n{result.stderr[-500:]}")


## Display

In [ ]:
import subprocess
from pathlib import Path
from IPython.display import Video, display

list_file = "/content/concat_list_102_110.txt"
output_file = "/content/slide_102_110_voiceover.mp4"

with open(list_file, "w") as f:
    for p in output_paths_102_110:
        if Path(p).exists():
            f.write(f"file '{Path(p).absolute()}'\n")

cmd = [
    "ffmpeg", "-y",
    "-f", "concat",
    "-safe", "0",
    "-i", list_file,
    "-c:v", "libx264",
    "-preset", "medium",
    "-crf", "20",
    "-r", "60",
    "-pix_fmt", "yuv420p",
    "-c:a", "aac",
    "-b:a", "192k",
    "-movflags", "+faststart",
    output_file
]

result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode == 0 and Path(output_file).exists():
    probe = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries",
         "format=duration,bit_rate", "-show_entries",
         "stream=r_frame_rate,width,height,bit_rate",
         "-of", "default=noprint_wrappers=1",
         output_file],
        capture_output=True, text=True
    )
    print(f"Final: {output_file}")
    print(probe.stdout)
    display(Video(output_file, embed=True, width=854))
else:
    print("FFmpeg error:")
    print(result.stderr[-500:])


# SLIDE 111-115


## Generative

In [ ]:
%%writefile /content/slides_111_115.py

# ══════════════════════════════════════════════════════════════
# FILE: slides_111_115.py — New Directions: Any-Dimensional Models
# Slides 111–115
# ══════════════════════════════════════════════════════════════

import nest_asyncio
nest_asyncio.apply()

import asyncio
import hashlib
import numpy as np
from pathlib import Path

from manim import *
from manim_voiceover import VoiceoverScene
from manim_voiceover.services.base import SpeechService

import edge_tts

# ─────────────────────────────────────────────
# EdgeTTSService
# ─────────────────────────────────────────────
VOICE = "en-US-GuyNeural"


class EdgeTTSService(SpeechService):
    def __init__(self, voice="en-US-GuyNeural", **kwargs):
        self.voice = voice
        super().__init__(**kwargs)

    def generate_from_text(self, text, cache_dir=None, path=None, **kwargs):
        if cache_dir is None:
            cache_dir = self.cache_dir
        Path(cache_dir).mkdir(parents=True, exist_ok=True)

        input_data = {"input_text": text, "service": "edge_tts", "voice": self.voice}

        cached = self.get_cached_result(input_data, cache_dir)
        if cached is not None:
            return cached

        audio_basename = hashlib.md5(text.encode()).hexdigest()
        audio_file = audio_basename + ".mp3"
        full_path = str(Path(cache_dir) / audio_file)

        communicate = edge_tts.Communicate(text, self.voice)
        loop = asyncio.get_event_loop()
        loop.run_until_complete(communicate.save(full_path))

        json_dict = {
            "input_text": text,
            "input_data": input_data,
            "original_audio": audio_file,
        }

        return json_dict


# ═══════════════════════════════════════════════════════════════
# SCENE 25: Title + Motivation (Slides 57–58)
# ═══════════════════════════════════════════════════════════════
class Scene25AnyDimTitle(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        # ── Title ──
        title1 = Text("New Directions:", font_size=44, color=WHITE, weight=BOLD)
        title2 = Text("Any-Dimensional Models", font_size=38, color=WHITE, weight=BOLD)
        title_group = VGroup(title1, title2).arrange(DOWN, buff=0.3).move_to(UP * 0.5)

        with self.voiceover(
          text=(
              "Moving beyond the recent results we've discussed, "
              "let's explore a new and exciting direction: "
              "any-dimensional models — models with a fixed set of parameters "
              "that can be evaluated on inputs of any size."
          )
        ) as tracker:
            self.play(Write(title1), run_time=tracker.duration * 0.3)
            self.play(Write(title2), run_time=tracker.duration * 0.3)
            self.wait(max(0, tracker.duration * 0.3))

        self.play(FadeOut(title_group), run_time=0.4)

        # ── Size Generalization concept ──
        with self.voiceover(
            text="For example, graph neural networks can process graphs of any number of nodes. Deep Sets handle point clouds of any size. The key question is: can we train on small inputs and generalize to larger ones? This is called size generalization."
        ) as tracker:
            # Section title
            sec_title = Text("Any-Dimensional Machine Learning", font_size=36, color=WHITE, weight=BOLD).to_edge(UP, buff=0.4)
            self.play(Write(sec_title), run_time=0.4)

            # Bullet point
            bullet = Text(
                "Fixed parameters, varying input sizes",
                font_size=18, color=WHITE
            ).next_to(sec_title, DOWN, buff=0.3)
            self.play(FadeIn(bullet), run_time=0.4)

            # Small graph (Training)
            train_label = Text("Training set", font_size=18, color=WHITE, weight=BOLD).shift(LEFT * 3.5 + UP * 0.3)
            small_vertices = list(range(1, 7))
            small_edges = [(1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (1, 4), (2, 5)]
            small_graph = Graph(
                small_vertices, small_edges,
                layout="spring", layout_scale=1.0,
                vertex_config={"radius": 0.1, "fill_color": BLUE, "fill_opacity": 0.9},
                edge_config={"stroke_width": 1.5, "stroke_color": GREY_B},
            ).shift(LEFT * 3.5 + DOWN * 1.4)

            train_box = SurroundingRectangle(small_graph, color=BLUE, stroke_width=1.5, buff=0.3, corner_radius=0.1)

            # Large graph (Test)
            test_label = Text("Test set", font_size=18, color=WHITE, weight=BOLD).shift(RIGHT * 3.5 + UP * 0.3)
            large_vertices = list(range(1, 16))
            np.random.seed(42)
            large_edges = []
            for i in range(1, 16):
                for j in range(i + 1, 16):
                    if np.random.random() < 0.2:
                        large_edges.append((i, j))
            large_graph = Graph(
                large_vertices, large_edges,
                layout="spring", layout_scale=1.3,
                vertex_config={"radius": 0.08, "fill_color": BLUE, "fill_opacity": 0.9},
                edge_config={"stroke_width": 1.0, "stroke_color": GREY_B},
            ).shift(RIGHT * 3.5 + DOWN * 1.2)

            test_box = SurroundingRectangle(large_graph, color=GREEN, stroke_width=1.5, buff=0.3, corner_radius=0.1)

            # Arrow between them
            gen_arrow = Arrow(LEFT * 0.8, RIGHT * 0.8, color=YELLOW, stroke_width=3, tip_length=0.15).shift(DOWN * 1.2)
            gen_label = Text("Generalize", font_size=16, color=YELLOW, weight=BOLD).next_to(gen_arrow, UP, buff=0.08)
            same_params = Text("Same parameters!", font_size=14, color=GREEN).next_to(gen_arrow, DOWN, buff=0.08)

            self.play(FadeIn(train_label), Create(small_graph), Create(train_box), run_time=tracker.duration * 0.2)
            self.play(FadeIn(test_label), Create(large_graph), Create(test_box), run_time=tracker.duration * 0.2)
            self.play(GrowArrow(gen_arrow), FadeIn(gen_label), FadeIn(same_params), run_time=tracker.duration * 0.2)
            self.wait(max(0, tracker.duration * 0.2))

        # ── Examples ──
        with self.voiceover(
            text="Examples include graph neural networks, Deep Sets, and machine learning models on point clouds — all sharing the property that one architecture handles any input dimension."
        ) as tracker:
            examples_text = VGroup(
                Text("Examples:", font_size=18, color=TEAL, weight=BOLD),
                Tex(r"$\bullet \quad \text{Graph Neural Networks}$", font_size=25, color=WHITE),
                Tex(r"$\bullet \quad \text{Deep Sets}$", font_size=25, color=WHITE),
                Tex(r"$\bullet \quad \text{Point Cloud Models}$", font_size=25, color=WHITE),
            ).arrange(DOWN, aligned_edge=LEFT, buff=0.1).shift(DOWN * 3.2)
            self.play(FadeIn(examples_text, shift=UP * 0.2), run_time=tracker.duration * 0.4)
            self.wait(max(0, tracker.duration * 0.4))

        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.5)
        self.wait(0.3)


# ═══════════════════════════════════════════════════════════════
# SCENE 26: Representation Stability (Slide 59)
# ═══════════════════════════════════════════════════════════════
class Scene26RepresentationStability(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title = Text("Representation Stability", font_size=32, color=WHITE, weight=BOLD)
        title.to_edge(UP, buff=0.4)
        self.play(Write(title), run_time=0.5)

        # ── Part 1: Linear functions — general vs invariant ──
        with self.voiceover(
            text="How can a fixed number of parameters handle any dimension? Consider linear functions of x in R d. A general linear function has d parameters — one weight per coordinate."
        ) as tracker:
            subtitle = Text("How fixed parameters are enough?", font_size=18, color=WHITE).next_to(title, DOWN, buff=0.3)
            self.play(FadeIn(subtitle), run_time=0.3)

            # General linear — gộp thành 1 MathTex rồi căn giữa
            gen_eq = MathTex(
                r"f(x) = w_1 x_1 + w_2 x_2 + \cdots + w_d x_d",
                r"\quad\Longrightarrow\;",
                r"d \text{ parameters}",
                font_size=26
            ).shift(UP * 0.5)
            gen_eq[0].set_color(WHITE)
            gen_eq[2].set_color(WHITE)
            gen_eq.set_x(0)  # <-- căn giữa theo trục x

            self.play(Write(gen_eq[0]), run_time=tracker.duration * 0.4)
            self.play(FadeIn(gen_eq[1:]), run_time=tracker.duration * 0.2)
            self.wait(max(0, tracker.duration * 0.2))

        with self.voiceover(
            text="But if we require permutation invariance — that is, f of x sigma equals f of x for all permutations sigma in S d — then all weights must be equal. The function becomes w times the sum of all coordinates. Only one parameter, regardless of d!"
        ) as tracker:
            # Invariance condition
            inv_cond = MathTex(
                r"f(x_{\sigma_1}, \ldots, x_{\sigma_d}) = f(x_1, \ldots, x_d),\quad \forall \sigma \in S_d",
                font_size=22, color=WHITE
            ).shift(DOWN * 0.3).set_x(0)  # <-- căn giữa

            # Result — gộp thành 1 MathTex
            inv_result = MathTex(
                r"\Longrightarrow\; f(x) = w(x_1 + x_2 + \cdots + x_d)",
                r"\quad\Longrightarrow\;",
                r"\textbf{only 1 parameter!}",
                font_size=26
            ).shift(DOWN * 1.1)
            inv_result[0].set_color(WHITE)
            inv_result[2].set_color(WHITE)
            inv_result.set_x(0)  # <-- căn giữa

            # Highlight box
            one_param_box = SurroundingRectangle(inv_result, color=GREEN, stroke_width=2, buff=0.15, corner_radius=0.08)

            self.play(Write(inv_cond), run_time=tracker.duration * 0.2)
            self.play(Write(inv_result[:2]), run_time=tracker.duration * 0.2)
            self.play(FadeIn(inv_result[2]), Create(one_param_box), run_time=tracker.duration * 0.2)
            self.wait(max(0, tracker.duration * 0.2))

        # ── Part 2: Quadratic case ──
        self.play(
            FadeOut(gen_eq), FadeOut(inv_cond),
            FadeOut(inv_result), FadeOut(one_param_box), FadeOut(subtitle),
            run_time=0.3
        )

        with self.voiceover(
            text="For quadratic polynomials, permutation invariance gives exactly two free parameters for any d: w 1 times the sum of x i squared, plus w 2 times the sum of x i x j for i not equal to j. Two parameters, independent of dimension!"
        ) as tracker:
            quad_title = Text("Quadratic invariant polynomials:", font_size=18, color=WHITE, weight=BOLD).next_to(title, DOWN, buff=0.4)
            quad_eq = MathTex(
                r"f(x) = w_1 \sum_i x_i^2 + w_2 \sum_{i \neq j} x_i x_j",
                font_size=32, color=WHITE
            ).shift(UP * 0.3)
            quad_params = MathTex(r"\Longrightarrow\; 2 \text{ parameters for any } d!", font_size=22, color=WHITE).next_to(quad_eq, DOWN, buff=0.3)

            # Show why: orbits of S_d on degree-2 monomials
            orbit_explain = VGroup(
                Text("Orbits of Sₐ on degree-2 monomials:", font_size=18, color=WHITE),
                MathTex(r"\{x_i^2\} \;\text{(diagonal)}", font_size=25, color=BLUE),
                MathTex(r"\{x_i x_j,\; i\neq j\} \;\text{(off-diagonal)}", font_size=25, color=ORANGE),
            ).arrange(DOWN, buff=0.1, aligned_edge=LEFT).shift(DOWN * 1.5)

            self.play(Write(quad_title), run_time=0.3)
            self.play(Write(quad_eq), run_time=tracker.duration * 0.25)
            self.play(FadeIn(quad_params), run_time=tracker.duration * 0.15)
            self.play(FadeIn(orbit_explain), run_time=tracker.duration * 0.2)
            self.wait(max(0, tracker.duration * 0.2))

        # ── Part 3: General degree k ──
        self.play(FadeOut(quad_title), FadeOut(quad_eq), FadeOut(quad_params), FadeOut(orbit_explain), run_time=0.3)
        with self.voiceover(
            text="In general, for degree k permutation-invariant polynomials, the number of parameters equals the number of integer partitions of k, which is approximately e to the power C root k. This grows with k but is completely independent of d — this is representation stability."
        ) as tracker:
            gen_title = Text("General degree k:", font_size=18, color=WHITE, weight=BOLD).next_to(title, DOWN, buff=0.4)

            # Vị trí x cố định cho 2 cột
            col1_x = -1.0  # cột Degree k
            col2_x = 1.5   # cột Params

            # Header
            header_deg = MathTex(r"\text{Degree } k", font_size=20, color=WHITE).move_to([col1_x, 0.8, 0])
            header_par = MathTex(r"\text{Params}", font_size=20, color=WHITE).move_to([col2_x, 0.8, 0])
            table_header = VGroup(header_deg, header_par)

            # Rows — đặt từng ô vào đúng cột x
            data = [("1", "1"), ("2", "2"), ("3", "3"), ("4", "5"), ("5", "7"), ("k", "p(k)")]
            table_rows = VGroup()
            start_y = 0.4
            row_spacing = 0.35

            for i, (deg, params) in enumerate(data):
                y = start_y - i * row_spacing
                deg_cell = MathTex(deg, font_size=20, color=YELLOW).move_to([col1_x, y, 0])
                par_cell = MathTex(params, font_size=20, color=GREEN).move_to([col2_x, y, 0])
                table_rows.add(VGroup(deg_cell, par_cell))

            # Asymptotic formula
            asymp = MathTex(
                r"p(k) \approx \exp\!\left(C\sqrt{k}\right)",
                font_size=28, color=YELLOW
            ).shift(DOWN * 2.0)
            asymp_note = Text("Independent of d!", font_size=18, color=WHITE, weight=BOLD).next_to(asymp, DOWN, buff=0.15)

            self.play(Write(gen_title), run_time=0.3)
            self.play(FadeIn(table_header), run_time=0.3)
            self.play(LaggedStart(*[FadeIn(r) for r in table_rows], lag_ratio=0.15), run_time=tracker.duration * 0.3)
            self.play(Write(asymp), FadeIn(asymp_note), run_time=tracker.duration * 0.2)
            self.wait(max(0, tracker.duration * 0.2))


        # ── Part 4: Key insight — train low-dim, evaluate high-dim ──
        self.play(
            FadeOut(gen_title), FadeOut(table_header), FadeOut(table_rows),
            FadeOut(asymp), FadeOut(asymp_note),
            run_time=0.3
        )

        with self.voiceover(
            text="The key insight: learn these few parameters from low-dimensional training data, then apply the exact same model to high-dimensional test data. The mathematical foundation comes from representation stability, a phenomenon in algebraic topology discovered by Church and Farb."
        ) as tracker:
            # Flow: train d=5 → same w → evaluate d=100
            flow = VGroup(
                VGroup(
                    RoundedRectangle(width=2.5, height=1.0, color=BLUE, fill_opacity=0.1, corner_radius=0.1, stroke_width=1.5),
                    Tex(r"Train on \\ $d = 5$", font_size=29, color=BLUE),
                ),
                Arrow(ORIGIN, RIGHT * 1.2, color=YELLOW, stroke_width=2.5, tip_length=0.12),
                VGroup(
                    RoundedRectangle(width=2.2, height=1.0, color=YELLOW, fill_opacity=0.1, corner_radius=0.1, stroke_width=1.5),
                    Tex(r"Same \\ parameters $w$", font_size=29, color=YELLOW),
                ),
                Arrow(ORIGIN, RIGHT * 1.2, color=YELLOW, stroke_width=2.5, tip_length=0.12),
                VGroup(
                    RoundedRectangle(width=2.5, height=1.0, color=GREEN, fill_opacity=0.1, corner_radius=0.1, stroke_width=1.5),
                    Tex(r"Evaluate on \\ $d = 100$", font_size=29, color=GREEN),
                ),
            )
            # Position text inside boxes
            for item in [flow[0], flow[2], flow[4]]:
                item[1].move_to(item[0])
            flow.arrange(RIGHT, buff=0.2).shift(UP * 0.3)

            # Credit
            credit = VGroup(
                RoundedRectangle(width=7.5, height=0.7, color=TEAL, fill_opacity=0.08, corner_radius=0.1, stroke_width=1.5),
                Tex(r"Representation Stability (Church \& Farb, 2013--2014)", font_size=29, color=TEAL),
            )
            # Lời khuyên nhỏ: Đừng quên thêm lệnh move_to cho credit giống như bạn đã làm với các box ở trên
            credit[1].move_to(credit[0])
            credit[1].move_to(credit[0])
            credit.shift(DOWN * 1.5)

            self.play(
                LaggedStart(*[FadeIn(f) for f in flow], lag_ratio=0.2),
                run_time=tracker.duration * 0.4
            )
            self.play(FadeIn(credit, scale=1.05), run_time=tracker.duration * 0.2)
            self.wait(max(0, tracker.duration * 0.3))

        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.5)
        self.wait(0.3)


# ═══════════════════════════════════════════════════════════════
# SCENE 27: Recent Advances (Slide 60)
# ═══════════════════════════════════════════════════════════════
class Scene27RecentAdvances(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title = Text("Recent Advances", font_size=36, color=WHITE, weight=BOLD)
        title.to_edge(UP, buff=0.4)
        self.play(Write(title), run_time=0.4)

        # ── Part 1: Timeline of contributions ──
        with self.voiceover(
            text="Recent work builds on representation stability systematically. Early graph invariant networks already used size-independent parameters. Levin and Diaz in 2024 proposed a general framework for any-dimensional equivariant neural networks, combining representation stability with standard architectures. Learning theory for invariant finite-dimensional kernels avoids group averaging and learns features that generalize over size."
        ) as tracker:
            timeline_items = VGroup(
                VGroup(
                    Text("Graph Invariant Networks", font_size=16, color=BLUE, weight=BOLD),
                    Text("(Maron et al., ICLR 2019)", font_size=13, color=WHITE),
                ).arrange(RIGHT, buff=0.35),
                VGroup(
                    Text("Any-dim Equivariant NNs", font_size=16, color=GREEN, weight=BOLD),
                    Text("(Levin & Diaz, AISTATS 2024)", font_size=13, color=WHITE),
                ).arrange(RIGHT, buff=0.35),
                VGroup(
                    Text("Learning Theory", font_size=16, color=ORANGE, weight=BOLD),
                    Text("— invariant finite-dim kernels, avoid group averaging", font_size=13, color=WHITE),
                ).arrange(RIGHT, buff=0.35),
                VGroup(
                    Text("Systematic Treatment", font_size=16, color=PURPLE, weight=BOLD),
                    Text("(Levin & Diaz, 2024; Diaz et al., 2025)", font_size=13, color=WHITE),
                ).arrange(RIGHT, buff=0.35),
            ).arrange(DOWN, aligned_edge=LEFT, buff=0.3).next_to(title, DOWN, buff=0.5)

            # Scale if too wide
            if timeline_items.width > 12:
                timeline_items.scale(12 / timeline_items.width)

            # Timeline dots on left
            dots_line = VGroup()
            for item in timeline_items:
                dot = Dot(item.get_left() + LEFT * 0.3, radius=0.06, color=YELLOW)
                dots_line.add(dot)

            connector = Line(
                dots_line[0].get_center(), dots_line[-1].get_center(),
                color=YELLOW, stroke_width=2
            )

            self.play(Create(connector), run_time=0.3)
            for i, (item, dot) in enumerate(zip(timeline_items, dots_line)):
                self.play(GrowFromCenter(dot), FadeIn(item, shift=RIGHT * 0.2), run_time=tracker.duration * 0.12)
            self.wait(max(0, tracker.duration * 0.2))

        # ── Part 2: Three key questions ──
        self.play(FadeOut(timeline_items), FadeOut(dots_line), FadeOut(connector), run_time=0.3)

        with self.voiceover(
            text="Three key open questions drive this field. First: how to identify which problems exhibit any-dimensional behavior? Second: how to parametrize models that work across dimensions? And third: how to properly evaluate models on objects of different sizes and dimensions?"
        ) as tracker:
            q_title = Text("Key Questions", font_size=18, color=WHITE, weight=BOLD).next_to(title, DOWN, buff=0.4)

            q1_box = RoundedRectangle(width=8, height=0.8, color=RED, fill_opacity=0.08, corner_radius=0.1, stroke_width=1.5)
            q1_text = Text("Q1: How to identify any-dimensional behavior?", font_size=17, color=RED)
            q1_text.move_to(q1_box).align_to(q1_box, LEFT).shift(RIGHT * 0.3)
            q2_box = RoundedRectangle(width=8, height=0.8, color=ORANGE, fill_opacity=0.08, corner_radius=0.1, stroke_width=1.5)
            q2_text = Text("Q2: How to parametrize the model?", font_size=17, color=ORANGE)
            q2_text.move_to(q2_box).align_to(q2_box, LEFT).shift(RIGHT * 0.3)
            q3_box = RoundedRectangle(width=8, height=0.8, color=TEAL, fill_opacity=0.08, corner_radius=0.1, stroke_width=1.5)
            q3_text = Text("Q3: How to evaluate the model?", font_size=17, color=TEAL)
            q3_text.move_to(q3_box).align_to(q3_box, LEFT).shift(RIGHT * 0.3)

            questions = VGroup(
                VGroup(q1_box, q1_text),
                VGroup(q2_box, q2_text),
                VGroup(q3_box, q3_text),
            ).arrange(DOWN, buff=0.25).next_to(q_title, DOWN, buff=0.3)

            self.play(Write(q_title), run_time=0.3)
            self.play(
                LaggedStart(*[FadeIn(q, shift=LEFT * 0.3) for q in questions], lag_ratio=0.3),
                run_time=tracker.duration * 0.5
            )
            self.wait(max(0, tracker.duration * 0.3))

        # ── Part 3: Applications ──
        self.play(FadeOut(q_title), FadeOut(questions), run_time=0.3)

        with self.voiceover(
            text="Applications extend beyond graphs. Any-dimensional models can learn PDEs in physics, where the number of grid points varies. They apply to optimization over convex cones and graph properties like max-cut that are well-defined for any size. And in information theory, symmetry simplifies finding the capacity of feedback channels."
        ) as tracker:
            app_title = Text("Applications", font_size=18, color=WHITE, weight=BOLD).next_to(title, DOWN, buff=0.4)

            apps = VGroup(
                VGroup(
                    Text("PDEs / Physics:", font_size=17, color=BLUE, weight=BOLD),
                    Text("variable grid sizes", font_size=15, color=WHITE),
                ).arrange(RIGHT, buff=0.2),
                VGroup(
                    Text("Optimization:", font_size=17, color=ORANGE, weight=BOLD),
                    Text("convex cones, max-cut, clique (any graph size)", font_size=15, color=WHITE),
                ).arrange(RIGHT, buff=0.2),
                VGroup(
                    Text("Information Theory:", font_size=17, color=PURPLE, weight=BOLD),
                    Text("channel capacity via group symmetry", font_size=15, color=WHITE),
                ).arrange(RIGHT, buff=0.2),
            ).arrange(DOWN, aligned_edge=LEFT, buff=0.3).next_to(app_title, DOWN, buff=0.3)

            if apps.width > 11:
                apps.scale(11 / apps.width)

            self.play(Write(app_title), run_time=0.3)
            self.play(
                LaggedStart(*[FadeIn(a, shift=RIGHT * 0.2) for a in apps], lag_ratio=0.3),
                run_time=tracker.duration * 0.5
            )
            self.wait(max(0, tracker.duration * 0.3))

        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.5)
        self.wait(0.3)


# ═══════════════════════════════════════════════════════════════
# SCENE 28: Open Questions + References (Slide 61)
# ═══════════════════════════════════════════════════════════════
class Scene28OpenQuestions(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title = Text("Open Questions", font_size=36, color=WHITE)
        title.to_edge(UP, buff=0.4)
        self.play(Write(title), run_time=0.4)

        # ── Open problems ──
        with self.voiceover(
            text="Several fundamental open problems remain. First: identify applications and build any-dimensional models for physics, PDEs, biology, material science, language, and sequence models. Second: do we lose expressivity? Universality is problem-specific — some functions cannot be represented with finite parameters across all dimensions. Third: how to evaluate models on objects of different sizes and dimensions — what are the right assumptions and metrics?"
        ) as tracker:

            # Nội dung từng box — dùng Tex, căn trái
            txt1 = Tex(
                r"1. Identify applications: physics, PDEs, biology, material science, language and sequence models",
                font_size=28, color=WHITE
            )
            txt2 = Tex(
                r"2. Expressivity: Do we lose universality? Answer is problem-specific!",
                font_size=28, color=WHITE
            )
            txt3 = Tex(
                r"3. Evaluation: How to compare models across sizes/dimensions? What assumptions are needed?",
                font_size=28, color=WHITE
            )

            # Tạo các box
            box_width = 11
            box_height = 0.9
            box1 = RoundedRectangle(width=box_width, height=box_height, color=RED, fill_opacity=0.06, corner_radius=0.1, stroke_width=1.5)
            box2 = RoundedRectangle(width=box_width, height=box_height, color=ORANGE, fill_opacity=0.06, corner_radius=0.1, stroke_width=1.5)
            box3 = RoundedRectangle(width=box_width, height=box_height, color=TEAL, fill_opacity=0.06, corner_radius=0.1, stroke_width=1.5)

            # Gộp text vào box, căn trái trong box
            item1 = VGroup(box1, txt1)
            item2 = VGroup(box2, txt2)
            item3 = VGroup(box3, txt3)

            for item in [item1, item2, item3]:
                item[1].move_to(item[0]).align_to(item[0], LEFT).shift(RIGHT * 0.3)

            problems = VGroup(item1, item2, item3).arrange(DOWN, buff=0.25)
            problems.next_to(title, DOWN, buff=0.5)

            self.play(
                LaggedStart(*[FadeIn(p, shift=UP * 0.2) for p in problems], lag_ratio=0.35),
                run_time=tracker.duration * 0.6
            )
            self.wait(max(0, tracker.duration * 0.3))

        # ── References ──
        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.4)

        with self.voiceover(
            text="Here are the key references for this section."
        ) as tracker:
            ref_title = Text("References", font_size=18, color=WHITE).to_edge(UP, buff=0.5)

            refs_data = [
                r"[1] B. Farb, ``Representation stability,'' ICM 2014",
                r"[2] T. Church, B. Farb, ``Representation theory and homological stability,'' Adv.\ Math.\ 2013",
                r"[3] H. Maron et al., ``Invariant and Equivariant Graph Networks,'' ICLR 2019",
                r"[4] E. Levin, M. Diaz, ``Any-dimensional equivariant neural networks,'' AISTATS 2024",
                r"[5] E. Levin, V. Chandrasekaran, ``Dimension-Free Descriptions of Convex Sets,'' arXiv 2025",
                r"[6] E. Levin, Y. Ma, M. Diaz, S. Villar, ``On Transferring Transferability,'' arXiv 2025",
                r"[7] M. Diaz et al., ``Invariant Kernels: Rank Stabilization,'' arXiv 2025",
                r"[8] E. Atay, E. Levin et al., ``Poset-Markov Channels,'' arXiv 2025",
            ]

            ref_items = VGroup(*[
                Tex(r, font_size=25, color=WHITE) for r in refs_data
            ]).arrange(DOWN, aligned_edge=LEFT, buff=0.2)
            ref_items.next_to(ref_title, DOWN, buff=0.35)

            ref_group = VGroup(ref_title, ref_items)
            # Scale to fit if needed
            if ref_group.height > 6.5:
                ref_group.scale(6.5 / ref_group.height)
            ref_group.center()

            self.play(FadeIn(ref_group, shift=UP * 0.2), run_time=1.0)
            self.wait(max(0, tracker.duration - 1.0))

        self.wait(5.0)
        self.play(FadeOut(ref_group), run_time=0.8)
        self.wait(0.5)


In [ ]:
import subprocess, sys
from pathlib import Path
from IPython.display import Video, display

# ═══════════════════════════════════════════════════════════
# RENDER scenes 111–115
# ═══════════════════════════════════════════════════════════
scene_file = "/content/slides_111_115.py"
scenes_to_render = [
    "Scene25AnyDimTitle",
    "Scene26RepresentationStability",
    "Scene27RecentAdvances",
    "Scene28OpenQuestions",
]

output_paths_111_115 = []

for scene_name in scenes_to_render:
    print(f"Rendering {scene_name}...")
    result = subprocess.run(
        [
            sys.executable, "-m", "manim", "render",
            "-qh",                    # 1080p, 60fps
            "--disable_caching",
            scene_file,
            scene_name,
        ],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"  Error:\n{result.stderr[-800:]}")
    else:
        found = list(Path("media").rglob(f"{scene_name}.mp4"))
        if found:
            output_paths_111_115.append(str(found[0]))
            print(f"  {found[0]}")
        else:
            print(f"  Rendered OK but file not found")

print(f"\nTotal: {len(output_paths_111_115)} videos")
for p in output_paths_111_115:
    print(f"  {p}")

# ═══════════════════════════════════════════════════════════
# CONCAT — thống nhất 60fps, CRF 20
# ═══════════════════════════════════════════════════════════
list_file = "/content/concat_list_111_115.txt"
with open(list_file, "w") as f:
    for p in output_paths_111_115:
        if Path(p).exists():
            f.write(f"file '{p}'\n")

output_file = "/content/slides_111_115_voiceover.mp4"

cmd = [
    "ffmpeg", "-y",
    "-f", "concat",
    "-safe", "0",
    "-i", list_file,
    "-c:v", "libx264",
    "-preset", "medium",
    "-crf", "20",
    "-r", "60",
    "-pix_fmt", "yuv420p",
    "-c:a", "aac",
    "-b:a", "192k",
    "-movflags", "+faststart",
    output_file
]

result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode == 0 and Path(output_file).exists():
    probe = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries",
         "format=duration,bit_rate", "-show_entries",
         "stream=r_frame_rate,width,height,bit_rate",
         "-of", "default=noprint_wrappers=1",
         output_file],
        capture_output=True, text=True
    )
    print(f"\nFinal: {output_file}")
    print(probe.stdout)
    display(Video(output_file, embed=True, width=854))
else:
    print(f"FFmpeg error:\n{result.stderr[-500:]}")


## Display

In [ ]:
import subprocess
from pathlib import Path
from IPython.display import Video, display
import re

# Gather all rendered scenes for slides 111-115
output_paths_111_115 = sorted(
    p for p in Path("media").rglob("Scene*.mp4")
    if re.search(r"Scene(\d+)", p.stem) and 25 <= int(re.search(r"Scene(\d+)", p.stem).group(1)) <= 28
)

list_file = "/content/concat_list_111_115.txt"
output_file = "/content/slides_111_115_voiceover.mp4"

with open(list_file, "w") as f:
    for p in output_paths_111_115:
        f.write(f"file '{p.absolute()}'\n")

cmd = [
    "ffmpeg", "-y",
    "-f", "concat",
    "-safe", "0",
    "-i", list_file,
    "-c:v", "libx264",
    "-preset", "medium",
    "-crf", "20",
    "-r", "60",
    "-pix_fmt", "yuv420p",
    "-c:a", "aac",
    "-b:a", "192k",
    "-movflags", "+faststart",
    output_file
]

result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode == 0 and Path(output_file).exists():
    probe = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries",
         "format=duration,bit_rate", "-show_entries",
         "stream=r_frame_rate,width,height,bit_rate",
         "-of", "default=noprint_wrappers=1",
         output_file],
        capture_output=True, text=True
    )
    print(f"Final: {output_file}")
    print(probe.stdout)
    display(Video(output_file, embed=True, width=854))
else:
    print("FFmpeg error:")
    print(result.stderr[-500:])


# SLIDE 116-122

## Generative

In [ ]:
# FILE: slides_116_122.py — Geometry Beyond Symmetries (Rewritten - High Quality)
# Slides 116–122 | Scenes 29–35
# FILE: slides_116_122.py — Geometry Beyond Symmetries (v3 - Clean & Spacious)
# Slides 116–122 | Scenes 29–35
%%writefile /content/slides_116_122.py
import nest_asyncio
nest_asyncio.apply()

import asyncio
import numpy as np
from pathlib import Path

from manim import *
from manim_voiceover import VoiceoverScene
from manim_voiceover.services.base import SpeechService

import edge_tts


# ─── EdgeTTS Service ─────────────────────────────────────────────────
class EdgeTTSService(SpeechService):
    def __init__(self, voice="en-US-GuyNeural", **kwargs):
        self.voice = voice
        super().__init__(**kwargs)

    def generate_from_text(self, text: str, cache_dir: str = None, path: str = None, **kwargs) -> dict:
        if cache_dir is None:
            cache_dir = self.cache_dir
        input_data = {"input_text": text, "service": "edge_tts", "voice": self.voice}
        cached = self.get_cached_result(input_data, Path(cache_dir))
        if cached is not None:
            return cached

        audio_basename = self.get_audio_basename(input_data)
        if path is None:
            audio_path = audio_basename + ".mp3"
        else:
            audio_path = path
        output_file = str(Path(cache_dir) / audio_path)

        async def _generate():
            communicate = edge_tts.Communicate(text, self.voice)
            await communicate.save(output_file)

        asyncio.run(_generate())

        json_dict = {
            "input_text": text,
            "input_data": input_data,
            "original_audio": audio_path,
        }
        return json_dict


VOICE = "en-US-GuyNeural"


# ═══════════════════════════════════════════════════════════════════════
# SCENE 29: Title — Geometry Beyond Symmetries (Slide 116)
# ═══════════════════════════════════════════════════════════════════════
class Scene29GeometryBeyondTitle(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))
        self.camera.background_color = "BLACK"

        with self.voiceover(
            text=(
                "So far, we have focused on group symmetries as a source of "
                "geometric structure in machine learning. "
                "But geometry offers much more. In this final section, we explore "
                "curvature, topology, and scaling — geometry beyond symmetries."
            )
        ) as tracker:
            title = Text("Geometry Beyond Symmetries",
                         font_size=45, color=WHITE, weight=BOLD)
            title.move_to(UP * 0.5)
            self.play(Write(title), run_time=2.0)
            self.wait(0.5)

            # Three keywords fade in below, well-spaced
            kw1 = Text("Curvature", font_size=30, color="#4FC3F7")
            kw2 = Text("Topology", font_size=30, color="#81C784")
            kw3 = Text("Scaling", font_size=30, color="#FFD54F")
            keywords = VGroup(kw1, kw2, kw3).arrange(RIGHT, buff=2.0)
            keywords.move_to(DOWN * 1.5)

            self.play(
                LaggedStart(
                    FadeIn(kw1, shift=UP * 0.3),
                    FadeIn(kw2, shift=UP * 0.3),
                    FadeIn(kw3, shift=UP * 0.3),
                    lag_ratio=0.4
                ),
                run_time=2.0
            )
            self.wait(1.0)

        self.play(FadeOut(title), FadeOut(keywords), run_time=0.8)
        self.wait(0.3)


# ═══════════════════════════════════════════════════════════════════════
# SCENE 30: Flexibility — Symmetry Breaking (Slide 117)
# ═══════════════════════════════════════════════════════════════════════
class Scene30Flexibility(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))
        self.camera.background_color = "BLACK"

        header = Text("Should We Always Encode Symmetries?",
                      font_size=36, color=WHITE, weight=BOLD).to_edge(UP, buff=0.6)
        self.play(FadeIn(header, shift=DOWN * 0.2), run_time=0.8)

        # --- Part 1: Symmetry Breaking in Diffusion ---
        with self.voiceover(
            text=(
                "Sometimes, hard-coding symmetry is too restrictive. "
                "In diffusion models, the noise distribution is perfectly symmetric. "
                "But to generate a specific image, the model must break that symmetry."
            )
        ) as tracker:
            # Symmetric noise: evenly spaced dots on a circle
            circle = Circle(radius=1.5, color="#4FC3F7", stroke_width=2)
            circle.move_to(LEFT * 3 + DOWN * 0.8)
            dots = VGroup(*[
                Dot(circle.point_from_proportion(i / 8), radius=0.07, color="#4FC3F7")
                for i in range(8)
            ])
            sym_label = Text("Symmetric noise", font_size=22, color="#4FC3F7")
            sym_label.next_to(circle, DOWN, buff=0.4)

            self.play(Create(circle), FadeIn(dots), Write(sym_label), run_time=1.5)

            # Arrow
            arrow = Arrow(LEFT * 1.0 + DOWN * 0.8, RIGHT * 1.0 + DOWN * 0.8,
                          color="#FFD54F", stroke_width=3, tip_length=0.2)
            arrow_label = Text("denoise", font_size=20, color="#FFD54F")
            arrow_label.next_to(arrow, UP, buff=0.15)
            self.play(GrowArrow(arrow), FadeIn(arrow_label), run_time=0.8)

            # Broken symmetry: one dot
            single_dot = Dot(RIGHT * 3 + DOWN * 0.8, radius=0.15, color="#EF5350")
            broken_label = Text("Specific output", font_size=22, color="#EF5350")
            broken_label.next_to(single_dot, DOWN, buff=0.4)
            self.play(FadeIn(single_dot, scale=0.3), Write(broken_label), run_time=1.0)
            self.wait(0.5)

        # --- Part 2: Spectrum ---
        with self.voiceover(
            text=(
                "This motivates a spectrum. On one end, strict equivariant models "
                "that are data-efficient but rigid. On the other, foundation models "
                "that learn from scale. The promising direction: "
                "flexible symmetries that adapt to the task. "
                "This was formalized by Raya and Ambrogioni at NeurIPS 2023."
            )
        ) as tracker:
            # Clear part 1
            part1 = VGroup(circle, dots, sym_label, arrow, arrow_label, single_dot, broken_label)
            self.play(FadeOut(part1), run_time=0.5)

            # Simple spectrum
            bar = RoundedRectangle(width=9, height=0.3, corner_radius=0.15,
                                   fill_opacity=1, stroke_width=0)
            bar.set_color_by_gradient("#4FC3F7", "#FFD54F", "#81C784")
            bar.move_to(DOWN * 0.3)

            left_l = Text("Strict Equivariance", font_size=20, color="#4FC3F7")
            left_l.next_to(bar, DOWN, buff=0.3).align_to(bar, LEFT)
            right_l = Text("Foundation Models", font_size=20, color="#81C784")
            right_l.next_to(bar, DOWN, buff=0.3).align_to(bar, RIGHT)

            self.play(FadeIn(bar), FadeIn(left_l), FadeIn(right_l), run_time=1.0)

            # Marker in middle
            marker = Triangle(fill_opacity=1, fill_color="#FFD54F",
                              stroke_width=0).scale(0.12).rotate(PI)
            marker.next_to(bar, UP, buff=0.05)
            marker_l = Text("Adaptive Symmetry", font_size=22,
                            color="#FFD54F", weight=BOLD)
            marker_l.next_to(marker, UP, buff=0.15)

            self.play(FadeIn(marker, shift=DOWN * 0.1), Write(marker_l), run_time=1.0)

            ref = Text("(Raya & Ambrogioni, NeurIPS 2023)", font_size=16, color=WHITE)
            ref.to_edge(DOWN, buff=0.5)
            self.play(FadeIn(ref), run_time=0.5)
            self.wait(1.0)

        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.8)
        self.wait(0.3)


# ═══════════════════════════════════════════════════════════════════════
# SCENE 31: Efficiency & Scaling Laws (Slide 118)
# ═══════════════════════════════════════════════════════════════════════
class Scene31Efficiency(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))
        self.camera.background_color = "BLACK"

        header = Text("Does Equivariance Matter at Scale?",
                      font_size=36, color=WHITE, weight=BOLD).to_edge(UP, buff=0.6)
        self.play(FadeIn(header, shift=DOWN * 0.2), run_time=0.8)

        with self.voiceover(
            text=(
                "Brehmer and colleagues asked: if we scale up compute and data, "
                "does equivariance still help? "
                "Their answer, published in Transactions on Machine Learning Research 2025, is yes."
            )
        ) as tracker:
            subtitle = Text("Brehmer et al., TMLR 2025", font_size=18, color=WHITE)
            subtitle.next_to(header, DOWN, buff=0.3)
            self.play(FadeIn(subtitle), run_time=0.5)
            self.wait(max(0, tracker.duration - 1.5))

        with self.voiceover(
            text=(
                "They found that both equivariant and non-equivariant models "
                "follow power-law scaling. But the equivariant curve sits "
                "consistently below, meaning lower loss at every compute budget. "
                "The gap never closes."
            )
        ) as tracker:
            axes = Axes(
                x_range=[0, 5, 1], y_range=[0, 3.5, 1],
                x_length=7, y_length=4,
                axis_config={"include_tip": True, "tip_length": 0.15,
                            "stroke_width": 2, "color": GREY_B,
                            "include_numbers": False},
            ).move_to(DOWN * 0.3)

            x_label = Text("log(Compute)", font_size=20, color=WHITE)
            x_label.next_to(axes.x_axis, DOWN, buff=0.3)
            y_label = Text("log(Loss)", font_size=20, color=WHITE)
            y_label.next_to(axes.y_axis, LEFT, buff=0.3)

            self.play(Create(axes), FadeIn(x_label), FadeIn(y_label), run_time=1.0)

            # Non-equivariant (higher) — giảm slope để không chạm trục x
            line_ne = axes.plot(lambda x: 3.2 - 0.4 * x, x_range=[0.3, 4.5],
                                color="#B0BEC5", stroke_width=3)
            label_ne = Text("Non-equivariant", font_size=18, color="#B0BEC5")
            label_ne.next_to(line_ne.get_end(), RIGHT, buff=0.15)

            # Equivariant (lower) — giữ khoảng cách đều, không chạm trục x
            line_eq = axes.plot(lambda x: 2.4 - 0.4 * x, x_range=[0.3, 4.5],
                                color="#4FC3F7", stroke_width=3)
            label_eq = Text("Equivariant", font_size=18, color="#4FC3F7")
            label_eq.next_to(line_eq.get_end(), RIGHT, buff=0.15)

            self.play(Create(line_ne), Write(label_ne), run_time=1.2)
            self.play(Create(line_eq), Write(label_eq), run_time=1.2)

            # Gap arrow — đặt ở giữa đồ thị
            gx = 2.5
            y_top = 3.2 - 0.4 * gx
            y_bot = 2.4 - 0.4 * gx
            gap_a = DoubleArrow(
                axes.c2p(gx, y_top),
                axes.c2p(gx, y_bot),
                color="#FFD54F", stroke_width=2, tip_length=0.1, buff=0.05
            )
            gap_text = Text("persistent gap", font_size=16, color="#FFD54F")
            gap_text.next_to(gap_a, RIGHT, buff=0.15)
            self.play(GrowFromCenter(gap_a), FadeIn(gap_text), run_time=0.8)
            self.wait(0.5)

        with self.voiceover(
            text=(
                "Moreover, equivariant models benefit most from scaling model size, "
                "while non-equivariant models benefit from training longer. "
                "Equivariance provides a persistent structural advantage."
            )
        ) as tracker:
            # Takeaway text at bottom
            takeaway = Text(
                "Equivariance helps at every scale — the gap persists.",
                font_size=22, color=WHITE
            ).to_edge(DOWN, buff=0.5)
            self.play(FadeIn(takeaway, shift=UP * 0.2), run_time=0.8)
            self.wait(1.5)

        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.8)
        self.wait(0.3)


# ═══════════════════════════════════════════════════════════════════════
# SCENE 32: What Kind of Symmetries? (Slide 119)
# ═══════════════════════════════════════════════════════════════════════
class Scene32WhatKind(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))
        self.camera.background_color = "BLACK"

        header = Text("Diagnosing Symmetry in Data",
                      font_size=36, color=WHITE, weight=BOLD).to_edge(UP, buff=0.6)
        self.play(FadeIn(header, shift=DOWN * 0.2), run_time=0.8)

        # --- Part 1: The test ---
        with self.voiceover(
            text=(
                "Before encoding symmetries, we should check: "
                "is the data distribution actually symmetric? "
                "Lawrence, Hofgard, and colleagues proposed a simple diagnostic at ICLR 2026. "
                "Take your dataset. Augment it with a random symmetry transformation. "
                "Then train a classifier to tell original from augmented. "
                "If it succeeds, your distribution breaks that symmetry."
            )
        ) as tracker:
            # Visual: D vs g·D
            box_d = RoundedRectangle(width=2.5, height=2.0, corner_radius=0.15,
                                     color="#4FC3F7", fill_opacity=0.08, stroke_width=2)
            box_d.move_to(LEFT * 3 + DOWN * 0.7)
            label_d = MathTex(r"\mathcal{D}", font_size=40, color="#4FC3F7")
            label_d.move_to(box_d)
            text_d = Text("Original", font_size=20, color="#4FC3F7")
            text_d.next_to(box_d, DOWN, buff=0.3)

            box_gd = RoundedRectangle(width=2.5, height=2.0, corner_radius=0.15,
                                      color="#81C784", fill_opacity=0.08, stroke_width=2)
            box_gd.move_to(RIGHT * 3 + DOWN * 0.7)
            label_gd = MathTex(r"g \cdot \mathcal{D}", font_size=40, color="#81C784")
            label_gd.move_to(box_gd)
            text_gd = Text("Augmented", font_size=20, color="#81C784")
            text_gd.next_to(box_gd, DOWN, buff=0.3)

            # Classifier in the middle
            clf_box = RoundedRectangle(width=2.0, height=1.0, corner_radius=0.1,
                                       color="#FFD54F", fill_opacity=0.15, stroke_width=2)
            clf_box.move_to(DOWN * 0.7)
            clf_text = Text("Classifier", font_size=20, color="#FFD54F")
            clf_text.move_to(clf_box)

            arr_l = Arrow(box_d.get_right(), clf_box.get_left(),
                          color=WHITE, stroke_width=2, tip_length=0.15, buff=0.1)
            arr_r = Arrow(box_gd.get_left(), clf_box.get_right(),
                          color=WHITE, stroke_width=2, tip_length=0.15, buff=0.1)

            self.play(
                Create(box_d), Write(label_d), FadeIn(text_d),
                Create(box_gd), Write(label_gd), FadeIn(text_gd),
                run_time=1.5
            )
            self.play(
                Create(clf_box), Write(clf_text),
                GrowArrow(arr_l), GrowArrow(arr_r),
                run_time=1.0
            )

            result_text = Text("Can distinguish? → Symmetry is broken!",
                               font_size=22, color="#EF5350")
            result_text.move_to(DOWN * 2.5)
            self.play(FadeIn(result_text, shift=UP * 0.2), run_time=0.8)
            self.wait(0.5)

        # --- Part 2: Results ---
        with self.voiceover(
            text=(
                "They tested standard benchmarks. QM9 shows high symmetry breaking. "
                "OC20 was even canonicalized, hiding the problem. "
                "The theoretical implication: when the distribution breaks symmetry, "
                "forcing invariance can hurt performance."
            )
        ) as tracker:
            part1 = VGroup(box_d, label_d, text_d, box_gd, label_gd, text_gd,
                           clf_box, clf_text, arr_l, arr_r, result_text)
            self.play(FadeOut(part1), run_time=0.5)

            # Clean results
            r1 = Text("QM9      →  high symmetry-breaking", font_size=24, color="#EF5350")
            r2 = Text("rMD17    →  moderate", font_size=24, color="#FFD54F")
            r3 = Text("OC20     →  canonicalized (hidden!)", font_size=24, color="#EF5350")
            results = VGroup(r1, r2, r3).arrange(DOWN, buff=0.5, aligned_edge=LEFT)
            results.move_to(DOWN * 0.3)

            self.play(
                LaggedStart(
                    FadeIn(r1, shift=RIGHT * 0.3),
                    FadeIn(r2, shift=RIGHT * 0.3),
                    FadeIn(r3, shift=RIGHT * 0.3),
                    lag_ratio=0.4
                ), run_time=2.0
            )

            ref = Text("(Lawrence, Hofgard et al., ICLR 2026)", font_size=16, color=WHITE)
            ref.to_edge(DOWN, buff=0.5)
            self.play(FadeIn(ref), run_time=0.5)
            self.wait(1.5)

        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.8)
        self.wait(0.3)


# ═══════════════════════════════════════════════════════════════════════
# SCENE 33: Curvature and Learnability (Slide 120)
# ═══════════════════════════════════════════════════════════════════════
class Scene33Curvature(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))
        self.camera.background_color = "BLACK"

        header = Text("Curvature and Learnability",
                      font_size=36, color=WHITE, weight=BOLD).to_edge(UP, buff=0.6)
        self.play(FadeIn(header, shift=DOWN * 0.2), run_time=0.8)

        # --- Part 1: Manifold hypothesis ---
        with self.voiceover(
            text=(
                "The manifold hypothesis says real data lives on a "
                "low-dimensional manifold inside high-dimensional space. "
                "If true, we might hope to escape the curse of dimensionality. "
                "But is low intrinsic dimension enough to guarantee learnability?"
            )
        ) as tracker:
            # R^D box background to give context
            rd_box = RoundedRectangle(width=8, height=4.5, corner_radius=0.2,
                                      color=GREY, fill_opacity=0.03, stroke_width=1)
            rd_box.move_to(DOWN * 0.3)
            rd_label = MathTex(r"\mathbb{R}^D", font_size=30, color=GREY_B)
            rd_label.next_to(rd_box, UR, buff=-0.5)

            # Manifold as a curvy line INSIDE the box
            manifold = ParametricFunction(
                lambda t: np.array([3.0 * t - 3.0, 0.8 * np.sin(2 * t) - 0.3, 0]),
                t_range=[0, 2.0], color="#4FC3F7", stroke_width=4
            )

            # Scatter some dots ON the manifold to show "data lives here"
            data_dots = VGroup(*[
                Dot(manifold.point_from_proportion(p), radius=0.06, color=YELLOW)
                for p in np.linspace(0.1, 0.9, 7)
            ])

            md_label = MathTex(r"M^d,\; d \ll D", font_size=26, color="#4FC3F7")
            md_label.next_to(manifold, DOWN, buff=0.4)

            self.play(Create(rd_box), FadeIn(rd_label), run_time=0.8)
            self.play(Create(manifold), run_time=1.0)
            self.play(FadeIn(data_dots, lag_ratio=0.1), Write(md_label), run_time=1.0)
            self.wait(0.5)

        # --- Part 2: Hardness ---
        with self.voiceover(
            text=(
                "Kiani, Wang, and Weber proved at NeurIPS 2024 that the answer is no. "
                "Bounded curvature alone does not make learning easy. "
                "It remains hard in the statistical query model."
            )
        ) as tracker:
            # Cross over the manifold label
            cross = Cross(md_label, color="#EF5350", stroke_width=4)

            hard_text = Text("Low dimension alone ≠ learnability",
                            font_size=24, color="#EF5350")
            hard_text.next_to(rd_box, DOWN, buff=0.4)

            self.play(Create(cross), run_time=0.8)
            self.play(Write(hard_text), run_time=1.0)
            self.wait(0.5)

        # --- Part 3: Three regimes ---
        with self.voiceover(
            text=(
                "They identified three regimes. "
                "First, manifolds with positive Ricci curvature and bounded volume, "
                "like spheres — these are efficiently learnable. "
                "Second, manifolds with bounded curvature but unbounded volume — "
                "these are provably hard. "
                "Third, heterogeneous manifolds with mixed properties — "
                "this is real-world data, and remains an open problem."
            )
        ) as tracker:
            # Clear previous
            prev = VGroup(rd_box, rd_label, manifold, data_dots, md_label, cross, hard_text)
            self.play(FadeOut(prev), run_time=0.5)

            # Three cards — fixed positions for alignment
            card_w, card_h = 3.2, 2.8
            card_y = DOWN * 0.3
            card_positions = [LEFT * 4 + card_y, ORIGIN + card_y, RIGHT * 4 + card_y]

            # Fixed y-positions for elements inside cards (relative to card center)
            title_y_offset = 0.95    # from card center
            icon_y_offset = 0.2      # from card center
            desc_y_offset = -0.85    # from card center

            # ── Card 1: Learnable ──
            c1_rect = RoundedRectangle(width=card_w, height=card_h, corner_radius=0.12,
                                      color="#81C784", fill_opacity=0.08, stroke_width=2)
            c1_rect.move_to(card_positions[0])
            c1_title = Text("Learnable", font_size=22, color="#81C784", weight=BOLD)
            c1_title.move_to(card_positions[0] + UP * title_y_offset)
            c1_icon = Circle(radius=0.35, color="#81C784", stroke_width=3)
            c1_icon.move_to(card_positions[0] + UP * icon_y_offset)
            c1_desc = Tex(r"$+$Ricci, bounded vol", font_size=22, color=WHITE)
            c1_desc.move_to(card_positions[0] + UP * desc_y_offset)

            # ── Card 2: Open Problem ──
            c2_rect = RoundedRectangle(width=card_w, height=card_h, corner_radius=0.12,
                                      color="#FFD54F", fill_opacity=0.08, stroke_width=2)
            c2_rect.move_to(card_positions[1])
            c2_title = Text("Open Problem", font_size=22, color="#FFD54F", weight=BOLD)
            c2_title.move_to(card_positions[1] + UP * title_y_offset)
            c2_icon = Text("?", font_size=40, color="#FFD54F", weight=BOLD)
            c2_icon.move_to(card_positions[1] + UP * icon_y_offset)
            c2_desc = Tex(r"heterogeneous dim", font_size=22, color=WHITE)
            c2_desc.move_to(card_positions[1] + UP * desc_y_offset)

            # ── Card 3: Provably Hard ──
            c3_rect = RoundedRectangle(width=card_w, height=card_h, corner_radius=0.12,
                                      color="#EF5350", fill_opacity=0.08, stroke_width=2)
            c3_rect.move_to(card_positions[2])
            c3_title = Text("Provably Hard", font_size=22, color="#EF5350", weight=BOLD)
            c3_title.move_to(card_positions[2] + UP * title_y_offset)
            c3_icon = VGroup(*[
                Line(ORIGIN, 0.35 * np.array([np.cos(a), np.sin(a), 0]),
                    color="#EF5350", stroke_width=2)
                for a in np.linspace(0, TAU, 6, endpoint=False)
            ])
            c3_icon.move_to(card_positions[2] + UP * icon_y_offset)
            c3_desc = Tex(r"bounded curv, $\infty$ vol", font_size=22, color=WHITE)
            c3_desc.move_to(card_positions[2] + UP * desc_y_offset)

            self.play(
                LaggedStart(
                    AnimationGroup(Create(c1_rect), Write(c1_title), Create(c1_icon), FadeIn(c1_desc)),
                    AnimationGroup(Create(c2_rect), Write(c2_title), Write(c2_icon), FadeIn(c2_desc)),
                    AnimationGroup(Create(c3_rect), Write(c3_title), Create(c3_icon), FadeIn(c3_desc)),
                    lag_ratio=0.4
                ),
                run_time=3.0
            )

            ref = Tex(r"(Kiani, Wang \& Weber, NeurIPS 2024)",
                      font_size=20, color=WHITE).to_edge(DOWN, buff=0.4)
            self.play(FadeIn(ref), run_time=0.5)
            self.wait(1.5)

        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.8)
        self.wait(0.3)


# ═══════════════════════════════════════════════════════════════════════
# SCENE 34: Topological Deep Learning — HOMP & Blindspots (Slide 121)
# ═══════════════════════════════════════════════════════════════════════
class Scene34TopologyI(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))
        self.camera.background_color = "BLACK"

        header = Text("Topological Deep Learning",
                      font_size=36, color=WHITE, weight=BOLD).to_edge(UP, buff=0.6)
        self.play(FadeIn(header, shift=DOWN * 0.2), run_time=0.8)

        # --- Part 1: What is TDL ---
        with self.voiceover(
            text=(
                "Topology gives us another lens on data geometry. "
                "Instead of just nodes and edges, topological deep learning "
                "uses simplicial complexes with higher-order cells: "
                "triangles, tetrahedra, and beyond. "
                "The standard framework is Higher-Order Message Passing, or HOMP."
            )
        ) as tracker:
            # Build simplicial complex step by step
            # Use well-spaced points
            pts = [
                np.array([-1.5, -1.0, 0]),
                np.array([0.0, 1.0, 0]),
                np.array([1.5, -1.0, 0]),
                np.array([3.0, 0.5, 0]),
            ]
            # Shift all down
            pts = [p + DOWN * 0.5 for p in pts]

            # Step 1: Nodes
            nodes = VGroup(*[Dot(p, radius=0.12, color=WHITE) for p in pts])
            n_label = Text("Nodes (0-cells)", font_size=20, color=WHITE)
            n_label.to_edge(DOWN, buff=0.5)
            self.play(FadeIn(nodes), Write(n_label), run_time=0.8)
            self.wait(0.3)

            # Step 2: Edges
            edge_pairs = [(0,1), (1,2), (0,2), (2,3), (1,3)]
            edges = VGroup(*[
                Line(pts[i], pts[j], color="#4FC3F7", stroke_width=3)
                for i, j in edge_pairs
            ])
            self.play(
                Transform(n_label, Text("+ Edges (1-cells)", font_size=20, color="#4FC3F7").to_edge(DOWN, buff=0.5)),
                Create(edges, lag_ratio=0.1),
                run_time=1.0
            )
            self.wait(0.3)

            # Step 3: Faces
            face1 = Polygon(*pts[:3], color="#81C784", fill_opacity=0.25, stroke_width=0)
            face2 = Polygon(pts[1], pts[2], pts[3], color="#81C784", fill_opacity=0.25, stroke_width=0)
            self.play(
                Transform(n_label, Text("+ Faces (2-cells)", font_size=20, color="#81C784").to_edge(DOWN, buff=0.5)),
                FadeIn(face1), FadeIn(face2),
                run_time=1.0
            )
            self.wait(0.5)

        # --- Part 2: Blindspots ---
        with self.voiceover(
            text=(
                "But HOMP has fundamental blindspots. "
                "Eitan, Gelberg, and colleagues proved at ICLR 2025 "
                "that HOMP cannot detect important topological properties: "
                "diameter, orientability, planarity, and homology. "
                "These are inherent limitations, not implementation bugs."
            )
        ) as tracker:
            # Clear complex
            complex_objs = VGroup(nodes, edges, face1, face2, n_label)
            self.play(FadeOut(complex_objs), run_time=0.5)

            # Two columns — very clean
            can_title = Text("CAN detect", font_size=24, color="#81C784", weight=BOLD)
            can_title.move_to(LEFT * 3 + UP * 0.8)
            can_items = VGroup(
                Text("• Components", font_size=22, color=WHITE),
                Text("• Loops", font_size=22, color=WHITE),
            ).arrange(DOWN, buff=0.3, aligned_edge=LEFT)
            can_items.next_to(can_title, DOWN, buff=0.4)

            cannot_title = Text("CANNOT detect", font_size=24, color="#EF5350", weight=BOLD)
            cannot_title.move_to(RIGHT * 3 + UP * 0.8)
            cannot_items = VGroup(
                Text("• Diameter", font_size=22, color=WHITE),
                Text("• Orientability", font_size=22, color=WHITE),
                Text("• Planarity", font_size=22, color=WHITE),
                Text("• Homology", font_size=22, color=WHITE),
            ).arrange(DOWN, buff=0.3, aligned_edge=LEFT)
            cannot_items.next_to(cannot_title, DOWN, buff=0.4)

            # Divider
            divider = DashedLine(UP * 1.5, DOWN * 2.5, color=WHITE, stroke_width=1)

            self.play(
                Write(can_title), Write(cannot_title), Create(divider),
                run_time=0.8
            )
            self.play(
                LaggedStart(*[FadeIn(item, shift=RIGHT*0.2) for item in can_items], lag_ratio=0.3),
                LaggedStart(*[FadeIn(item, shift=LEFT*0.2) for item in cannot_items], lag_ratio=0.3),
                run_time=2.0
            )

            ref = Text("(Eitan, Gelberg et al., ICLR 2025)",
                       font_size=16, color=WHITE).to_edge(DOWN, buff=0.4)
            self.play(FadeIn(ref), run_time=0.5)
            self.wait(1.0)

        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.8)
        self.wait(0.3)


# ═══════════════════════════════════════════════════════════════════════
# SCENE 35: MCN/SMCN & Torus vs Klein Bottle (Slide 122)
# ═══════════════════════════════════════════════════════════════════════
class Scene35TopologyII(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))
        self.camera.background_color = "BLACK"

        header = Text("Solutions: MCN and SMCN",
                      font_size=36, color=WHITE, weight=BOLD).to_edge(UP, buff=0.6)
        self.play(FadeIn(header, shift=DOWN * 0.2), run_time=0.8)

        # --- Part 1: MCN vs SMCN ---
        with self.voiceover(
            text=(
                "To fix these blindspots, Multi-Cellular Networks, or MCN, "
                "achieve full expressivity using ideas from higher-order "
                "Weisfeiler-Leman tests. But they are computationally expensive. "
                "The scalable variant, SMCN, trades a little expressivity "
                "for practical efficiency, and outperforms both HOMP and standard GNNs."
            )
        ) as tracker:
            # Three cards in a row: HOMP -> MCN -> SMCN
            card_data = [
                ("HOMP", "Limited", GREY_B, "#B0BEC5"),
                ("MCN", "Full expressivity\nHigh cost", "#81C784", "#81C784"),
                ("SMCN", "Scalable\nExpressive", "#FFD54F", "#FFD54F"),
            ]
            cards = VGroup()
            for name, desc, border_color, text_color in card_data:
                rect = RoundedRectangle(width=4.0, height=1.8, corner_radius=0.1,
                                        color=border_color, fill_opacity=0.05, stroke_width=2)
                title = Text(name, font_size=24, color=text_color, weight=BOLD)
                title.next_to(rect.get_top(), DOWN, buff=0.2)
                description = Tex(desc, font_size=27, color=WHITE)
                description.next_to(title, DOWN, buff=0.2)
                card = VGroup(rect, title, description)
                cards.add(card)

            cards.arrange(RIGHT, buff=0.8).move_to(DOWN * 0.3)

            # Arrows between cards
            arr1 = Arrow(cards[0][0].get_right(), cards[1][0].get_left(),
                         color=WHITE, stroke_width=2, tip_length=0.12, buff=0.1)
            arr2 = Arrow(cards[1][0].get_right(), cards[2][0].get_left(),
                         color="#FFD54F", stroke_width=2, tip_length=0.12, buff=0.1)

            self.play(FadeIn(cards[0]), run_time=0.5)
            self.play(GrowArrow(arr1), FadeIn(cards[1]), run_time=0.8)
            self.play(GrowArrow(arr2), FadeIn(cards[2]), run_time=0.8)
            self.wait(0.5)

        # --- Part 2: Torus vs Klein bottle ---
        with self.voiceover(
            text=(
                "Here is a key example. A torus and a Klein bottle "
                "look the same locally. Same cells, same neighborhoods. "
                "The difference is orientability. "
                "On a torus, edge identifications are consistent. "
                "On a Klein bottle, one pair of edges is reversed. "
                "HOMP gives the same output for both. "
                "MCN and SMCN can tell them apart."
            )
        ) as tracker:
            # Clear cards
            self.play(FadeOut(VGroup(cards, arr1, arr2)), run_time=0.5)

            # Polygon identification diagrams (the standard math way)
            # Torus: square with matching arrows
            sq_size = 2.0

            # --- TORUS ---
            torus_sq = Square(side_length=sq_size, color="#81C784", stroke_width=3)
            torus_sq.move_to(LEFT * 3 + DOWN * 0.3)

            # Top & bottom: both arrows point right (same direction)
            t_top = Arrow(
                torus_sq.get_corner(UL) + RIGHT * 0.4,
                torus_sq.get_corner(UR) + LEFT * 0.4,
                color="#81C784", stroke_width=3, tip_length=0.15, buff=0
            )
            t_bot = Arrow(
                torus_sq.get_corner(DL) + RIGHT * 0.4,
                torus_sq.get_corner(DR) + LEFT * 0.4,
                color="#81C784", stroke_width=3, tip_length=0.15, buff=0
            )
            # Left & right: both arrows point up
            t_left = Arrow(
                torus_sq.get_corner(DL) + UP * 0.4,
                torus_sq.get_corner(UL) + DOWN * 0.4,
                color="#4FC3F7", stroke_width=3, tip_length=0.15, buff=0
            )
            t_right = Arrow(
                torus_sq.get_corner(DR) + UP * 0.4,
                torus_sq.get_corner(UR) + DOWN * 0.4,
                color="#4FC3F7", stroke_width=3, tip_length=0.15, buff=0
            )
            torus_label = Text("Torus", font_size=24, color="#81C784", weight=BOLD)
            torus_label.next_to(torus_sq, DOWN, buff=0.7)
            torus_orient = Text("(orientable)", font_size=18, color="#81C784")
            torus_orient.next_to(torus_label, DOWN, buff=0.1)

            # --- KLEIN BOTTLE ---
            klein_sq = Square(side_length=sq_size, color="#EF5350", stroke_width=3)
            klein_sq.move_to(RIGHT * 3 + DOWN * 0.3)

            # Top: arrow right, Bottom: arrow LEFT (reversed!)
            k_top = Arrow(
                klein_sq.get_corner(UL) + RIGHT * 0.4,
                klein_sq.get_corner(UR) + LEFT * 0.4,
                color="#EF5350", stroke_width=3, tip_length=0.15, buff=0
            )
            k_bot = Arrow(
                klein_sq.get_corner(DR) + LEFT * 0.4,
                klein_sq.get_corner(DL) + RIGHT * 0.4,
                color="#EF5350", stroke_width=3, tip_length=0.15, buff=0
            )
            # Left & right: same direction (up)
            k_left = Arrow(
                klein_sq.get_corner(DL) + UP * 0.4,
                klein_sq.get_corner(UL) + DOWN * 0.4,
                color="#4FC3F7", stroke_width=3, tip_length=0.15, buff=0
            )
            k_right = Arrow(
                klein_sq.get_corner(DR) + UP * 0.4,
                klein_sq.get_corner(UR) + DOWN * 0.4,
                color="#4FC3F7", stroke_width=3, tip_length=0.15, buff=0
            )
            klein_label = Text("Klein Bottle", font_size=24, color="#EF5350", weight=BOLD)
            klein_label.next_to(klein_sq, DOWN, buff=0.7)
            klein_orient = Text("(non-orientable)", font_size=18, color="#EF5350")
            klein_orient.next_to(klein_label, DOWN, buff=0.1)

            # Animate torus
            self.play(Create(torus_sq), run_time=0.5)
            self.play(
                GrowArrow(t_top), GrowArrow(t_bot),
                GrowArrow(t_left), GrowArrow(t_right),
                run_time=1.0
            )
            self.play(Write(torus_label), FadeIn(torus_orient), run_time=0.5)

            # Animate klein
            self.play(Create(klein_sq), run_time=0.5)
            self.play(
                GrowArrow(k_top), GrowArrow(k_bot),
                GrowArrow(k_left), GrowArrow(k_right),
                run_time=1.0
            )
            self.play(Write(klein_label), FadeIn(klein_orient), run_time=0.5)

            # Highlight the reversed arrow
            highlight_rect = SurroundingRectangle(k_bot, color="#FFD54F",
                                                  stroke_width=2, buff=0.08)
            rev_note = Text("← reversed!", font_size=18, color="#FFD54F")
            rev_note.next_to(highlight_rect, DOWN, buff=0.15)
            self.play(Create(highlight_rect), FadeIn(rev_note), run_time=0.8)

            # Result
            result = Text("HOMP: same output  |  MCN/SMCN: distinguished!",
                          font_size=20, color=WHITE)
            result.to_edge(DOWN, buff=0.3)
            self.play(FadeIn(result, shift=UP * 0.2), run_time=0.8)
            self.wait(1.5)

        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.8)
        self.wait(0.5)


In [ ]:
import subprocess, sys
from pathlib import Path
from IPython.display import Video, display

# ═══════════════════════════════════════════════════════════
# RENDER scenes 116–122
# ═══════════════════════════════════════════════════════════
scene_file = "/content/slides_116_122.py"
scenes_to_render = [
    "Scene29GeometryBeyondTitle",
    "Scene30Flexibility",
    "Scene31Efficiency",
    "Scene32WhatKind",
    "Scene33Curvature",
    "Scene34TopologyI",
    "Scene35TopologyII",
]

output_paths_116_122 = []

for scene_name in scenes_to_render:
    print(f"Rendering {scene_name}...")
    result = subprocess.run(
        [
            sys.executable, "-m", "manim", "render",
            "-qh",                    # 1080p, 60fps
            "--disable_caching",
            scene_file,
            scene_name,
        ],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"  Error:\n{result.stderr[-800:]}")
    else:
        found = list(Path("media").rglob(f"{scene_name}.mp4"))
        if found:
            output_paths_116_122.append(str(found[0]))
            print(f"  {found[0]}")
        else:
            print(f"  Rendered OK but file not found")

print(f"\nTotal: {len(output_paths_116_122)} videos")
for p in output_paths_116_122:
    print(f"  {p}")

# ═══════════════════════════════════════════════════════════
# CONCAT — thống nhất 60fps, CRF 20
# ═══════════════════════════════════════════════════════════
list_file = "/content/concat_list_116_122.txt"
with open(list_file, "w") as f:
    for p in output_paths_116_122:
        if Path(p).exists():
            f.write(f"file '{p}'\n")

output_file = "/content/slides_116_122_voiceover.mp4"

cmd = [
    "ffmpeg", "-y",
    "-f", "concat",
    "-safe", "0",
    "-i", list_file,
    "-c:v", "libx264",
    "-preset", "medium",
    "-crf", "20",
    "-r", "60",
    "-pix_fmt", "yuv420p",
    "-c:a", "aac",
    "-b:a", "192k",
    "-movflags", "+faststart",
    output_file
]

result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode == 0 and Path(output_file).exists():
    probe = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries",
         "format=duration,bit_rate", "-show_entries",
         "stream=r_frame_rate,width,height,bit_rate",
         "-of", "default=noprint_wrappers=1",
         output_file],
        capture_output=True, text=True
    )
    print(f"\nFinal: {output_file}")
    print(probe.stdout)
    display(Video(output_file, embed=True, width=854))
else:
    print(f"FFmpeg error:\n{result.stderr[-500:]}")


## Display

In [ ]:
import subprocess
from pathlib import Path
from IPython.display import Video, display
import re

# Tìm tất cả các file Scene29 đến Scene35
output_paths_116_122 = sorted(
    p for p in Path("media").rglob("Scene*.mp4")
    if re.search(r"Scene(\d+)", p.stem) and 29 <= int(re.search(r"Scene(\d+)", p.stem).group(1)) <= 35
)

list_file = "/content/concat_list_116_122.txt"
output_file = "/content/slides_116_122_voiceover.mp4"

# Tạo file danh sách cho FFmpeg
with open(list_file, "w") as f:
    for p in output_paths_116_122:
        f.write(f"file '{p.absolute()}'\n")

# Ghép bằng FFmpeg — re-encode thống nhất 60fps, CRF 20
cmd = [
    "ffmpeg", "-y",
    "-f", "concat",
    "-safe", "0",
    "-i", list_file,
    "-c:v", "libx264",
    "-preset", "medium",
    "-crf", "20",
    "-r", "60",
    "-pix_fmt", "yuv420p",
    "-c:a", "aac",
    "-b:a", "192k",
    "-movflags", "+faststart",
    output_file
]

result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode == 0 and Path(output_file).exists():
    probe = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries",
         "format=duration,bit_rate", "-show_entries",
         "stream=r_frame_rate,width,height,bit_rate",
         "-of", "default=noprint_wrappers=1",
         output_file],
        capture_output=True, text=True
    )
    print(f"Thành công: {output_file}")
    print(probe.stdout)
    display(Video(output_file, embed=True, width=854))
else:
    print("Lỗi FFmpeg:")
    print(result.stderr[-500:])


# SLIDE 123-126


## Generative

In [ ]:
%%writefile /content/slides_123_126.py
# FILE: slides_123_126.py — Acknowledgements & Thank You
# Slides 123–126

import nest_asyncio
nest_asyncio.apply()

import asyncio
import numpy as np
from pathlib import Path

from manim import *
from manim_voiceover import VoiceoverScene
from manim_voiceover.services.base import SpeechService

import edge_tts
class EdgeTTSService(SpeechService):
    def __init__(self, voice="en-US-GuyNeural", **kwargs):
        self.voice = voice
        super().__init__(**kwargs)

    def generate_from_text(self, text: str, cache_dir: str = None, path: str = None, **kwargs) -> dict:
        if cache_dir is None:
            cache_dir = self.cache_dir
        input_data = {"input_text": text, "service": "edge_tts", "voice": self.voice}
        cached = self.get_cached_result(input_data, Path(cache_dir))
        if cached is not None:
            return cached

        audio_basename = self.get_audio_basename(input_data)
        if path is None:
            audio_path = audio_basename + ".mp3"
        else:
            audio_path = path
        output_file = str(Path(cache_dir) / audio_path)

        async def _generate():
            communicate = edge_tts.Communicate(text, self.voice)
            await communicate.save(output_file)

        asyncio.run(_generate())

        json_dict = {
            "input_text": text,
            "input_data": input_data,
            "original_audio": audio_path,
        }
        return json_dict


VOICE = "en-US-GuyNeural"
# ═══════════════════════════════════════════════════════════════════
# SCENE 36: Acknowledgements & Outro (Slides 123–126, condensed)
# ═══════════════════════════════════════════════════════════════════
class Scene36Outro(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        # ── Part 1: Acknowledgements ──
        with self.voiceover(
            text=(
                "The presenters acknowledge support from TILOS, "
                "the Humboldt Foundation, AIVO, and their many collaborators "
                "at MIT, Harvard, and TUM."
            )
        ) as tracker:
            title = Text("Acknowledgements", font_size=40, color=WHITE, weight=BOLD)
            title.to_edge(UP, buff=0.5)
            self.play(FadeIn(title, shift=DOWN * 0.2), run_time=0.8)

            # Fixed x positions
            label_x = -4.5   # right-edge of labels
            badges_x = 0.5   # center of badges group

            # ── Funding row ──
            funding_label = Text("Funding:", font_size=20, color=GREY_A)
            funding_label.move_to(UP * 1.2).set_x(label_x).align_to(funding_label, RIGHT)
            # Actually: set right edge at fixed x
            funding_label.shift(RIGHT * (label_x - funding_label.get_right()[0]))

            funding_badges = VGroup(
                self._badge("TILOS", "#81C784"),
                self._badge("Humboldt", "#4FC3F7"),
                self._badge("AIVO", "#FFD54F"),
            ).arrange(RIGHT, buff=0.6)
            funding_badges.move_to(UP * 1.2).set_x(badges_x)

            # ── Institutions row ──
            inst_label = Text("Institutions:", font_size=20, color=GREY_A)
            inst_label.move_to(DOWN * 0.0)
            # Align right edge to same x as funding label
            inst_label.shift(RIGHT * (label_x - inst_label.get_right()[0]))

            inst_badges = VGroup(
                self._badge("MIT", RED),
                self._badge("Harvard", "#A41034"),
                self._badge("TUM", "#0065BD"),
            ).arrange(RIGHT, buff=0.6)
            inst_badges.move_to(DOWN * 0.0).set_x(badges_x)

            # ── Collaborators ──
            collab_label = Text("Collaborators:", font_size=20, color=GREY_A)
            collab_label.move_to(DOWN * 1.2)
            collab_label.shift(RIGHT * (label_x - collab_label.get_right()[0]))

            names = [
                "A. Loukas", "H. Maron", "T. Poggio", "S. Villar", "T. Smidt", "V. Garg",
                "M. Weber", "N. Dym", "R. Levie", "R. Walters", "D. Lim", "E. Santos-Escriche"
            ]
            row1 = VGroup(*[Text(n, font_size=16, color=GREY_B) for n in names[:6]]).arrange(RIGHT, buff=0.45)
            row2 = VGroup(*[Text(n, font_size=16, color=GREY_B) for n in names[6:]]).arrange(RIGHT, buff=0.45)
            name_grid = VGroup(row1, row2).arrange(DOWN, buff=0.25)
            name_grid.move_to(DOWN * 1.8).set_x(badges_x)

            # Animate
            # self.play(FadeIn(title), run_time=0.5)
            self.play(FadeIn(funding_label), FadeIn(funding_badges), run_time=1.0)
            self.play(FadeIn(inst_label), FadeIn(inst_badges), run_time=0.8)
            self.play(FadeIn(collab_label), FadeIn(name_grid), run_time=1.0)
            self.wait(max(0, tracker.duration - 4.0))


        # ── Part 2: Thank you + Disclaimer ──
        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.5)

        with self.voiceover(
            text=(
                "Thank you for watching. This video was produced as a lab assignment "
                "and may contain errors or inaccuracies. "
                "Please refer to the original tutorial slides and papers "
                "for authoritative information. "
                # "For questions, reach out at b z t at mit dot edu."
            )
        ) as tracker:
            # Thank you
            thank = Text("Thank You for Watching!", font_size=44, color=WHITE, weight=BOLD)
            thank.move_to(UP * 1.5)
            self.play(FadeIn(thank, scale=0.7), run_time=1.0)

            # Disclaimer box
            disc_box = RoundedRectangle(width=10, height=2.2, corner_radius=0.15,
                                        color=YELLOW, fill_opacity=0.03, stroke_width=1.5)
            disc_box.move_to(DOWN * 0.5)

            disc_lines = VGroup(
                Text("This video is a lab assignment and may contain inaccuracies.",
                     font_size=18, color=YELLOW),
                Text("Please refer to the original tutorial and papers.",
                     font_size=18, color=WHITE),
            ).arrange(DOWN, buff=0.2).move_to(disc_box)

            self.play(Create(disc_box), run_time=0.5)
            self.play(
                LaggedStart(*[FadeIn(l, shift=UP * 0.1) for l in disc_lines], lag_ratio=0.3),
                run_time=1.2
            )

            # Contact
            # contact = Text("Contact: bzt@mit.edu", font_size=20, color="#4FC3F7")
            # contact.move_to(DOWN * 2.5)
            # self.play(FadeIn(contact), run_time=0.5)

            self.wait(max(0, tracker.duration - 4.0))

        # Hold final frame
        self.wait(3.0)
        self.play(*[FadeOut(m) for m in self.mobjects], run_time=1.0)
        self.wait(0.5)

    def _badge(self, name, color):
        box = RoundedRectangle(width=2.2, height=0.7, corner_radius=0.1,
                               color=color, fill_opacity=0.1, stroke_width=1.5)
        label = Text(name, font_size=18, color=color, weight=BOLD).move_to(box)
        return VGroup(box, label)


In [ ]:
import subprocess, sys
from pathlib import Path
from IPython.display import Video, display

# ═══════════════════════════════════════════════════════════
# RENDER scenes 123–126
# ═══════════════════════════════════════════════════════════
scene_file = "/content/slides_123_126.py"
scenes_to_render = [
    "Scene36Outro",
]

output_paths_123_126 = []

for scene_name in scenes_to_render:
    print(f"Rendering {scene_name}...")
    result = subprocess.run(
        [
            sys.executable, "-m", "manim", "render",
            "-qh",                    # 1080p, 60fps
            "--disable_caching",
            scene_file,
            scene_name,
        ],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"  Error:\n{result.stderr[-800:]}")
    else:
        found = list(Path("media").rglob(f"{scene_name}.mp4"))
        if found:
            output_paths_123_126.append(str(found[0]))
            print(f"  {found[0]}")
        else:
            print(f"  Rendered OK but file not found")

print(f"\nTotal: {len(output_paths_123_126)} videos")
for p in output_paths_123_126:
    print(f"  {p}")

# ═══════════════════════════════════════════════════════════
# CONCAT — thống nhất 60fps, CRF 20
# (Dù chỉ 1 scene, vẫn re-encode để đảm bảo format khớp khi ghép cuối)
# ═══════════════════════════════════════════════════════════
list_file = "/content/concat_list_123_126.txt"
with open(list_file, "w") as f:
    for p in output_paths_123_126:
        if Path(p).exists():
            f.write(f"file '{p}'\n")

output_file = "/content/slide_123_126_voiceover.mp4"

cmd = [
    "ffmpeg", "-y",
    "-f", "concat",
    "-safe", "0",
    "-i", list_file,
    "-c:v", "libx264",
    "-preset", "medium",
    "-crf", "20",
    "-r", "60",
    "-pix_fmt", "yuv420p",
    "-c:a", "aac",
    "-b:a", "192k",
    "-movflags", "+faststart",
    output_file
]

result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode == 0 and Path(output_file).exists():
    probe = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries",
         "format=duration,bit_rate", "-show_entries",
         "stream=r_frame_rate,width,height,bit_rate",
         "-of", "default=noprint_wrappers=1",
         output_file],
        capture_output=True, text=True
    )
    print(f"\nFinal: {output_file}")
    print(probe.stdout)
    display(Video(output_file, embed=True, width=854))
else:
    print(f"FFmpeg error:\n{result.stderr[-500:]}")


## Display

In [ ]:
import subprocess
from pathlib import Path
from IPython.display import Video, display

concat_list = "/content/concat_list_123_126.txt"
output_file = "/content/slide_123_126_voiceover.mp4"

with open(concat_list, "w") as f:
    for p in output_paths_123_126:
        if Path(p).exists():
            f.write(f"file '{p}'\n")

result = subprocess.run(
    [
        "ffmpeg", "-y",
        "-f", "concat",
        "-safe", "0",
        "-i", concat_list,
        "-c:v", "libx264",
        "-preset", "medium",
        "-crf", "20",
        "-r", "60",
        "-pix_fmt", "yuv420p",
        "-c:a", "aac",
        "-b:a", "192k",
        "-movflags", "+faststart",
        output_file,
    ],
    capture_output=True, text=True
)

if result.returncode == 0 and Path(output_file).exists():
    probe = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries",
         "format=duration,bit_rate", "-show_entries",
         "stream=r_frame_rate,width,height,bit_rate",
         "-of", "default=noprint_wrappers=1",
         output_file],
        capture_output=True, text=True
    )
    print(f"Final: {output_file}")
    print(probe.stdout)
    display(Video(output_file, embed=True, width=854))
else:
    print(f"FFmpeg error:\n{result.stderr[-500:]}")


# Final


In [ ]:
import subprocess
from pathlib import Path
from IPython.display import Video, display

final_segments = [
    '/content/slide_81_85_voiceover.mp4',
    '/content/slide_86_92_voiceover.mp4',
    '/content/slide_93_101_voiceover.mp4',
    '/content/slide_102_110_voiceover.mp4',
    '/content/slides_111_115_voiceover.mp4',
    '/content/slides_116_122_voiceover.mp4',
    '/content/slide_123_126_voiceover.mp4'
]

full_concat_list = '/content/final_full_concat_list.txt'
missing = []
with open(full_concat_list, 'w') as f:
    for p in final_segments:
        if Path(p).exists():
            f.write(f"file '{p}'\n")
        else:
            missing.append(p)

if missing:
    print(f"Missing {len(missing)} segment(s):")
    for m in missing:
        print(f"   {m}")

output_full = '/content/full_presentation_voiceover.mp4'

cmd = [
    'ffmpeg', '-y',
    '-f', 'concat',
    '-safe', '0',
    '-i', full_concat_list,
    '-c:v', 'libx264',
    '-preset', 'medium',
    '-crf', '20',
    '-r', '60',
    '-pix_fmt', 'yuv420p',
    '-c:a', 'aac',
    '-b:a', '192k',
    '-movflags', '+faststart',
    output_full
]

print("Merging all segments into final video...")
result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode == 0 and Path(output_full).exists():
    probe = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries",
         "format=duration,bit_rate", "-show_entries",
         "stream=r_frame_rate,width,height,bit_rate",
         "-of", "default=noprint_wrappers=1",
         output_full],
        capture_output=True, text=True
    )
    duration_str = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "default=noprint_wrappers=1:nokey=1", output_full],
        capture_output=True, text=True
    ).stdout.strip()
    duration = float(duration_str)

    print(f'Full video created successfully!')
    print(f'Path: {output_full}')
    print(f'⏱️ Total Duration: {duration/60:.1f} minutes ({duration:.0f}s)')
    print(f'\nStream info:')
    print(probe.stdout)
    display(Video(output_full, embed=True, width=854))
else:
    print('FFmpeg error:')
    print(result.stderr[-800:])
